In [0]:
# ******************************************************************************************
# || PROYECTO       : MIGRACION DWH - CLOUD
# || NOMBRE         : UD_EGP.py
# || TABLA DESTINO  : mb_silver_prod.dwh_ods.ud_egp_ms
# || TABLA FUENTE   : mb_silver_prod.tpz_edy.ud_egd
# ||                : mb_silver_prod.tpz_edy.md_t_cond_inte_deve_vige
# ||                : mb_silver_prod.tpz_edy.md_t_cond_inte_deve_sust
# ||                : mb_silver_prod.tpz_edy.co_productos
# ||                : mb_silver_prod.tpz_edy.bs_historia_plazo
# ||                : mb_silver_prod.tpz_edy.cr_categorias_cartera
# ||                : mb_silver_prod.tpz_edy.devengamiento_por_cuota
# ||                : mb_silver_prod.tpz_edy.bs_planpagos
# ||                : mb_silver_prod.tpz_edy.saldos_activos
# ||                : mb_silver_prod.tpz_edy.saldos_pasivos
# ||                : mb_silver_prod.tpz_edy.cl_clientes
# ||                : mb_silver_prod.tpz_edy.pc_soliccastigo
# ||                : mb_silver_prod.tpz_edy.cr_lineacredito
# ||                : mb_silver_prod.tpz_edy.sl_segurodesgravamen
# ||                : mb_silver_prod.tpz_edy.sl_solicitudcreditopersona
# ||                : mb_silver_prod.tpz_edy.sl_notasporsol
# ||                : mb_silver_prod.tpz_edy.sl_solicitudcredito
# ||                : mb_silver_prod.tpz_edy.cl_grupossociales
# ||                : mb_silver_prod.tpz_edy.sol_rel_sub_cre
# ||                : mb_silver_prod.tpz_edy.sl_criterios
# ||                : mb_silver_prod.tpz_edy.sl_prop_credito
# ||                : mb_silver_prod.tpz_edy.cre_pro_cre_rie
# ||                : mb_silver_prod.tpz_edy.sl_solicitudcredcancela
# ||                : mb_silver_prod.tpz_edy.hd_puesto_cargo_analista
# ||                : mb_silver_prod.tpz_edy.cl_clientpersona
# ||                : mb_silver_prod.tpz_edy.rc_basenegmotivo
# ||                : mb_silver_prod.tpz_edy.cl_personasfisicas
# ||                : mb_silver_prod.tpz_edy.cl_personasjuridicas
# ||                : mb_silver_prod.tpz_edy.rc_rcc_cuerpo
# ||                : mb_silver_prod.tpz_edy.rc_cuentascontables
# ||                : mb_silver_prod.tpz_edy.asientos
# ||                : mb_silver_prod.tpz_edy.autorizaciones
# ||                : mb_silver_prod.tpz_edy.sl_rel_sol_destino
# ||                : mb_silver_prod.tpz_edy.rh_funcionario
# ||                : mb_silver_prod.tpz_edy.adr_car_pue
# ||                : mb_silver_prod.tpz_edy.tc_sucursales
# ||                : mb_silver_prod.tpz_edy.gastos_por_cuota
# ||                : mb_silver_prod.tpz_edy.prestamos_fogapi
# ||                : mb_silver_prod.tpz_edy.gr_relaciongtiacredsolic
# ||                : mb_silver_prod.tpz_edy.gr_garantias
# ||                : mb_silver_prod.tpz_edy.bs_pays_detail
# ||                : mb_silver_prod.tpz_edy.bs_charge_detail
# ||                : mb_silver_prod.tpz_edy.rep_cartera_morosa
# ||                : mb_silver_prod.tpz_edy.ud_egp
# ||                : mb_silver_prod.tpz_edy.cr_recupercartera
# ||                : mb_silver_prod.tpz_edy.historia_vista
# ||                : mb_silver_prod.tpz_edy.co_concepcont
# ||                : mb_silver_prod.tpz_edy.co_monedas
# ||                : mb_silver_prod.tpz_edy.md_cronogramas_creditos
# ||                : mb_silver_prod.tpz_edy.sm_cronograma
# ||                : mb_silver_prod.tpz_edy.ci_cargos
# ||                : mb_silver_prod.tpz_edy.ci_cargos_tarifas
# ||                : mb_silver_prod.tpz_edy.tc_parametros
# || OBJETIVO       : Logicas para cargar la tabla maestra de ud_egp de la malla diaria del DWH
# || TIPO           : PY
# || REPROCESABLE   : SI-RERUN
# || OBSERVACIÓN    : NA
# || SCHEDULER      : NA
# || JOB            : NA
# || VERSION      DESARROLLADOR         PROVEEDOR        PO               FECHA        DESCRIPCION
# || --------------------------------------------------------------------------------
# || 1           Albert Hermitaño     MS       Enith Rodriguez 04/06/2026   Ejecucion de proceso UD_EGP
# ******************************************************************************************

In [0]:
dbutils.widgets.text("ambiente", "")
dbutils.widgets.text("fechaproceso", "")

In [0]:
var_ambiente = dbutils.widgets.get("ambiente")
var_fechaproceso = dbutils.widgets.get("fechaproceso")

# Validación de parámetros obligatorios
if not var_ambiente or not var_ambiente.strip():
    dbutils.notebook.exit("ERROR: El parámetro 'ambiente' es obligatorio y está vacío.")
if not var_fechaproceso or not var_fechaproceso.strip():
    dbutils.notebook.exit("ERROR: El parámetro 'fechaproceso' es obligatorio y está vacío.")

In [0]:
import logging
import time
from datetime import date, datetime
from functools import reduce

from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    add_months, aes_decrypt, broadcast, coalesce, col, concat, concat_ws,
    count, countDistinct, current_date, current_timestamp, date_add,
    date_format, date_sub, datediff, dayofmonth, expr, first, floor,
    last_day, lit, max, min, month, pmod, regexp_replace, round, row_number,
    sha2, substring, sum, to_date, trim, trunc, upper, when, year,
)
from pyspark.sql.types import (
    BinaryType, DateType, DecimalType, IntegerType, LongType, StringType,
    StructField, StructType, TimestampType,
)
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

import traceback

logger = logging.getLogger("UD_EGP")
logger.setLevel(logging.INFO)


In [0]:
%run /Data/Core/Topaz/Utilitarios/Utils/Utils

In [0]:
controlar_disponibilidad_tablas(
    tablas=[
        "dwh_ods.ud_egd",
        "dwh_ods.hd_puesto_cargo_analista",
        "dwh_ods.md_cronogramas_creditos",
        "tpz_edy.sl_solicitudcredcancela",
        "tpz_edy.gr_relaciongtiacredsolic",
        "tpz_edy.co_monedas",
    ],
    ambiente=var_ambiente,
    fechaproceso=var_fechaproceso
)

In [0]:
def get_parametros_proceso(ambiente, des_proceso):
    """
    Obtiene los parámetros de proceso desde m_proceso_carga y los retorna como diccionario.
    :param ambiente: ambiente de ejecución (dev/prod)
    :param des_proceso: Descripción del proceso (Des_proceso_carga)
    :return: dict {Cod_parametro: Des_valor_parametro}
    """
    df_parametros = spark.table(f"mb_utilitarios_{ambiente}.dbo.m_proceso_carga") \
        .filter(col("Des_proceso_carga") == des_proceso) \
        .select("Cod_parametro", "Des_valor_parametro")

    dict_parametros = {row["Cod_parametro"]: row["Des_valor_parametro"] for row in df_parametros.toLocalIterator()}

    if not dict_parametros:
        dbutils.notebook.exit(f"ERROR: No se encontraron parámetros para el proceso '{des_proceso}'")

    return dict_parametros


def registrar_bitacora(
    func_main,
    ambiente,
    fechaproceso,
    des_proceso,
    nom_tabla_destino,
    nom_esquema_destino="dwh_ods",
    des_capa="Silver",
    num_secuencia=1,
    des_proyecto="Migracion DWH->LHCL",
    cod_aplicativo="DWH",
    des_archivo_carga="-",
    tip_tecnologia="Databricks",
    max_retries=3,
    backoff_base=2,
):
    """
    Ejecuta el proceso principal y registra en bitácora con retry + backoff.
    :param func_main: función principal a ejecutar
    :param ambiente: ambiente de ejecución (dev/prod)
    :param fechaproceso: fecha de proceso (str YYYY-MM-DD)
    :param des_proceso: descripción del proceso
    :param nom_tabla_destino: nombre de la tabla destino
    :param nom_esquema_destino: esquema destino
    :param des_capa: capa (Silver, Gold, etc.)
    :param num_secuencia: número de secuencia de ejecución
    :param des_proyecto: nombre del proyecto
    :param cod_aplicativo: código de aplicativo
    :param des_archivo_carga: archivo de carga
    :param tip_tecnologia: tecnología de carga
    :param max_retries: número máximo de reintentos para escritura en bitácora
    :param backoff_base: base exponencial en segundos para espera entre reintentos
    """
    tabla_bitacora = f"mb_utilitarios_{ambiente}.dbo.h_bitacora"

    # Determinar tipo de carga
    df_check = spark.table(tabla_bitacora) \
        .filter((col("Des_proceso") == des_proceso) & (col("Fec_carga") == fechaproceso))
    tip_carga = "Reproceso" if df_check.count() > 0 else "Rutina"

    # Capturar tiempo de inicio antes de ejecutar el proceso
    Fec_inicio_proceso = datetime.now()

    # Ejecutar proceso principal y capturar estado
    error_msg = None

    try:
        func_main()
        estado = "Correcto"
    except Exception as e:
        estado = "Error"
        error_msg = f"{type(e).__name__}: {str(e)}"
        logger.error("Fallo la ejecucion del proceso: %s", error_msg)

    Fec_fin_proceso = datetime.now()

    # Registrar en bitácora con retry + exponential backoff
    schema_bitacora = StructType([
        StructField("Fec_inicio_proceso", TimestampType(), True),
        StructField("Fec_fin_proceso", TimestampType(), True),
        StructField("Des_proceso", StringType(), True),
        StructField("Des_capa", StringType(), True),
        StructField("Num_secuencia_ejecucion", IntegerType(), True),
        StructField("Des_proyecto", StringType(), True),
        StructField("Cod_aplicativo", StringType(), True),
        StructField("Des_archivo_carga", StringType(), True),
        StructField("Tip_tecnologia_carga", StringType(), True),
        StructField("Nom_esquema_destino", StringType(), True),
        StructField("Nom_tabla_destino", StringType(), True),
        StructField("Des_estado_carga", StringType(), True),
        StructField("Fec_carga", DateType(), True),
        StructField("Tip_carga_procesamiento", StringType(), True),
        StructField("_ingestion_time", TimestampType(), True),
        StructField("_processing_time", TimestampType(), True)
    ])

    row = [(Fec_inicio_proceso, Fec_fin_proceso, des_proceso, des_capa,
            num_secuencia, des_proyecto, cod_aplicativo, des_archivo_carga,
            tip_tecnologia, nom_esquema_destino, nom_tabla_destino, estado,
            date.fromisoformat(fechaproceso), tip_carga,
            datetime.now(), datetime.now())]

    df_bitacora = spark.createDataFrame(row, schema=schema_bitacora)

    intento = 1
    exito = False
    while intento <= max_retries and not exito:
        try:
            df_bitacora.write.mode("append").saveAsTable(tabla_bitacora)
            logger.info("Bitacora registrada (intento %s/%s) - estado: %s",
                        intento, max_retries, estado)
            exito = True
        except Exception as e:
            if intento == max_retries:
                logger.error("No se pudo registrar la bitacora tras %s intentos: %s",
                             max_retries, e)
                raise
            wait_time = backoff_base ** intento
            logger.warning("Reintento %s/%s en %ss. Error: %s",
                           intento, max_retries, wait_time, e)
            time.sleep(wait_time)
        intento += 1

    # Si el proceso falló, salir del notebook DESPUÉS de registrar en bitácora
    if estado == "Error":
        dbutils.notebook.exit(f"Proceso finalizó con estado ERROR: {error_msg}")

In [0]:
class UD_EGP:
    def __init__(self, ambiente, fechaproceso, dict_parametros):
        """
        :param ambiente: ambiente de ejecución (dev/prod)
        :param fechaproceso: fecha de proceso (str YYYY-MM-DD)
        :param dict_parametros: diccionario de parámetros del proceso
        """
        self.spark                                  = SparkSession.builder.getOrCreate()
        self.CONS_AMBIENTE                          = ambiente
        self.FEC_PROCESO                            = fechaproceso
        self.CONS_TABLE_FINAL                      = dict_parametros["PRM_TABLE_FINAL"]
        self.CONS_RUTA_TABLE_FINAL                     = dict_parametros["PRM_RUTA_TABLE_FINAL"]
        self.CONS_TABLE_UD_EGD                      = dict_parametros["PRM_TABLE_UD_EGD"]
        self.CONS_RUTA_UD_EGD                       = dict_parametros["PRM_TABLE_RUTA_UD_EGD"]
        self.CONS_TABLE_CO_PRODUCTOS                = dict_parametros["PRM_TABLE_CO_PRODUCTOS"]
        self.CONS_RUTA_CO_PRODUCTOS                 = dict_parametros["PRM_TABLE_RUTA_CO_PRODUCTOS"]
        self.CONS_TABLE_ESQUEMACONTABLE                = dict_parametros["PRM_TABLE_ESQUEMACONTABLE"]
        self.CONS_RUTA_ESQUEMACONTABLE                 = dict_parametros["PRM_TABLE_RUTA_ESQUEMACONTABLE"]
        self.CONS_TABLE_PARAMETROS_JTS                = dict_parametros["PRM_TABLE_PARAMETROS_JTS"]
        self.CONS_RUTA_PARAMETROS_JTS                 = dict_parametros["PRM_TABLE_RUTA_PARAMETROS_JTS"]
        self.CONS_TABLE_BS_HISTORIA_PLAZO           = dict_parametros["PRM_TABLE_BS_HISTORIA_PLAZO"]
        self.CONS_RUTA_BS_HISTORIA_PLAZO            = dict_parametros["PRM_TABLE_RUTA_BS_HISTORIA_PLAZO"]
        self.CONS_TABLE_CR_CRITERIOS           = dict_parametros["PRM_TABLE_CR_CRITERIOS"]
        self.CONS_RUTA_CR_CRITERIOS            = dict_parametros["PRM_TABLE_RUTA_CR_CRITERIOS"]
        self.CONS_TABLE_CR_CATEGORIAS_CARTERA       = dict_parametros["PRM_TABLE_CR_CATEGORIAS_CARTERA"]
        self.CONS_RUTA_CR_CATEGORIAS_CARTERA        = dict_parametros["PRM_TABLE_RUTA_CR_CATEGORIAS_CARTERA"]
        self.CONS_TABLE_DEVENGAMIENTO_POR_CUOTA     = dict_parametros["PRM_TABLE_DEVENGAMIENTO_POR_CUOTA"]
        self.CONS_RUTA_DEVENGAMIENTO_POR_CUOTA      = dict_parametros["PRM_TABLE_RUTA_DEVENGAMIENTO_POR_CUOTA"]
        self.CONS_TABLE_BS_PLANPAGOS                = dict_parametros["PRM_TABLE_BS_PLANPAGOS"]
        self.CONS_RUTA_BS_PLANPAGOS                 = dict_parametros["PRM_TABLE_RUTA_BS_PLANPAGOS"]
        self.CONS_TABLE_SALDOS_ACTIVOS              = dict_parametros["PRM_TABLE_SALDOS_ACTIVOS"]
        self.CONS_RUTA_SALDOS_ACTIVOS               = dict_parametros["PRM_TABLE_RUTA_SALDOS_ACTIVOS"]
        self.CONS_TABLE_SALDOS_PASIVOS              = dict_parametros["PRM_TABLE_SALDOS_PASIVOS"]
        self.CONS_RUTA_SALDOS_PASIVOS               = dict_parametros["PRM_TABLE_RUTA_SALDOS_PASIVOS"]
        self.CONS_TABLE_CL_CLIENTES                 = dict_parametros["PRM_TABLE_CL_CLIENTES"]
        self.CONS_RUTA_CL_CLIENTES                  = dict_parametros["PRM_TABLE_RUTA_CL_CLIENTES"]
        self.CONS_TABLE_CO_PLANCTAS       = dict_parametros["PRM_TABLE_CO_PLANCTAS"]
        self.CONS_RUTA_CO_PLANCTAS        = dict_parametros["PRM_TABLE_RUTA_CO_PLANCTAS"]
        self.CONS_TABLE_PC_SOLICCASTIGO             = dict_parametros["PRM_TABLE_PC_SOLICCASTIGO"]
        self.CONS_RUTA_PC_SOLICCASTIGO              = dict_parametros["PRM_TABLE_RUTA_PC_SOLICCASTIGO"]
        self.CONS_TABLE_CR_LINEACREDITO             = dict_parametros["PRM_TABLE_CR_LINEACREDITO"]
        self.CONS_RUTA_CR_LINEACREDITO              = dict_parametros["PRM_TABLE_RUTA_CR_LINEACREDITO"]
        self.CONS_TABLE_SL_SEGURODESGRAVAMEN        = dict_parametros["PRM_TABLE_SL_SEGURODESGRAVAMEN"]
        self.CONS_RUTA_SL_SEGURODESGRAVAMEN         = dict_parametros["PRM_TABLE_RUTA_SL_SEGURODESGRAVAMEN"]
        self.CONS_TABLE_SL_SOLICITUDCREDITOPERSONA  = dict_parametros["PRM_TABLE_SL_SOLICITUDCREDITOPERSONA"]
        self.CONS_RUTA_SL_SOLICITUDCREDITOPERSONA   = dict_parametros["PRM_TABLE_RUTA_SL_SOLICITUDCREDITOPERSONA"]
        self.CONS_TABLE_SL_NOTASPORSOL              = dict_parametros["PRM_TABLE_SL_NOTASPORSOL"]
        self.CONS_RUTA_SL_NOTASPORSOL               = dict_parametros["PRM_TABLE_RUTA_SL_NOTASPORSOL"]
        self.CONS_TABLE_SL_SOLICITUDCREDITO         = dict_parametros["PRM_TABLE_SL_SOLICITUDCREDITO"]
        self.CONS_RUTA_SL_SOLICITUDCREDITO          = dict_parametros["PRM_TABLE_RUTA_SL_SOLICITUDCREDITO"]
        self.CONS_TABLE_CL_GRUPOSSOCIALES           = dict_parametros["PRM_TABLE_CL_GRUPOSSOCIALES"]
        self.CONS_RUTA_CL_GRUPOSSOCIALES            = dict_parametros["PRM_TABLE_RUTA_CL_GRUPOSSOCIALES"]
        self.CONS_TABLE_SOL_REL_SUB_CRE             = dict_parametros["PRM_TABLE_SOL_REL_SUB_CRE"]
        self.CONS_RUTA_SOL_REL_SUB_CRE              = dict_parametros["PRM_TABLE_RUTA_SOL_REL_SUB_CRE"]
        self.CONS_TABLE_SL_CRITERIOS                = dict_parametros["PRM_TABLE_SL_CRITERIOS"]
        self.CONS_RUTA_SL_CRITERIOS                 = dict_parametros["PRM_TABLE_RUTA_SL_CRITERIOS"]
        self.CONS_TABLE_SL_PROP_CREDITO             = dict_parametros["PRM_TABLE_SL_PROP_CREDITO"]
        self.CONS_RUTA_SL_PROP_CREDITO              = dict_parametros["PRM_TABLE_RUTA_SL_PROP_CREDITO"]
        self.CONS_TABLE_CRE_PRO_CRE_RIE             = dict_parametros["PRM_TABLE_CRE_PRO_CRE_RIE"]
        self.CONS_RUTA_CRE_PRO_CRE_RIE              = dict_parametros["PRM_TABLE_RUTA_CRE_PRO_CRE_RIE"]
        self.CONS_TABLE_SL_SOLICITUDCREDCANCELA     = dict_parametros["PRM_TABLE_SL_SOLICITUDCREDCANCELA"]
        self.CONS_RUTA_SL_SOLICITUDCREDCANCELA      = dict_parametros["PRM_TABLE_RUTA_SL_SOLICITUDCREDCANCELA"]
        self.CONS_TABLE_HD_PUESTO_CARGO_ANALISTA    = dict_parametros["PRM_TABLE_HD_PUESTO_CARGO_ANALISTA"]
        self.CONS_RUTA_HD_PUESTO_CARGO_ANALISTA     = dict_parametros["PRM_TABLE_RUTA_HD_PUESTO_CARGO_ANALISTA"]
        self.CONS_TABLE_CL_CLIENTPERSONA            = dict_parametros["PRM_TABLE_CL_CLIENTPERSONA"]
        self.CONS_RUTA_CL_CLIENTPERSONA             = dict_parametros["PRM_TABLE_RUTA_CL_CLIENTPERSONA"]
        self.CONS_TABLE_TMP_SALDOS_01            = dict_parametros["PRM_TABLE_TMP_SALDOS_01"]
        self.CONS_RUTA_TMP_SALDOS_01             = dict_parametros["PRM_TABLE_RUTA_TMP_SALDOS_01"]
        self.CONS_TABLE_TMP_SALDOS_03            = dict_parametros["PRM_TABLE_TMP_SALDOS_03"]
        self.CONS_RUTA_TMP_SALDOS_03             = dict_parametros["PRM_TABLE_RUTA_TMP_SALDOS_03"]
        self.CONS_TABLE_TMP_SALDOS_04            = dict_parametros["PRM_TABLE_TMP_SALDOS_04"]
        self.CONS_RUTA_TMP_SALDOS_04             = dict_parametros["PRM_TABLE_RUTA_TMP_SALDOS_04"]
        self.CONS_TABLE_RC_BASENEGMOTIVO            = dict_parametros["PRM_TABLE_RC_BASENEGMOTIVO"]
        self.CONS_RUTA_RC_BASENEGMOTIVO             = dict_parametros["PRM_TABLE_RUTA_RC_BASENEGMOTIVO"]
        self.CONS_TABLE_CL_PERSONASFISICAS          = dict_parametros["PRM_TABLE_CL_PERSONASFISICAS"]
        self.CONS_RUTA_CL_PERSONASFISICAS           = dict_parametros["PRM_TABLE_RUTA_CL_PERSONASFISICAS"]
        self.CONS_TABLE_CL_PERSONASJURIDICAS        = dict_parametros["PRM_TABLE_CL_PERSONASJURIDICAS"]
        self.CONS_RUTA_CL_PERSONASJURIDICAS         = dict_parametros["PRM_TABLE_RUTA_CL_PERSONASJURIDICAS"]
        self.CONS_TABLE_RC_RCC_CUERPO               = dict_parametros["PRM_TABLE_RC_RCC_CUERPO"]
        self.CONS_RUTA_RC_RCC_CUERPO                = dict_parametros["PRM_TABLE_RUTA_RC_RCC_CUERPO"]
        self.CONS_TABLE_RC_CUENTASCONTABLES         = dict_parametros["PRM_TABLE_RC_CUENTASCONTABLES"]
        self.CONS_RUTA_RC_CUENTASCONTABLES          = dict_parametros["PRM_TABLE_RUTA_RC_CUENTASCONTABLES"]
        self.CONS_TABLE_ASIENTOS                    = dict_parametros["PRM_TABLE_ASIENTOS"]
        self.CONS_RUTA_ASIENTOS                     = dict_parametros["PRM_TABLE_RUTA_ASIENTOS"]
        self.CONS_TABLE_AUTORIZACIONES              = dict_parametros["PRM_TABLE_AUTORIZACIONES"]
        self.CONS_RUTA_AUTORIZACIONES               = dict_parametros["PRM_TABLE_RUTA_AUTORIZACIONES"]
        self.CONS_TABLE_SL_REL_SOL_DESTINO          = dict_parametros["PRM_TABLE_SL_REL_SOL_DESTINO"]
        self.CONS_RUTA_SL_REL_SOL_DESTINO            = dict_parametros["PRM_TABLE_RUTA_SL_REL_SOL_DESTINO"]
        self.CONS_TABLE_RH_FUNCIONARIO               = dict_parametros["PRM_TABLE_RH_FUNCIONARIO"]
        self.CONS_RUTA_RH_FUNCIONARIO                = dict_parametros["PRM_TABLE_RUTA_RH_FUNCIONARIO"]
        self.CONS_TABLE_ADR_CAR_PUE                  = dict_parametros["PRM_TABLE_ADR_CAR_PUE"]
        self.CONS_RUTA_ADR_CAR_PUE                   = dict_parametros["PRM_TABLE_RUTA_ADR_CAR_PUE"]
        self.CONS_TABLE_TC_SUCURSALES                = dict_parametros["PRM_TABLE_TC_SUCURSALES"]
        self.CONS_RUTA_TC_SUCURSALES                 = dict_parametros["PRM_TABLE_RUTA_TC_SUCURSALES"]
        self.CONS_TABLE_GASTOS_POR_CUOTA             = dict_parametros["PRM_TABLE_GASTOS_POR_CUOTA"]
        self.CONS_RUTA_GASTOS_POR_CUOTA              = dict_parametros["PRM_TABLE_RUTA_GASTOS_POR_CUOTA"]
        self.CONS_TABLE_PRESTAMOS_FOGAPI             = dict_parametros["PRM_TABLE_PRESTAMOS_FOGAPI"]
        self.CONS_RUTA_PRESTAMOS_FOGAPI              = dict_parametros["PRM_TABLE_RUTA_PRESTAMOS_FOGAPI"]
        self.CONS_TABLE_GR_RELACIONGTIACREDSOLIC     = dict_parametros["PRM_TABLE_GR_RELACIONGTIACREDSOLIC"]
        self.CONS_RUTA_GR_RELACIONGTIACREDSOLIC      = dict_parametros["PRM_TABLE_RUTA_GR_RELACIONGTIACREDSOLIC"]
        self.CONS_TABLE_GR_GARANTIAS                 = dict_parametros["PRM_TABLE_GR_GARANTIAS"]
        self.CONS_RUTA_GR_GARANTIAS                  = dict_parametros["PRM_TABLE_RUTA_GR_GARANTIAS"]
        self.CONS_TABLE_BS_PAYS_DETAIL               = dict_parametros["PRM_TABLE_BS_PAYS_DETAIL"]
        self.CONS_RUTA_BS_PAYS_DETAIL                = dict_parametros["PRM_TABLE_RUTA_BS_PAYS_DETAIL"]
        self.CONS_TABLE_BS_CHARGE_DETAIL             = dict_parametros["PRM_TABLE_BS_CHARGE_DETAIL"]
        self.CONS_RUTA_BS_CHARGE_DETAIL              = dict_parametros["PRM_TABLE_RUTA_BS_CHARGE_DETAIL"]
        self.CONS_TABLE_REP_CARTERA_MOROSA           = dict_parametros["PRM_TABLE_REP_CARTERA_MOROSA"]
        self.CONS_RUTA_REP_CARTERA_MOROSA            = dict_parametros["PRM_TABLE_RUTA_REP_CARTERA_MOROSA"]
        self.CONS_TABLE_HD_EGP                       = dict_parametros["PRM_TABLE_HD_EGP"]
        self.CONS_RUTA_HD_EGP                        = dict_parametros["PRM_TABLE_RUTA_HD_EGP"]
        self.CONS_TABLE_CR_RECUPERCARTERA            = dict_parametros["PRM_TABLE_CR_RECUPERCARTERA"]
        self.CONS_RUTA_CR_RECUPERCARTERA             = dict_parametros["PRM_TABLE_RUTA_CR_RECUPERCARTERA"]
        self.CONS_TABLE_HISTORIA_VISTA               = dict_parametros["PRM_TABLE_HISTORIA_VISTA"]
        self.CONS_RUTA_HISTORIA_VISTA                = dict_parametros["PRM_TABLE_RUTA_HISTORIA_VISTA"]
        self.CONS_TABLE_CO_CONCEPCONT                = dict_parametros["PRM_TABLE_CO_CONCEPCONT"]
        self.CONS_RUTA_CO_CONCEPCONT                 = dict_parametros["PRM_TABLE_RUTA_CO_CONCEPCONT"]
        self.CONS_TABLE_CO_MONEDAS                   = dict_parametros["PRM_TABLE_CO_MONEDAS"]
        self.CONS_RUTA_CO_MONEDAS                    = dict_parametros["PRM_TABLE_RUTA_CO_MONEDAS"]
        self.CONS_TABLE_MD_CRONOGRAMAS_CREDITOS      = dict_parametros["PRM_TABLE_MD_CRONOGRAMAS_CREDITOS"]
        self.CONS_RUTA_MD_CRONOGRAMAS_CREDITOS       = dict_parametros["PRM_TABLE_RUTA_MD_CRONOGRAMAS_CREDITOS"]
        self.CONS_TABLE_SM_CRONOGRAMA                = dict_parametros["PRM_TABLE_SM_CRONOGRAMA"]
        self.CONS_RUTA_SM_CRONOGRAMA                 = dict_parametros["PRM_TABLE_RUTA_SM_CRONOGRAMA"]
        self.CONS_TABLE_CI_CARGOS                    = dict_parametros["PRM_TABLE_CI_CARGOS"]
        self.CONS_RUTA_CI_CARGOS                     = dict_parametros["PRM_TABLE_RUTA_CI_CARGOS"]
        self.CONS_TABLE_CI_CARGOS_TARIFAS            = dict_parametros["PRM_TABLE_CI_CARGOS_TARIFAS"]
        self.CONS_RUTA_CI_CARGOS_TARIFAS             = dict_parametros["PRM_TABLE_RUTA_CI_CARGOS_TARIFAS"]
        self.CONS_TABLE_TC_PARAMETROS                = dict_parametros["PRM_TABLE_TC_PARAMETROS"]
        self.CONS_RUTA_TC_PARAMETROS                 = dict_parametros["PRM_TABLE_RUTA_TC_PARAMETROS"]
        self.CONS_ESQU_EDY                          = dict_parametros["PRM_SCHEMA_EDY"]
        self.CONS_MODE_OVERWRITE                    = "overwrite"
        self.CONS_COL_PARTITION                     = "Fe_proceso"
        self.CONS_FORMAT_DELTA                       = "delta"
        self.CONS_STORAGELOCATION = (
            f"abfss://silver@dlslakehouse{self.CONS_AMBIENTE}001.dfs.core.windows.net/MB_Peru/UNIVERSAL/"
        )

        
        self.key_aes = dbutils.secrets.get(f"ScopeDatabricks{self.CONS_AMBIENTE}001","Secret-key-aes-encrypt")

        self.df_stg_t_saldos_activos = None
        self.df_stg_t_saldos_pasivos = None
        self.df_stg_tmp_egp_1 = None
        self.df_stg_tmp_egp_2 = None

        # Configuración global de Spark (una sola vez en init)
        self.spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

    def read_data(self, table_path):
        """Función genérica para obtener datos"""
        return self.spark.read.format(self.CONS_FORMAT_DELTA).load(table_path)

    def delete_data(self, ambiente, table):
        """Función para eliminar la data"""
        DeltaTable.forName(
            self.spark, f"mb_silver_{ambiente}.dwh_ods.{table}"
        ).delete()

    def formato_data(self, df, schema):
        """Aplica el esquema (cast) al dataframe"""
        campos = [col(c.name).cast(c.dataType).alias(c.name) for c in schema]
        return df.select(campos)
    
    def medir(self, nombre, df):
        """Registra el tiempo que toma materializar un DataFrame.

        Se usa para ubicar la etapa mas costosa del proceso.
        """
        start = time.perf_counter()
        df.count()
        logger.info("%s demoro %.2f segundos", nombre, time.perf_counter() - start)
    
    def write_delta_table(self, df, table_path, partition_col):
        """Escribe el dataframe en delta con overwrite dinámico por partición"""
        try:
            df.write \
                .format(self.CONS_FORMAT_DELTA) \
                .mode(partition_col) \
                .save(table_path)
        except Exception as e:
            raise RuntimeError(f"Error func: write_delta_table - {str(e)}")

    #funciones de insumos

    def get_ods_ud_egd(self): #check
        """Obtiene los datos de la tabla ud_egd"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_UD_EGD}/{self.CONS_TABLE_UD_EGD}"
        return self.read_data(table_path).select(
            col("CO_CLIENTE"),
            col("IN_ESTA_PASI"),
            col("CO_TIPO_PASI")
        )
    
    def get_edy_cl_clientes(self): #check
        """Obtiene los datos de la tabla cl_clientes"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CL_CLIENTES}/{self.CONS_TABLE_CL_CLIENTES}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)

    def get_stg_t_co_productos(self): #check
        """Obtiene los datos de la tabla co_productos"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CO_PRODUCTOS}/{self.CONS_TABLE_CO_PRODUCTOS}"
        return self.read_data(table_path).filter(col("tz_lock") == 0).select(
                col("C6250").alias("C6250"),
                col("C6251").alias("C6251"),
                col("C6252").alias("C6252"),
                col("C7996").alias("C7996"),
                col("C7997").alias("C7997"),
                col("C6253").alias("C6253"),
                col("C6271").alias("C6271"),
                col("C6734").alias("C6734"),	
                col("C8021").alias("C8021"),
                col("C6272").alias("C6272"),
                col("C7998").alias("C7998"),
                col("C8002").alias("C8002"),
                col("C7999").alias("C7999"),
                col("PRODBLCRO").alias("PRODBLCRO"),
                col("C7995").alias("C7995"),
                col("C6261").alias("C6261"), 
                col("C6292").alias("C6292"), 
                col("C8000").alias("C8000"),
                col("C6275").cast("decimal(18,0)").alias("C6275"),
                col("C6291").cast("decimal(18,0)").alias("C6291")
        )   

    def get_edy_bs_historia_plazo(self): #check
        """Obtiene los datos de la tabla bs_historia_plazo"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_BS_HISTORIA_PLAZO}/{self.CONS_TABLE_BS_HISTORIA_PLAZO}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0)

    def get_stg_t_bs_historia_plazo_summary3(self): #check
        """Obtiene los datos de la tabla bs_historia_plazo"""
        df_bs_historia_plazo = self.get_edy_bs_historia_plazo()
        return df_bs_historia_plazo.filter((col("TIPOMOV") == lit("R")) & (col("RUBROCONTABLE").like("14_6%"))).groupBy(
            col("SALDOS_JTS_OID")
        ).agg(
            max(col("FECHAVALOR")).alias("FECHAVALOR")
        )

    def get_stg_t_bs_historia_plazo_summary(self): #check
        """Obtiene los datos de la tabla bs_historia_plazo"""
        df_bs_historia_plazo = self.get_edy_bs_historia_plazo()
        return df_bs_historia_plazo.filter(col("TIPOMOV") == lit("P")).groupBy(
            col("SALDOS_JTS_OID")
        ).agg(
            max(col("FECHAVALOR")).alias("FECHAVALOR"),
            max(col("FECHAPROCESOMOV")).alias("FECHAPROCESOMOV")
        )

    def get_edy_cr_categorias_cartera(self): #check
        """Obtiene los datos de la tabla cr_categorias_cartera"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CR_CATEGORIAS_CARTERA}/{self.CONS_TABLE_CR_CATEGORIAS_CARTERA}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_devengamiento_por_cuota(self): #check
        """Obtiene los datos de la tabla devengamiento_por_cuota"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_DEVENGAMIENTO_POR_CUOTA}/{self.CONS_TABLE_DEVENGAMIENTO_POR_CUOTA}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0)

    def get_edy_bs_planpagos(self): #check
        """Obtiene los datos de la tabla bs_planpagos"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_BS_PLANPAGOS}/{self.CONS_TABLE_BS_PLANPAGOS}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_stg_t_deveng_por_cta_venc_st(self): #check
        """Obtiene los datos de la tabla t_deveng_por_cta_venc_st"""

        df_edy_devengamiento_por_cuota = self.get_edy_devengamiento_por_cuota()
        df_edy_bs_planpagos = self.get_edy_bs_planpagos()

        df_max_fecha = (
            df_edy_devengamiento_por_cuota
            .agg(max("FECHA").alias("MAX_FECHA"))
        )

        w_devengamiento_cuotas = (
            df_edy_devengamiento_por_cuota.alias("dpc")
            .crossJoin(df_max_fecha)
            .join(
                df_edy_bs_planpagos.alias("pp"),
                (col("pp.SALDO_JTS_OID") == col("dpc.SALDO_JTS_OID")) &
                (col("pp.C2300") == col("dpc.NRO_CUOTA")),
                "inner"
            )
            .filter(
                (col("dpc.FECHA") == col("MAX_FECHA")) &
                (col("dpc.NRO_CUOTA") > 0) 
            )
            .groupBy(
                col("dpc.SALDO_JTS_OID").alias("SALDOS_JTS_OID")
            )
            .agg(
                sum(
                    when(
                        (col("dpc.INT_DEVENGADO") - col("dpc.INT_PAGADO")) > 0,
                        col("dpc.INT_DEVENGADO") - col("dpc.INT_PAGADO")
                    ).otherwise(lit(0))
                ).alias("INTERES_DEVENGADO"),
                sum(col("pp.C2309")).alias("CAPITAL"),
                sum(col("pp.C2310")).alias("INTERES"),
                max(col("dpc.ESTADOATRASO")).alias("ESTADOATRASO")
            )
        )

        df_resultado = (
            df_edy_devengamiento_por_cuota.alias("D")
            .crossJoin(df_max_fecha)
            .filter(
                col("D.FECHA") == col("MAX_FECHA")
            )
            .join(
                w_devengamiento_cuotas
                    .select(col("SALDOS_JTS_OID").alias("SALDOS_JTS_OID_W"))
                    .dropDuplicates(),
                col("D.SALDO_JTS_OID") == col("SALDOS_JTS_OID_W"),
                "left_anti"
            )
            .select(
                col("D.SALDO_JTS_OID").alias("SALDOS_JTS_OID"),
                col("D.ESTADOATRASO").alias("ESTADOATRASO")
            )
        )

        return df_resultado
    
    def get_stg_t_deveng_por_cta_venc(self): #check
        """Obtiene los datos de la tabla t_deveng_por_cta_venc"""

        df_edy_devengamiento_por_cuota = self.get_edy_devengamiento_por_cuota()
        df_edy_bs_planpagos = self.get_edy_bs_planpagos().alias("pp")

        max_fecha = (
            df_edy_devengamiento_por_cuota
            .agg(max("FECHA").alias("MAX_FECHA"))
            .first()["MAX_FECHA"]
        )

        df_resultado = (
            df_edy_devengamiento_por_cuota.alias("dpc")
            .join(
                df_edy_bs_planpagos,
                (col("pp.SALDO_JTS_OID") == col("dpc.SALDO_JTS_OID")) &
                (col("pp.C2300") == col("dpc.NRO_CUOTA")),
                "inner"
            )
            .filter(
                (col("dpc.FECHA") == lit(max_fecha)) &
                (col("dpc.NRO_CUOTA") > 0) 
            )
            .groupBy(
                col("dpc.SALDO_JTS_OID").alias("SALDOS_JTS_OID")
            )
            .agg(
                sum(
                    when(
                        (col("dpc.INT_DEVENGADO") - col("dpc.INT_PAGADO")) > 0,
                        col("dpc.INT_DEVENGADO") - col("dpc.INT_PAGADO")
                    ).otherwise(lit(0))
                ).alias("INTERES_DEVENGADO"),

                sum(col("pp.C2309")).alias("CAPITAL"),

                sum(col("pp.C2310")).alias("INTERES"),

                max(col("dpc.ESTADOATRASO")).alias("ESTADOATRASO")
            )
        )

        return df_resultado
    
    def get_edy_saldos_activos(self): #check
        """Obtiene los datos de la tabla saldos_activos"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_SALDOS_ACTIVOS}/{self.CONS_TABLE_SALDOS_ACTIVOS}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)

    def get_edy_saldos_pasivos(self): #check
        """Obtiene los datos de la tabla saldos_pasivos"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_SALDOS_PASIVOS}/{self.CONS_TABLE_SALDOS_PASIVOS}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)

    def get_stg_t_psje_venc_01(self): #check
        """Obtiene los datos de la tabla t_psje_venc_01"""

        df_edy_saldos = self.get_edy_saldos_activos().alias("s")
        df_edy_cr_categorias_cartera = self.get_edy_cr_categorias_cartera()
        df_edy_devengamiento_por_cuota = self.get_edy_devengamiento_por_cuota()
        df_edy_bs_historia_plazo = self.get_edy_bs_historia_plazo().alias("hp")
        
        df_cc_c = (
            df_edy_cr_categorias_cartera.alias("cc")
            .filter(col("cc.CODIGO_ATRASO_DESTINO") == "C")
            .select(
                col("cc.RUBRO_ORIGEN").alias("RUBRO_ORIGEN_C")
            )
            .dropDuplicates()
            .withColumn("EXISTE_CC_C", lit(1))
            .alias("cc_c")
        )

        df_cc_n = (
            df_edy_cr_categorias_cartera.alias("cc2")
            .filter(col("cc2.CODIGO_ATRASO_DESTINO") == "N")
            .groupBy(
                col("cc2.RUBRO_ORIGEN").alias("RUBRO_ORIGEN_N")
            )
            .agg(
                max(col("cc2.INCUMPLIMIENTOS")).alias("INCUMPLIMIENTOS")
            )
            .alias("cc_n")
        )

        df_devengamiento = (
            df_edy_devengamiento_por_cuota.alias("dpc")
            .filter(
                col("dpc.RUBRO_CONTABLE").like("14_5%")
            )
            .groupBy(
                col("dpc.SALDO_JTS_OID").alias("SALDO_JTS_OID_DP")
            )
            .agg(
                min(col("dpc.FECHA")).alias("MIN_FECHA_DEVENGAMIENTO")
            )
            .withColumn("EXISTE_DP", lit(1))
            .alias("dev")
        )

        df_historia_plazo = (
            df_edy_bs_historia_plazo
            .filter(
                (col("hp.TIPOMOV") == "R") &
                (col("hp.RUBROCONTABLE").like("14_5%"))
            )
            .groupBy(
                col("hp.SALDOS_JTS_OID").alias("SALDOS_JTS_OID_HP")
            )
            .agg(
                min(col("hp.FECHAVALOR")).alias("MIN_FECHA_HISTORIA_PLAZO")
            )
            .withColumn("EXISTE_HP", lit(1))
            .alias("hist")
        )

        df_resultado = (
            df_edy_saldos
            .join(
                df_cc_c,
                col("cc_c.RUBRO_ORIGEN_C") == col("s.C1692"),
                "left"
            )
            .join(
                df_cc_n,
                col("cc_n.RUBRO_ORIGEN_N") == col("s.C1692"),
                "left"
            )
            .join(
                df_devengamiento,
                col("dev.SALDO_JTS_OID_DP") == col("s.JTS_OID"),
                "left"
            )
            .join(
                df_historia_plazo,
                col("hist.SALDOS_JTS_OID_HP") == col("s.JTS_OID"),
                "left"
            )
            .select(
                col("s.JTS_OID"),

                when(
                    col("cc_c.EXISTE_CC_C") == 1,
                    when(
                        col("dev.EXISTE_DP") == 1,
                        col("dev.MIN_FECHA_DEVENGAMIENTO")
                    ).otherwise(
                        date_add(
                            col("s.C1628"),
                            coalesce(col("cc_n.INCUMPLIMIENTOS"), lit(0)).cast("int")
                        )
                    )
                ).otherwise(
                    when(
                        col("hist.EXISTE_HP") == 1,
                        col("hist.MIN_FECHA_HISTORIA_PLAZO")
                    ).otherwise(
                        date_add(
                            col("s.C1628"),
                            coalesce(col("cc_n.INCUMPLIMIENTOS"), lit(0)).cast("int")
                        )
                    )
                ).alias("FE_PSJE_VENC")
            )
        )

        return df_resultado

    def get_edy_pc_soliccastigo(self): #check
        """Obtiene los datos de la tabla pc_soliccastigo"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_PC_SOLICCASTIGO}/{self.CONS_TABLE_PC_SOLICCASTIGO}"
        return self.read_data(table_path).filter(col("tz_lock") == 0).select(
            col("FECCAST"),
            col("NROPRES")
        )

    def get_edy_tmp_saldos_01(self): #check
        """Obtiene los datos de la tabla tmp_saldos_01"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_TMP_SALDOS_01}/{self.CONS_TABLE_TMP_SALDOS_01}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)

    def get_edy_tmp_saldos_03(self): #check
        """Obtiene los datos de la tabla tmp_saldos_03"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_TMP_SALDOS_03}/{self.CONS_TABLE_TMP_SALDOS_03}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)

    def get_edy_tmp_saldos_04(self): #check
        """Obtiene los datos de la tabla tmp_saldos_04"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_TMP_SALDOS_04}/{self.CONS_TABLE_TMP_SALDOS_04}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)

    def get_stg_t_saldos(self, refresh=False):
        """Obtiene los datos de la tabla t_saldos"""

        if (
                not refresh
                and self.df_stg_t_saldos_activos is not None
                and self.df_stg_t_saldos_pasivos is not None
            ):
                return self.df_stg_t_saldos_activos, self.df_stg_t_saldos_pasivos


        df_edy_pc_soliccastigo = self.get_edy_pc_soliccastigo().alias("sc")
        df_edy_tmp_saldos_01 = self.get_edy_tmp_saldos_01().alias("s")
        df_edy_tmp_saldos_03 = self.get_edy_tmp_saldos_03().alias("s")
        df_edy_tmp_saldos_04 = self.get_edy_tmp_saldos_04().alias("s")

        df_sc_feccast = (
            df_edy_pc_soliccastigo
            .groupBy(col("sc.NROPRES").alias("NROPRES"))
            .agg(
                max(col("sc.FECCAST")).alias("FECCAST")
            )
            .alias("sc_max")
        )

        df_stg_t_saldos_1 = (
            df_edy_tmp_saldos_01
            .filter(
                col("s.TIP_PRO") == 5
            )
            .join(
                df_sc_feccast,
                col("sc_max.NROPRES") == col("s.CUENTA"),
                "left"
            )
            .select(
                col("s.CUENTA").alias("CUENTA"),
                col("s.MONEDA").alias("MONEDA"),
                col("s.SUCURSAL").alias("SUCURSAL"),
                col("s.PRODUCTO").alias("PRODUCTO"),
                col("s.OPERACION").alias("OPERACION"),
                col("s.ORDINAL").alias("ORDINAL"),
                col("s.JTS_OID").alias("JTS_OID"),
                col("s.NUM_CLI").alias("C1803"),
                col("s.FEC_APE").alias("C1620"),
                col("s.FEC_VAL_INI").alias("C1621"),
                col("s.FEC_VTO").alias("C1627"),
                col("s.PER_CUO").alias("C1642"),
                col("s.TOT_CUO").alias("C1644"),
                col("s.IMP_TAS_INT").alias("C1632"),
                col("s.ESTADO").alias("C1728"),
                col("s.FEC_PRO_VEN").alias("C1628"),
                col("s.IMP_MON_PRE").alias("C1601"),
                col("s.SAL_ACT").alias("C1604"),
                col("s.IMP_SAL_MN").alias("SALDOMN"),
                col("s.IMP_LIN_CRE").alias("C1661"),
                when(col("s.REFINANCIADO") == lit(" "), lit(None)).otherwise(col("s.REFINANCIADO")).alias("REFINANCIACION"),
                col("s.OFI_CUE").alias("C1670"),
                col("s.CRITERIO").alias("CRITERIO"),
                col("s.IMP_CUE_MOR").alias("C1711"),
                col("s.ENT_ACU_ICM").alias("ENTREGA_ACUENTA_ICMORA"),
                col("s.TAS_ICM").alias("TASA_ICMORA"),
                col("s.IMP_TAS_MOR").alias("C1633"),
                col("s.C1659").alias("C1659"),
                col("s.C1679").alias("C1679"),
                col("s.RUB_CON").alias("C1730"),
                col("s.IMP_INT_CON").alias("C1609"),
                col("s.IMP_INT_REA").alias("C1610"),
                col("s.IMP_BON_ADE").alias("C9661"),
                col("s.C6645").alias("C6645"),
                col("s.SOLICITUD").alias("C1704"),
                col("s.TIP_PRO").alias("C9314"),
                col("s.C1705").alias("C1705"),
                col("s.CUO_REA").alias("C1645"),
                col("sc_max.FECCAST").alias("FECCAST"),
                col("s.IMP_INT_LIQ").alias("C1608"),
                col("s.C2294").alias("C2294"),
                col("s.TIP_CAT").alias("C4280"),
                col("s.MB_REFERENCIA_GRUPAL").alias("MB_REFERENCIA_GRUPAL"),
                col("s.USUTOPAZ").alias("USUTOPAZ"),
                col("s.MB_CREDITO_VIGENTEO").alias("MB_CREDITO_VIGENTEO"),
                col("s.C1612").alias("C1612"),
                col("s.MB_CREDITO_MIGRADO").alias("MB_CREDITO_MIGRADO"),
                lit(0).alias("C1692"),
                col("s.REFINANCIADO").alias("REFINANCIADO"),
                col("s.INT_CATR").alias("INTE_COMP"),
                col("s.INT_MORA").alias("MORA_CONT"),
                lit(None).alias("C1686"),
                lit(None).alias("EMPLEADO"),
                lit(None).alias("C1651")
            )
        )

        key = self.key_aes

        df_stg_t_saldos_2 = (
            df_edy_tmp_saldos_03
            .filter(
                col("s.C9314") == 4
            )
            .select(
                aes_decrypt(
                        col("s.CUENTA_dac"),
                        lit(key),
                        lit("ECB"),
                        lit("PKCS")
                    ).cast("string")
                .alias("CUENTA"),
                col("s.MONEDA").alias("MONEDA"),
                col("s.SUCURSAL").alias("SUCURSAL"),
                col("s.PRODUCTO").alias("PRODUCTO"),
                col("s.OPERACION").alias("OPERACION"),
                col("s.ORDINAL").alias("ORDINAL"),
                col("s.JTS_OID").alias("JTS_OID"),
                col("s.C1803").alias("C1803"),
                lit(None).alias("C1620"),
                col("s.C1621").alias("C1621"),
                col("s.C1627").alias("C1627"),
                col("s.C1642").alias("C1642"),
                col("s.C1644").alias("C1644"),
                col("s.C1632").alias("C1632"),
                col("s.C1728").alias("C1728"),
                col("s.C1628").alias("C1628"),
                col("s.C1601").alias("C1601"),
                col("s.SALD_CTA").alias("C1604"),
                col("s.SALDOMN").alias("SALDOMN"),
                col("s.C1661").alias("C1661"),
                col("s.REFINANCIACION").alias("REFINANCIACION"),
                col("s.C1670").alias("C1670"),
                col("s.CRITERIO").alias("CRITERIO"),
                col("s.C1711").alias("C1711"),
                col("s.ENTREGA_ACUENTA_ICMORA").alias("ENTREGA_ACUENTA_ICMORA"),
                col("s.TASA_ICMORA").alias("TASA_ICMORA"),
                col("s.C1633").alias("C1633"),
                col("s.C1659").alias("C1659"),
                col("s.C1679").alias("C1679"),
                col("s.C1730").alias("C1730"),
                col("s.C1609").alias("C1609"),
                col("s.C1610").alias("C1610"),
                col("s.C9661").alias("C9661"),
                col("s.C6645").alias("C6645"),
                col("s.C1704").alias("C1704"),
                col("s.C9314").alias("C9314"),
                col("s.C1705").alias("C1705"),
                col("s.C1645").alias("C1645"),
                lit(" ").alias("FECCAST"),
                col("s.C1608").alias("C1608"),
                col("s.C2294").alias("C2294"),
                col("s.C4280").alias("C4280"),
                lit(0).alias("MB_REFERENCIA_GRUPAL"),
                lit(" ").alias("USUTOPAZ"),
                lit(" ").alias("MB_CREDITO_VIGENTEO"),
                lit(0).alias("C1612"),
                lit(" ").alias("MB_CREDITO_MIGRADO"),
                lit(0).alias("C1692"),
                lit(" ").alias("REFINANCIADO"),
                lit(0).alias("INTE_COMP"),
                lit(0).alias("MORA_CONT"),
                col("s.C1686").alias("C1686"),
                col("s.EMPLEADO").alias("EMPLEADO"),
                col("s.C1651").alias("C1651")
            )
        )

        df_stg_t_saldos_3 = (
            df_edy_tmp_saldos_04
            .filter(
                col("s.C9314").isin(2, 3)
            )
            .select(
                aes_decrypt(
                        col("s.CUENTA_dac"),
                        lit(key),
                        lit("ECB"),
                        lit("PKCS")
                    ).cast("string")
                .alias("CUENTA"),
                col("s.MONEDA").alias("MONEDA"),
                col("s.SUCURSAL").alias("SUCURSAL"),
                col("s.PRODUCTO").alias("PRODUCTO"),
                col("s.OPERACION").alias("OPERACION"),
                col("s.ORDINAL").alias("ORDINAL"),
                col("s.JTS_OID").alias("JTS_OID"),
                col("s.C1803").alias("C1803"),
                lit(None).alias("C1620"),
                col("s.C1621").alias("C1621"),
                col("s.C1627").alias("C1627"),
                col("s.C1642").alias("C1642"),
                col("s.C1644").alias("C1644"),
                col("s.C1632").alias("C1632"),
                col("s.C1728").alias("C1728"),
                col("s.C1628").alias("C1628"),
                col("s.C1601").alias("C1601"),
                col("s.C1604").alias("C1604"),
                col("s.SALDOMN").alias("SALDOMN"),
                col("s.C1661").alias("C1661"),
                col("s.REFINANCIACION").alias("REFINANCIACION"),
                col("s.C1670").alias("C1670"),
                col("s.CRITERIO").alias("CRITERIO"),
                col("s.C1711").alias("C1711"),
                col("s.ENTREGA_ACUENTA_ICMORA").alias("ENTREGA_ACUENTA_ICMORA"),
                col("s.TASA_ICMORA").alias("TASA_ICMORA"),
                col("s.C1633").alias("C1633"),
                col("s.C1659").alias("C1659"),
                col("s.C1679").alias("C1679"),
                col("s.RUB_CTBLE").alias("C1730"),
                col("s.C1609").alias("C1609"),
                col("s.C1610").alias("C1610"),
                col("s.C9661").alias("C9661"),
                col("s.C6645").alias("C6645"),
                col("s.C1704").alias("C1704"),
                col("s.C9314").alias("C9314"),
                col("s.C1705").alias("C1705"),
                col("s.C1645").alias("C1645"),
                lit(" ").alias("FECCAST"),
                col("s.C1608").alias("C1608"),
                col("s.C2294").alias("C2294"),
                col("s.C4280").alias("C4280"),
                lit(0).alias("MB_REFERENCIA_GRUPAL"),
                lit(" ").alias("USUTOPAZ"),
                lit(" ").alias("MB_CREDITO_VIGENTEO"),
                lit(0).alias("C1612"),
                lit(" ").alias("MB_CREDITO_MIGRADO"),
                lit(0).alias("C1692"),
                lit(" ").alias("REFINANCIADO"),
                lit(0).alias("INTE_COMP"),
                lit(0).alias("MORA_CONT"),
                col("s.C1686").alias("C1686"),
                col("s.EMPLEADO").alias("EMPLEADO"),
                col("s.COD_CAN").alias("C1651")
            )
        )

        df_stg_t_saldos_activos = df_stg_t_saldos_1
        df_stg_t_saldos_pasivos = df_stg_t_saldos_2.unionByName(df_stg_t_saldos_3)

        
        self.df_stg_t_saldos_activos = (
            df_stg_t_saldos_activos
            .persist(StorageLevel.MEMORY_ONLY)
        )

        self.df_stg_t_saldos_pasivos = (
            df_stg_t_saldos_pasivos
            .persist(StorageLevel.MEMORY_ONLY)
        )

        return self.df_stg_t_saldos_activos, self.df_stg_t_saldos_pasivos

    def get_stg_t_saldos_intereses_deven(self): #check
        """Obtiene los datos de la tabla t_saldos_intereses_deven"""

        df_edy_cl_clientes = self.get_edy_cl_clientes().alias("CALIFCLIENT")
        monto_interes_expr = (
            col("S.C1609") - (col("S.C1610") + col("S.C9661"))
        )

        monto_interes_positivo_expr = (
            when(
                monto_interes_expr > 0,
                monto_interes_expr
            ).otherwise(lit(0))
        )

        califalin_num_expr = (
            coalesce(
                col("CALIFCLIENT.CALIFALIN").cast("decimal(18,0)"),
                lit(0)
            )
        )

        df_stg_t_saldos_activos, df_stg_t_saldos_pasivos = self.get_stg_t_saldos()

        df_resultado = (
            df_stg_t_saldos_activos.alias("S")
            .join(
                df_edy_cl_clientes,
                (col("CALIFCLIENT.C0902") == col("S.C1803")) &
                (col("CALIFCLIENT.CALIFALIN") != lit(" ")),
                "left"
            )
            .filter(
                (col("S.C1604") < 0) &
                (col("S.C9314") == 5)
            )
            .groupBy(
                col("S.CUENTA").alias("CUENTA")
            )
            .agg(
                sum(
                    when(
                        (col("S.C1728") == "N") &
                        (califalin_num_expr <= 2),
                        monto_interes_positivo_expr
                    ).otherwise(lit(0))
                ).alias("MO_INTE_DEVE_VIGE"),

                sum(
                    when(
                        (col("S.C1728") != "N") |
                        (
                            (col("S.C1728") == "N") &
                            (califalin_num_expr >= 3)
                        ),
                        monto_interes_positivo_expr
                    ).otherwise(lit(0))
                ).alias("MO_INTE_DEVE_SUST")
            )
        )


        return df_resultado

    def get_stg_t_int_vigente(self): #validar data entry
        """Obtiene los datos de la tabla t_int_vigente"""

        # Tabla origen
        df = self.get_edy_devengamiento_por_cuota()

        # Obtener la fecha máxima (subquery)
        max_fecha = df.agg(max("FECHA").alias("max_fecha")).first()["max_fecha"]

        # Aplicar filtros y transformación
        df_result = (
            df.filter(
                (col("TZ_LOCK") == 0) &
                (col("FECHA") == lit(max_fecha)) &
                (col("NRO_CUOTA") < 0)
            )
            .select(
                col("SALDO_JTS_OID").alias("SALDOS_JTS_OID"),
                when(
                    (col("INT_DEVENGADO") - col("INT_PAGADO")) > 0,
                    col("INT_DEVENGADO") - col("INT_PAGADO")
                ).otherwise(lit(0)).alias("INTERES_VIGENTE")
            )
        )
    
        return df_result
    
    def get_stg_t_cr_lineacredito(self): #check
        """Obtiene los datos de la tabla t_cr_lineacredito"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CR_LINEACREDITO}/{self.CONS_TABLE_CR_LINEACREDITO}"
        return self.read_data(table_path).filter(col("tz_lock") == 0).select(
            col("NROLINEA").alias("NROLINEA"),
            col("AGENCIA").alias("AGENCIA"),
            col("CODUS").alias("CODUS"),
            col("CLTE").alias("CLTE"),
            col("ESTADO").alias("ESTADO"),
            col("PROD").alias("PROD"),
            col("ANCRED").alias("ANCRED"),
            col("CUOTA").alias("CUOTA"),
            col("MODDISP").alias("MODDISP"),
            col("MOTIVO").alias("MOTIVO"),
            col("FECCAMBEST").alias("FECCAMBEST"),
            col("FECAPER").alias("FECAPER"),
            col("FECREVAL").alias("FECREVAL"),
            col("CANCEL").alias("CANCEL"),
            col("MTODISP").alias("MTODISP"),
            col("MTOOT").alias("MTOOT"),
            col("MTOUTIL").alias("MTOUTIL"),
            col("PLAZO").alias("PLAZO"),
            col("PLAZDISP").alias("PLAZDISP"),
            col("FECVTO").alias("FECVTO"),
            col("NROSOL").alias("NROSOL")
        )

    def get_edy_sl_segurodesgravamen(self): #check
        """Obtiene los datos de la tabla sl_segurodesgravamen"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_SL_SEGURODESGRAVAMEN}/{self.CONS_TABLE_SL_SEGURODESGRAVAMEN}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_sl_solicitudcreditopersona(self): #check
        """Obtiene los datos de la tabla sl_solicitudcreditopersona"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_SL_SOLICITUDCREDITOPERSONA}/{self.CONS_TABLE_SL_SOLICITUDCREDITOPERSONA}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_sl_notasporsol(self): #check
        """Obtiene los datos de la tabla sl_notasporsol"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_SL_NOTASPORSOL}/{self.CONS_TABLE_SL_NOTASPORSOL}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_sl_solicitudcredito(self): #check
        """Obtiene los datos de la tabla sl_solicitudcredito"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_SL_SOLICITUDCREDITO}/{self.CONS_TABLE_SL_SOLICITUDCREDITO}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_cl_grupossociales(self): #check
        """Obtiene los datos de la tabla cl_grupossociales"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CL_GRUPOSSOCIALES}/{self.CONS_TABLE_CL_GRUPOSSOCIALES}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_sol_rel_sub_cre(self): #check
        """Obtiene los datos de la tabla sol_rel_sub_cre"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_SOL_REL_SUB_CRE}/{self.CONS_TABLE_SOL_REL_SUB_CRE}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_sl_criterios(self): #check
        """Obtiene los datos de la tabla sl_criterios"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_SL_CRITERIOS}/{self.CONS_TABLE_SL_CRITERIOS}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_sl_prop_credito(self): #check
        """Obtiene los datos de la tabla sl_prop_credito"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_SL_PROP_CREDITO}/{self.CONS_TABLE_SL_PROP_CREDITO}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_cre_pro_cre_rie(self): #check
        """Obtiene los datos de la tabla cre_pro_cre_rie"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CRE_PRO_CRE_RIE}/{self.CONS_TABLE_CRE_PRO_CRE_RIE}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)

    def get_stg_t_sl_solicitudcredito(self): #check
        """Obtiene los datos de la tabla t_sl_solicitudcredito"""

        df_edy_sl_segurodesgravamen = self.get_edy_sl_segurodesgravamen().alias("sd")
        df_edy_sl_solicitudcreditopersona = self.get_edy_sl_solicitudcreditopersona().alias("sp")
        df_edy_sl_notasporsol = self.get_edy_sl_notasporsol()
        df_edy_sl_solicitudcredito= self.get_edy_sl_solicitudcredito().alias("s")
        df_edy_cl_grupossociales = self.get_edy_cl_grupossociales().alias("g")
        df_edy_sol_rel_sub_cre = self.get_edy_sol_rel_sub_cre().alias("sr")
        df_edy_sl_criterios = self.get_edy_sl_criterios().alias("c")
        df_edy_sl_prop_credito = self.get_edy_sl_prop_credito().alias("spc")
        df_edy_cre_pro_cre_rie = self.get_edy_cre_pro_cre_rie().alias("cpc")

        subq1 = (
            df_edy_sl_segurodesgravamen
            .join(
                df_edy_sl_solicitudcreditopersona,
                (col("sd.NUM_SOLICITUD") == col("sp.C5080")) &
                (col("sd.TIPO_DOC") == col("sp.C5081")) &
                (col("sd.NRO_DOC_dac") == col("sp.C5082_dac")) &
                (col("sd.PAIS") == col("sp.C5083")) &
                (col("sp.C5084") == lit("T")),
                "inner"
            )
            .select(
                col("sd.COD_CIA_SEG").alias("COD_CIA_SEG"),
                col("sd.SEG_EXT").alias("SEG_EXT"),
                col("sd.NUM_SOLICITUD").alias("NUM_SOLICITUD")
            )
            .groupBy(
                "COD_CIA_SEG",
                "SEG_EXT",
                "NUM_SOLICITUD"
            )
            .agg(lit(1).alias("_tmp"))
            .drop("_tmp")
            .alias("subq1")
        )

        max_ordinal_notas = (
            df_edy_sl_notasporsol.alias("b")
            .groupBy(
                col("b.NROSOLICITUD").alias("NROSOLICITUD")
            )
            .agg(
                max(col("b.ORDINAL")).alias("MAX_ORDINAL")
            )
            .alias("mx")
        )

        subq2 = (
            df_edy_sl_notasporsol.alias("a")
            .join(
                max_ordinal_notas,
                (col("a.NROSOLICITUD") == col("mx.NROSOLICITUD")) &
                (col("a.ORDINAL") == col("mx.MAX_ORDINAL")),
                "inner"
            )
            .select(
                col("a.TIPONOTA").alias("TIPONOTA"),
                col("a.COMENTARIO").alias("COMENTARIO"),
                col("a.NROSOLICITUD").alias("NROSOLICITUD")
            )
            .groupBy(
                "TIPONOTA",
                "COMENTARIO",
                "NROSOLICITUD"
            )
            .agg(lit(1).alias("_tmp"))
            .drop("_tmp")
            .alias("subq2")
        )

        df_resultado = (
            df_edy_sl_solicitudcredito
            .join(
                df_edy_cl_grupossociales,
                (col("s.GRUPO") == col("g.C1371")) &
                (col("s.GRUPO") != 0),
                "left"
            )
            .join(
                df_edy_sol_rel_sub_cre,
                col("sr.NUM_SOL") == col("s.C5000"),
                "left"
            )
            .join(
                df_edy_sl_criterios,
                (col("c.SOLICITUD") == col("s.C5000")) &
                (col("c.CODITEM") == 2),
                "left"
            )
            .join(
                df_edy_sl_prop_credito,
                (col("spc.SOLICITUD") == col("s.C5000")) &
                (col("spc.ITEM") == 17) ,
                "left"
            )
            .join(
                df_edy_cre_pro_cre_rie,
                col("cpc.NUM_SOL") == col("s.C5000"),
                "left"
            )
            .join(
                subq1,
                col("subq1.NUM_SOLICITUD") == col("s.C5000"),
                "left"
            )
            .join(
                subq2,
                col("subq2.NROSOLICITUD") == col("s.C5000"),
                "left"
            )
            .select(
                col("s.C5000").alias("C5000"),
                col("s.GRUPO").alias("GRUPO"),
                col("s.C5061").alias("C5061"),
                col("s.C5004").alias("C5004"),
                col("s.C5074").alias("C5074"),
                col("s.C5044").alias("C5044"),
                col("s.C5261").alias("C5261"),

                when(col("g.C1371").isNotNull(), lit(1))
                .otherwise(lit(0))
                .alias("IN_GRUPAL"),

                col("s.FCREAC1ERET").alias("FCREAC1ERET"),
                col("s.C5062").alias("C5062"),
                col("s.C5036").alias("C5036"),
                col("s.C5035").alias("C5035"),
                col("s.C5008").alias("C5008"),
                col("s.MONEDA").alias("MONEDA"),
                col("s.C2320").alias("C2320"),
                col("s.C5001").alias("C5001"),
                col("s.C5006").alias("C5006"),
                col("s.C5187").alias("C5187"),
                col("s.C5063").alias("C5063"),
                col("s.C5002").alias("C5002"),
                col("s.C5053").alias("C5053"),
                col("s.FORMADESEMBOLSO").alias("FORMADESEMBOLSO"),
                col("s.MB_DESEMBOLSO_JTS_OID").alias("MB_DESEMBOLSO_JTS_OID"),
                col("s.TASA_OPTIMA").alias("TASA_OPTIMA"),
                col("s.SEGURO").alias("SEGURO"),
                col("s.VALOR_SEGURO").alias("VALOR_SEGURO"),

                coalesce(col("sr.COD_INV"), lit(0)).alias("COD_INV"),

                col("s.IDMETODOLOGIA").alias("IDMETODOLOGIA"),
                col("s.TIPSOL").alias("TIPSOL"),
                col("s.DIA_PAGO").alias("DIA_PAGO"),
                col("s.C5229").alias("C5229"),
                col("s.C5054").alias("C5054"),
                col("s.C5023").alias("C5023"),
                col("s.C5223").alias("C5223"),
                col("s.C5024").alias("C5024"),
                col("s.TASA_MINIMA").alias("TASA_MINIMA"),
                col("s.TASA_MAXIMA").alias("TASA_MAXIMA"),
                col("s.C5281").alias("C5281"),
                col("s.C5039").alias("C5039"),
                col("s.FLG_CON").alias("FLG_CON"),

                coalesce(col("c.RESULTADO"), lit(0)).alias("VL_ENDEUDAMIENTO"),

                when(
                    col("s.C5063") == 12,
                    col("subq2.TIPONOTA")
                ).alias("CO_MOTI_RECH"),

                when(
                    col("s.C5063") == 12,
                    col("subq2.COMENTARIO")
                ).otherwise(lit(" ")).alias("DE_COME_RECH"),

                coalesce(col("subq1.COD_CIA_SEG"), lit(0)).alias("CO_COMP_ASEG"),
                coalesce(col("subq1.SEG_EXT"), lit(" ")).alias("IN_SEGU_DESG_EXTE"),

                coalesce(
                    concat(
                        col("spc.DATO_dac"),
                        lit(" "),
                        col("cpc.DES_CRE_APO_CLI_dac")
                    ),
                    lit(" ")
                ).alias("DE_PROP_DEST"),

                col("s.C5060").alias("C5060"),

                # @001 columnas adicionales
                col("s.C5231").alias("C5231"),
                col("s.TIPO_CREDITO").alias("TIPO_CREDITO"),
                col("s.DEUDA_SOLICITUD").alias("DEUDA_SOLICITUD"),
                col("s.ACTECONINTERNA").alias("ACTECONINTERNA"),
                col("s.ADICIONALREFINANCIADO").alias("ADICIONALREFINANCIADO"),
                col("s.ALE_ANA_CAL").alias("ALE_ANA_CAL"),
                col("s.ALE_ANA_FEC").alias("ALE_ANA_FEC"),
                col("s.ALE_ANA_SUS").alias("ALE_ANA_SUS"),
                col("s.ALE_SUP_CAL").alias("ALE_SUP_CAL"),
                col("s.ALE_SUP_FEC").alias("ALE_SUP_FEC"),
                col("s.C2841").alias("C2841"),
                col("s.C4794").alias("C4794"),
                col("s.C5003").alias("C5003"),
                col("s.C5009").alias("C5009"),
                col("s.C5022").alias("C5022"),
                col("s.C5025").alias("C5025"),
                col("s.C5026").alias("C5026"),
                col("s.C5028").alias("C5028"),
                col("s.C5029").alias("C5029"),
                col("s.C5030").alias("C5030"),
                col("s.C5031").alias("C5031"),
                col("s.C5033_dac").alias("C5033"),
                col("s.C5034").alias("C5034"),
                col("s.C5037").alias("C5037"),
                col("s.C5038").alias("C5038"),
                col("s.C5040").alias("C5040"),
                col("s.C5041").alias("C5041"),
                col("s.C5042").alias("C5042"),
                col("s.C5043_dac").alias("C5043"),
                col("s.C5045").alias("C5045"),
                col("s.C5047").alias("C5047"),
                col("s.C5049").alias("C5049"),
                col("s.C5050").alias("C5050"),
                col("s.C5059").alias("C5059"),
                col("s.C5064").alias("C5064"),
                col("s.C5070").alias("C5070"),
                col("s.C5183").alias("C5183"),
                col("s.C5186").alias("C5186"),
                col("s.C5190").alias("C5190"),
                col("s.C5221").alias("C5221"),
                col("s.C5222").alias("C5222"),
                col("s.C5225").alias("C5225"),
                col("s.C5226").alias("C5226"),
                col("s.C5227").alias("C5227"),
                col("s.C5230").alias("C5230"),
                col("s.C5233").alias("C5233"),
                col("s.C5270").alias("C5270"),
                col("s.C5271_dac").alias("C5271"),
                col("s.C5272_dac").alias("C5272"),
                col("s.C5273").alias("C5273"),
                col("s.C5274_dac").alias("C5274"),
                col("s.C5275_dac").alias("C5275"),
                col("s.C5280").alias("C5280"),
                col("s.C5283").alias("C5283"),
                col("s.C6945").alias("C6945"),
                col("s.CAR_USU_DESEM").alias("CAR_USU_DESEM"),
                col("s.CAR_USU_DIS").alias("CAR_USU_DIS"),
                col("s.CODACTCIIU").alias("CODACTCIIU"),
                col("s.CODACTCONTR").alias("CODACTCONTR"),
                col("s.CODBANCODESEMBOLSO").alias("CODBANCODESEMBOLSO"),
                col("s.CONTCARTADESMX").alias("CONTCARTADESMX"),
                col("s.CON_CAR_DES_BAN").alias("CON_CAR_DES_BAN"),
                col("s.CTA_CTBLE_DES_BAN_dac").alias("CTA_CTBLE_DES_BAN"),
                col("s.CUENTADESEMBOLSO_dac").alias("CUENTADESEMBOLSO"),
                col("s.CUO_MAX_DIS").alias("CUO_MAX_DIS"),
                col("s.CUO_MAX_LIN").alias("CUO_MAX_LIN"),
                col("s.DEBITO_AUTOMATICO").alias("DEBITO_AUTOMATICO"),
                col("s.DEUDA_ACTUAL").alias("DEUDA_ACTUAL"),
                col("s.DEVOLUCION_SD").alias("DEVOLUCION_SD"),
                col("s.DIADEPAGO").alias("DIADEPAGO"),
                col("s.FECHINIACTICLI").alias("FECHINIACTICLI"),
                col("s.FEC_DER_HUB").alias("FEC_DER_HUB"),
                col("s.FLG_ADM_HUB").alias("FLG_ADM_HUB"),
                col("s.FORMAPROVISION").alias("FORMAPROVISION"),
                col("s.HOR_DER_HUB").alias("HOR_DER_HUB"),
                col("s.IDGRUPOCOBRO").alias("IDGRUPOCOBRO"),
                col("s.IMPCARTADESMB").alias("IMPCARTADESMB"),
                col("s.INI_USU_DESEM").alias("INI_USU_DESEM"),
                col("s.INT_CAPITALIZA").alias("INT_CAPITALIZA"),
                col("s.LINEAFINANC").alias("LINEAFINANC"),
                col("s.LUGARDESEMB").alias("LUGARDESEMB"),
                col("s.MAR_HUB").alias("MAR_HUB"),
                col("s.MB_COBRANZA_JTS_OID").alias("MB_COBRANZA_JTS_OID"),
                col("s.MB_PRECANCELACION_MIGRADO").alias("MB_PRECANCELACION_MIGRADO"),
                col("s.MB_VIVIENDA").alias("MB_VIVIENDA"),
                col("s.MICROSEGURO").alias("MICROSEGURO"),
                col("s.MIGRADO").alias("MIGRADO"),
                col("s.MODALIDADDESEMBOLSO").alias("MODALIDADDESEMBOLSO"),
                col("s.MODDISP").alias("MODDISP"),
                col("s.MONTOCONDONACION").alias("MONTOCONDONACION"),
                col("s.MONTO_A_DESCONTAR").alias("MONTO_A_DESCONTAR"),
                col("s.MONTO_FLAT").alias("MONTO_FLAT"),
                col("s.MONTO_ITF").alias("MONTO_ITF"),
                col("s.NODOCCONOCDEUDA").alias("NODOCCONOCDEUDA"),
                col("s.NOFIRMACONYGAR").alias("NOFIRMACONYGAR"),
                col("s.NOFIRMACONYTIT").alias("NOFIRMACONYTIT"),
                col("s.NROACTA").alias("NROACTA"),
                col("s.NROLIN").alias("NROLIN"),
                col("s.NROOPERACION").alias("NROOPERACION"),
                col("s.OPCIONESREFINANCIACION").alias("OPCIONESREFINANCIACION"),
                col("s.PAGOMINIMO").alias("PAGOMINIMO"),
                col("s.PARENTESCO").alias("PARENTESCO"),
                col("s.PLAZODISP").alias("PLAZODISP"),
                col("s.POLIZA").alias("POLIZA"),
                col("s.PORCENTAJEITF").alias("PORCENTAJEITF"),
                col("s.PORCENTAJE_FLAT").alias("PORCENTAJE_FLAT"),
                col("s.REVOLVENTE").alias("REVOLVENTE"),
                col("s.RIESGO_TOTAL").alias("RIESGO_TOTAL"),
                col("s.RIESGO_VINCULADO").alias("RIESGO_VINCULADO"),
                col("s.SALDO_MIBANCO").alias("SALDO_MIBANCO"),
                col("s.SEGMENTO_COMERCIAL").alias("SEGMENTO_COMERCIAL"),
                col("s.SIN_DESGRAVAMEN").alias("SIN_DESGRAVAMEN"),
                col("s.SOLICCABECERA").alias("SOLICCABECERA"),
                col("s.SUCBANCODESEMBOLSO").alias("SUCBANCODESEMBOLSO"),
                col("s.SUSTENTADO").alias("SUSTENTADO"),
                col("s.TCAMBIO").alias("TCAMBIO"),
                col("s.TIPOCALIFICAC").alias("TIPOCALIFICAC"),
                col("s.TIPOREVOLVENTE").alias("TIPOREVOLVENTE"),
                col("s.USUDESEMB").alias("USUDESEMB"),
                col("s.USU_MOD_CON").alias("USU_MOD_CON")
            )
        )

        return df_resultado
    
    def get_stg_t_sl_solicitudcredcancela(self): #check
        """Obtiene los datos de la tabla t_sl_solicitudcredcancela"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_SL_SOLICITUDCREDCANCELA}/{self.CONS_TABLE_SL_SOLICITUDCREDCANCELA}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0).select(
            col("CRE_NUMSOLICITUD").alias("CRE_NUMSOLICITUD"),
            col("CRE_SUCURSAL").alias("CRE_SUCURSAL"),
            col("CRE_PRODUCTO").alias("CRE_PRODUCTO"),
            col("CRE_CUENTA").alias("CRE_CUENTA"),
            col("CRE_MONEDA").alias("CRE_MONEDA"),
            col("CRE_OPERACION").alias("CRE_OPERACION"),
            col("CRE_ORDINAL").alias("CRE_ORDINAL"),
            col("CRE_MONTOCANC").alias("CRE_MONTOCANC"),
            col("CRE_MOTIVOCANC").alias("CRE_MOTIVOCANC"),
            col("CRE_IMPORTE").alias("CRE_IMPORTE")
        ) 

    def get_stg_t_bs_historia_plazo_slcred(self): #check
        """Obtiene los datos de la tabla t_bs_historia_plazo_slcred"""
        
        max_fecha = self.get_edy_bs_historia_plazo().agg(max("FECHAPROCESOMOV")).first()[0]

        return self.get_edy_bs_historia_plazo().alias("bshp").filter(
            (col("bshp.TIPOMOV").isin("A", "P")) &
            (to_date(col("bshp.FECHAPROCESOMOV")) == max_fecha)
        ).select(
            col("bshp.JTS_OID").alias("JTS_OID"),
            col("bshp.FECHAVALOR").alias("FECHAVALOR"),
            col("bshp.TIPOMOV").alias("TIPOMOV"),
            col("bshp.NROASIENTOMOV").alias("NROASIENTOMOV"),
            col("bshp.SUCURSALMOV").alias("SUCURSALMOV"),
            col("bshp.FECHAPROCESOMOV").alias("FECHAPROCESOMOV"),
            col("bshp.CAPITALPAGADO").alias("CAPITALPAGADO"),
            col("bshp.INTERESPAGADO").alias("INTERESPAGADO"),
            col("bshp.MORAPAGADA").alias("MORAPAGADA"),
            col("bshp.SALDOS_JTS_OID").alias("SALDOS_JTS_OID")
        )

    def get_edy_asientos(self): #check
        """Obtiene los datos de la tabla asientos"""

        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_ASIENTOS}/{self.CONS_TABLE_ASIENTOS}"

        df_base = self.read_data(table_path).filter(col("tz_lock") == 0)

        df_base = df_base.withColumn(
            "fechaproceso_date",
            to_date(col("fechaproceso"))
        )

        # Obtenemos las últimas 7 fechas distintas de ejecución
        df_ultimas_7_fechas = (
            df_base
            .select("fechaproceso_date")
            .dropDuplicates()
            .orderBy(col("fechaproceso_date").desc())
            .limit(7)
        )

        # Filtramos la tabla original usando esas 7 fechas
        df_filtrado = (
            df_base
            .join(
                df_ultimas_7_fechas,
                on="fechaproceso_date",
                how="inner"
            )
        )

        return df_filtrado

    def get_stg_t_asientos_06(self): #check
        """Obtiene los datos de la tabla t_asientos_06"""

        df_edy_asientos = self.get_edy_asientos()

        max_fecha = self.get_edy_asientos().agg(max("FECHAPROCESO")).first()[0]

        return df_edy_asientos.filter(
            (col("estado") == 77) &
            (to_date(col("FECHAPROCESO")) == max_fecha)
        ).select(
            col("FECHAPROCESO"),
            col("SUCURSAL"),
            col("ASIENTO"),
            col("OPERACION"),
            col("HORAFIN"),
            col("INIUSR"),
            col("HORAINICIO"),
            col("DESCRIPCION")
        )

    def get_stg_t_saldos_sl_solicitudcredito(self): #check
        """Obtiene los datos de la tabla t_saldos_sl_solicitudcredito"""

        df_stg_t_saldos_activos, df_stg_t_saldos_pasivos = self.get_stg_t_saldos()

        df_stg_t_saldos_activos_f = df_stg_t_saldos_activos.select(
            col("sucursal"),
            col("producto"),
            col("cuenta"),
            col("moneda"),
            col("operacion"),
            col("ordinal"),
            col("C1704"),
            col("JTS_OID"),
            col("C1661"),
            col("C9314"),
            col("C1604"),
            col("C1803"),
            col("C1621"),
            col("C1627"),
            col("C1642"),
            col("C1644"),
            col("C1632"),
            col("C1728"),
            col("C1628"),
            col("C1601"),
            col("SALDOMN"),
            col("REFINANCIACION"),
            col("C1670"),
            col("CRITERIO"),
            col("C1711"),
            col("ENTREGA_ACUENTA_ICMORA"),
            col("TASA_ICMORA"),
            col("C1633"),
            col("C1659"),
            col("C1679"),
            col("C1730"),
            col("C1609"),
            col("C1610"),
            col("C9661"),
            col("C6645"),
            col("C1705"),
            col("C1645"),
            col("FECCAST"),
            col("C1608"),
            col("C2294"),
            col("C4280"),
            col("MB_REFERENCIA_GRUPAL"),
            col("USUTOPAZ"),
            col("C1612"),
            col("MB_CREDITO_MIGRADO"),
            col("REFINANCIADO"),
            col("C1692"),
            col("INTE_COMP"),
            col("MORA_CONT"),
            col("C1620")
        )

        df_stg_t_saldos_pasivos_f = df_stg_t_saldos_pasivos.select(
            col("sucursal"),
            col("producto"),
            col("cuenta"),
            col("moneda"),
            col("operacion"),
            col("ordinal"),
            col("C1704"),
            col("JTS_OID"),
            col("C1661"),
            col("C9314"),
            col("C1604"),
            col("C1803"),
            col("C1621"),
            col("C1627"),
            col("C1642"),
            col("C1644"),
            col("C1632"),
            col("C1728"),
            col("C1628"),
            col("C1601"),
            col("SALDOMN"),
            col("REFINANCIACION"),
            col("C1670"),
            col("CRITERIO"),
            col("C1711"),
            col("ENTREGA_ACUENTA_ICMORA"),
            col("TASA_ICMORA"),
            col("C1633"),
            col("C1659"),
            col("C1679"),
            col("C1730"),
            col("C1609"),
            col("C1610"),
            col("C9661"),
            col("C6645"),
            col("C1705"),
            col("C1645"),
            col("FECCAST"),
            col("C1608"),
            col("C2294"),
            col("C4280"),
            col("MB_REFERENCIA_GRUPAL"),
            col("USUTOPAZ"),
            col("C1612"),
            col("MB_CREDITO_MIGRADO"),
            col("REFINANCIADO"),
            col("C1692"),
            col("INTE_COMP"),
            col("MORA_CONT"),
            col("C1620")
        )

        saldos_unidos = df_stg_t_saldos_activos_f.union(df_stg_t_saldos_pasivos_f)

        scc = self.get_stg_t_sl_solicitudcredcancela()
        saldos = df_stg_t_saldos_activos_f

        solcred = self.get_stg_t_sl_solicitudcredito().alias("solcred")

        hp = (
            self.get_stg_t_bs_historia_plazo_slcred()
            .filter(col("TIPOMOV") == "A")
            .alias("hp")
        )
        asientos = self.get_stg_t_asientos_06().alias("a")

        subq1 = (
            scc.alias("scc")
            .join(
                saldos_unidos.alias("s2"),
                (col("scc.CRE_SUCURSAL") == col("s2.SUCURSAL"))
                & (col("scc.CRE_PRODUCTO") == col("s2.PRODUCTO"))
                & (col("scc.CRE_CUENTA") == col("s2.CUENTA"))
                & (col("scc.CRE_MONEDA") == col("s2.MONEDA"))
                & (col("scc.CRE_OPERACION") == col("s2.OPERACION"))
                & (col("scc.CRE_ORDINAL") == col("s2.ORDINAL")),
                "inner"
            )
            .join(
                self.get_stg_t_sl_solicitudcredito().alias("sol"),
                col("sol.C5000") == col("s2.C1704"),
                "inner"
            )
            .select(
                col("sol.C5061").alias("SUBQ1_C5061"),
                col("sol.C5062").alias("SUBQ1_C5062"),
                col("sol.C5000").alias("SUBQ1_C5000")
            )
            .dropDuplicates()
            .alias("subq1")
        )

        subq2_max = (
            scc.filter(col("CRE_MOTIVOCANC") == "P")
            .groupBy("CRE_NUMSOLICITUD")
            .agg(max("CRE_MONTOCANC").alias("MAX_CRE_MONTOCANC"))
        )

        subq2 = (
            scc.filter(col("CRE_MOTIVOCANC") == "P").alias("scc_p")
            .join(
                subq2_max.alias("mx_p"),
                (col("scc_p.CRE_NUMSOLICITUD") == col("mx_p.CRE_NUMSOLICITUD"))
                & (col("scc_p.CRE_MONTOCANC") == col("mx_p.MAX_CRE_MONTOCANC")),
                "inner"
            )
            .select(
                col("scc_p.CRE_PRODUCTO").alias("SUBQ2_CRE_PRODUCTO"),
                col("scc_p.CRE_NUMSOLICITUD").alias("SUBQ2_CRE_NUMSOLICITUD"),
                col("scc_p.CRE_MONTOCANC").alias("SUBQ2_CRE_MONTOCANC")
            )
            .alias("subq2")
        )

        subq3_max = (
            scc.filter(col("CRE_MOTIVOCANC") == "R")
            .groupBy("CRE_NUMSOLICITUD")
            .agg(max("CRE_MONTOCANC").alias("MAX_CRE_MONTOCANC"))
        )

        subq3 = (
            scc.filter(col("CRE_MOTIVOCANC") == "R").alias("scc_r")
            .join(
                subq3_max.alias("mx_r"),
                (col("scc_r.CRE_NUMSOLICITUD") == col("mx_r.CRE_NUMSOLICITUD"))
                & (col("scc_r.CRE_MONTOCANC") == col("mx_r.MAX_CRE_MONTOCANC")),
                "inner"
            )
            .select(
                col("scc_r.CRE_PRODUCTO").alias("SUBQ3_CRE_PRODUCTO"),
                col("scc_r.CRE_NUMSOLICITUD").alias("SUBQ3_CRE_NUMSOLICITUD"),
                col("scc_r.CRE_MONTOCANC").alias("SUBQ3_CRE_MONTOCANC")
            )
            .alias("subq3")
        )

        linea_sol = (
            self.get_stg_t_cr_lineacredito().alias("linea")
            .join(
                self.get_stg_t_sl_solicitudcredito().alias("sol_linea"),
                col("linea.NROSOL") == col("sol_linea.C5000"),
                "inner"
            )
            .select(
                col("linea.NROLINEA").alias("LINEA_NROLINEA"),
                col("sol_linea.C5061").alias("LINEA_C5061"),
                col("sol_linea.C5062").alias("LINEA_C5062")
            )
            .alias("linea_sol")
        )

        saldos_base_df = (
            saldos.alias("saldos")
            .join(solcred, col("saldos.C1704") == col("solcred.C5000"), "inner")
            .join(hp, col("hp.SALDOS_JTS_OID") == col("saldos.JTS_OID"), "left")
            .join(
                asientos,
                (col("a.ASIENTO") == col("hp.NROASIENTOMOV"))
                & (col("a.SUCURSAL") == col("hp.SUCURSALMOV"))
                & (col("a.FECHAPROCESO") == col("hp.FECHAPROCESOMOV")),
                "left"
            )
            .join(subq1, col("subq1.SUBQ1_C5000") == col("saldos.C1704"), "left")
            .join(subq2, col("subq2.SUBQ2_CRE_NUMSOLICITUD") == col("saldos.C1704"), "left")
            .join(subq3, col("subq3.SUBQ3_CRE_NUMSOLICITUD") == col("saldos.C1704"), "left")
            .join(linea_sol, col("linea_sol.LINEA_NROLINEA") == col("saldos.C1661"), "left")
            .filter((col("saldos.C9314") == 5) & (col("saldos.C1604") < 0))
            .select(
                col("saldos.CUENTA").alias("CUENTA"),
                col("saldos.MONEDA").alias("MONEDA"),
                col("saldos.SUCURSAL").alias("SUCURSAL"),
                col("saldos.PRODUCTO").alias("PRODUCTO"),
                col("saldos.OPERACION").alias("OPERACION"),
                col("saldos.ORDINAL").alias("ORDINAL"),
                col("saldos.JTS_OID").alias("JTS_OID"),
                col("saldos.C1803").alias("C1803"),
                col("saldos.C1621").alias("C1621"),
                col("saldos.C1627").alias("C1627"),
                col("saldos.C1642").alias("C1642"),
                col("saldos.C1644").alias("C1644"),
                col("saldos.C1632").alias("C1632"),
                col("saldos.C1728").alias("C1728"),
                col("saldos.C1628").alias("C1628"),
                col("saldos.C1601").alias("C1601"),
                col("saldos.C1604").alias("C1604"),
                col("saldos.SALDOMN").alias("SALDOMN"),
                col("saldos.C1661").alias("C1661"),
                col("saldos.REFINANCIACION").alias("REFINANCIACION"),
                col("saldos.C1670").alias("C1670"),
                col("saldos.CRITERIO").alias("CRITERIO"),
                col("saldos.C1711").alias("C1711"),
                col("saldos.ENTREGA_ACUENTA_ICMORA").alias("ENTREGA_ACUENTA_ICMORA"),
                col("saldos.TASA_ICMORA").alias("TASA_ICMORA"),
                col("saldos.C1633").alias("C1633"),
                col("saldos.C1659").alias("C1659"),
                col("saldos.C1679").alias("C1679"),
                col("saldos.C1730").alias("C1730"),
                col("saldos.C1609").alias("C1609"),
                col("saldos.C1610").alias("C1610"),
                col("saldos.C9661").alias("C9661"),
                col("saldos.C6645").alias("C6645"),
                col("saldos.C1704").alias("C1704"),
                col("saldos.C9314").alias("C9314"),
                col("saldos.C1705").alias("C1705"),
                col("saldos.C1645").alias("C1645"),
                col("saldos.FECCAST").alias("FECCAST"),
                col("saldos.C1608").alias("C1608"),
                col("saldos.C2294").alias("C2294"),
                col("saldos.C4280").alias("C4280"),
                col("saldos.MB_REFERENCIA_GRUPAL").alias("MB_REFERENCIA_GRUPAL"),
                col("saldos.USUTOPAZ").alias("USUTOPAZ"),
                col("saldos.C1612").alias("C1612"),
                col("saldos.MB_CREDITO_MIGRADO").alias("MB_CREDITO_MIGRADO"),
                col("saldos.REFINANCIADO").alias("REFINANCIADO"),
                col("saldos.C1692").alias("C1692"),
                col("solcred.C5000").alias("C5000"),
                col("solcred.GRUPO").alias("GRUPO"),
                col("solcred.C5061").alias("C5061"),
                col("solcred.C5004").alias("C5004"),
                col("solcred.C5074").alias("C5074"),
                col("solcred.C5044").alias("C5044"),
                col("solcred.C5261").alias("C5261"),
                col("solcred.IN_GRUPAL").alias("IN_GRUPAL"),
                col("solcred.FCREAC1ERET").alias("FCREAC1ERET"),
                col("solcred.C5062").alias("C5062"),
                col("solcred.C5036").alias("C5036"),
                col("solcred.C5035").alias("C5035"),
                col("solcred.C5008").alias("C5008"),
                col("solcred.C2320").alias("C2320"),
                col("solcred.C5001").alias("C5001"),
                col("solcred.C5006").alias("C5006"),
                col("solcred.FORMADESEMBOLSO").alias("FORMADESEMBOLSO"),
                col("solcred.MB_DESEMBOLSO_JTS_OID").alias("MB_DESEMBOLSO_JTS_OID"),
                col("solcred.TASA_OPTIMA").alias("TASA_OPTIMA"),
                col("solcred.SEGURO").alias("SEGURO"),
                col("solcred.VALOR_SEGURO").alias("VALOR_SEGURO"),
                col("solcred.COD_INV").alias("COD_INV"),
                col("solcred.CO_COMP_ASEG").alias("CO_COMP_ASEG"),
                col("solcred.IN_SEGU_DESG_EXTE").alias("IN_SEGU_DESG_EXTE"),
                when(
                    col("solcred.C5004") == "PRESTAMO PRE-PAGO",
                    col("subq1.SUBQ1_C5061")
                ).when(
                    col("saldos.C1661") > 0,
                    col("solcred.C5061")
                ).otherwise(col("solcred.C5061")).alias("USU_EVAL"),
                when(
                    col("solcred.C5004") == "PRESTAMO PRE-PAGO",
                    col("subq1.SUBQ1_C5062")
                ).when(
                    col("saldos.C1661") > 0,
                    col("solcred.C5006")
                ).otherwise(col("solcred.C5062")).alias("FECHA_EVAL"),
                when(
                    col("solcred.C5004") == "PRESTAMO PRE-PAGO",
                    col("subq1.SUBQ1_C5061")
                ).when(
                    col("saldos.C1661") > 0,
                    col("linea_sol.LINEA_C5061")
                ).otherwise(col("solcred.C5061")).alias("USU_ORIGINAL"),
                when(
                    col("solcred.C5004") == "PRESTAMO PRE-PAGO",
                    col("subq1.SUBQ1_C5062")
                ).when(
                    col("saldos.C1661") > 0,
                    col("linea_sol.LINEA_C5062")
                ).otherwise(col("solcred.C5062")).alias("FECHA_ORIGINAL"),
                col("solcred.TASA_MINIMA").alias("TASA_MINIMA"),
                col("solcred.TASA_MAXIMA").alias("TASA_MAXIMA"),
                when(
                    col("solcred.C5004") == "PRESTAMO PRE-PAGO",
                    col("subq2.SUBQ2_CRE_PRODUCTO")
                ).when(
                    col("solcred.C5004") == "REFINANCIADO",
                    col("subq3.SUBQ3_CRE_PRODUCTO")
                ).otherwise(lit(0)).alias("CO_PROD_ORIG"),
                coalesce(col("a.OPERACION"), lit(0)).alias("CO_OPERATIVA"),
                col("saldos.INTE_COMP").alias("INTE_COMP"),
                col("saldos.MORA_CONT").alias("MORA_CONT"),
                col("saldos.C1620").alias("C1620"),
                col("solcred.C5037").alias("C5037")
            )
        )

        return saldos_base_df
    
    def get_stg_t_tasas_tea(self): #check
        """Obtiene los datos de la tabla t_tasas_tea"""

        saldos_sol_creds = self.get_stg_t_saldos_sl_solicitudcredito().alias("SALDOS_SOL_CREDS")
        prod = self.get_stg_t_co_productos().alias("PROD")

        df_result = (
            saldos_sol_creds
            .join(
                prod,
                col("PROD.C6250") == col("SALDOS_SOL_CREDS.PRODUCTO"),
                "left"
            )
            .select(
                col("SALDOS_SOL_CREDS.JTS_OID").alias("SALDOS_JTS_OID"),

                round(
                    when(
                        col("PROD.C6253") == lit("E"),
                        col("SALDOS_SOL_CREDS.C1632")
                    )
                    .when(
                        col("PROD.C6253") == lit("M"),
                        (
                            (
                                pow(
                                    1 + ((col("SALDOS_SOL_CREDS.C1632") / 12) / 100),
                                    12
                                )
                            ) - 1
                        ) * 100
                    )
                    .when(
                        col("PROD.C6253") == lit("N"),
                        col("SALDOS_SOL_CREDS.C1632")
                    ),
                    7
                ).alias("VL_TEA")
            )
        )


        return df_result

    def get_stg_tmp_egp_1(self, refresh=False):
        """Obtiene los datos de la tabla tmp_egp_1"""

        if (
                not refresh
                and self.df_stg_tmp_egp_1 is not None
            ):
                return self.df_stg_tmp_egp_1

        fe_proceso = self.FEC_PROCESO

        saldos = self.get_stg_t_saldos_sl_solicitudcredito()

        tea = (
            self.get_stg_t_tasas_tea()
            .select("SALDOS_JTS_OID", "VL_TEA")
            .alias("tea")
        )

        bshp = (
            self.get_stg_t_bs_historia_plazo_summary()
            .select("SALDOS_JTS_OID", col("FECHAVALOR").alias("FE_ULTI_PAGO"))
            .alias("bshp")
        )
        pv = (
            self.get_stg_t_psje_venc_01()
            .select(col("JTS_OID").alias("PV_JTS_OID"), "FE_PSJE_VENC")
            .alias("pv")
        )
        dev_venc = (
            self.get_stg_t_deveng_por_cta_venc()
            .select("SALDOS_JTS_OID", "CAPITAL", "INTERES_DEVENGADO")
            .alias("dev_venc")
        )
        int_vig = (
            self.get_stg_t_int_vigente()
            .select("SALDOS_JTS_OID", "INTERES_VIGENTE")
            .alias("int_vig")
        )
        int_deven = (
            self.get_stg_t_saldos_intereses_deven()
            .select("CUENTA", "MO_INTE_DEVE_VIGE", "MO_INTE_DEVE_SUST")
            .alias("int_deven")
        )
        linea_cred = (
            self.get_stg_t_cr_lineacredito()
            .select(
                col("NROLINEA").alias("LINEA_NROLINEA"),
                "MTODISP",
                "FECVTO"
            )
            .alias("linea_cred")
        )

        saldos_sol_creds_df = (
            saldos.alias("saldos")
            .join(tea, col("saldos.JTS_OID") == col("tea.SALDOS_JTS_OID"), "left")
            .join(bshp, col("saldos.JTS_OID") == col("bshp.SALDOS_JTS_OID"), "left")
            .join(pv, col("saldos.JTS_OID") == col("pv.PV_JTS_OID"), "left")
            .join(dev_venc, col("saldos.JTS_OID") == col("dev_venc.SALDOS_JTS_OID"), "left")
            .join(int_vig, col("saldos.JTS_OID") == col("int_vig.SALDOS_JTS_OID"), "left")
            .join(int_deven, col("saldos.CUENTA") == col("int_deven.CUENTA"), "left")
            .join(linea_cred, col("saldos.C1661") == col("linea_cred.LINEA_NROLINEA"), "left")
            .select(
                col("saldos.JTS_OID").alias("SALDOS_JTS_OID"),
                (year(lit(fe_proceso)) * 100 + month(lit(fe_proceso))).alias("NU_PERI_MES"),
                lit(fe_proceso).alias("FE_SALDO"),
                lit(fe_proceso).alias("FE_PROCESO"),
                col("saldos.CUENTA").alias("NU_PRESTAMO"),
                col("saldos.C1803").alias("CO_CLIENTE"),
                when(col("saldos.CUENTA").isin(100862267, 100179589, 100205639, 100220139), lit(1)).otherwise(lit(0)).alias("IN_SIN_HIPO_INSC"),
                when(
                    col("saldos.C1642") == 0,
                    datediff(col("saldos.C1627"), col("saldos.C1621"))
                ).otherwise(col("saldos.C1644") * col("saldos.C1642")).alias("NU_PLAZ_DIAS"),
                col("tea.VL_TEA").alias("VL_TEA"),
                col("saldos.C1627").alias("FE_VENCIMIENTO"),
                col("saldos.C1621").alias("FE_DESEMBOLSO"),
                col("bshp.FE_ULTI_PAGO").alias("FE_ULTI_PAGO"),
                col("pv.FE_PSJE_VENC").alias("FE_PSJE_VENC"),
                when(col("saldos.C1728") == "C", col("saldos.FECCAST")).alias("FE_CASTIGO"),
                when(datediff(lit(fe_proceso), col("saldos.C1628")) < 0, lit(0)).otherwise(datediff(lit(fe_proceso), col("saldos.C1628"))).alias("NU_DIAS_ATRA"),
                coalesce(col("saldos.C1601"), lit(0)).alias("MO_DESEMBOLSO"),
                coalesce(
                    when(
                        col("saldos.C4280") != "C",
                        when(col("saldos.C1728") != "N", col("saldos.C1604") * lit(-1)).otherwise(lit(0))
                    ).otherwise(col("dev_venc.CAPITAL")),
                    lit(0)
                ).alias("MO_SALD_VENC"),
                coalesce(
                    when(
                        col("saldos.C4280") != "C",
                        when(col("saldos.C1728") == "N", col("saldos.C1604") * lit(-1)).otherwise(lit(0))
                    ).otherwise((col("saldos.C1604") * lit(-1)) - col("dev_venc.CAPITAL")),
                    lit(0)
                ).alias("MO_SALD_VIGE"),
                coalesce(col("saldos.C1604") * lit(-1), lit(0)).alias("MO_SALD_ACTU"),
                coalesce(
                    when(
                        col("saldos.C4280") != "C",
                        when(col("saldos.C1728") != "C", coalesce(col("int_deven.MO_INTE_DEVE_VIGE"), lit(0))).otherwise(lit(0))
                    ).otherwise(lit(0)),
                    lit(0)
                ).alias("MO_INTE_DEVE_VIGE"),
                coalesce(
                    when(
                        col("saldos.C4280") != "C",
                        when(col("saldos.C1728") != "C", coalesce(col("int_deven.MO_INTE_DEVE_SUST"), lit(0))).otherwise(col("saldos.C1608"))
                    ).otherwise(
                        coalesce(col("dev_venc.INTERES_DEVENGADO"), lit(0)) + coalesce(col("int_vig.INTERES_VIGENTE"), lit(0))
                    ),
                    lit(0)
                ).alias("MO_INTE_DEVE_SUST"),
                col("saldos.C1645").alias("NU_CUOT_PAGA"),
                coalesce(col("saldos.SALDOMN") * lit(-1), lit(0)).alias("MO_SALD_CAPI_MN"),
                col("saldos.C1661").alias("NU_LINE_CRED"),
                coalesce(col("linea_cred.MTODISP"), lit(0)).alias("MO_SALD_DISP_LINE"),
                col("linea_cred.FECVTO").alias("FE_VENC_LINE"),
                when(col("saldos.REFINANCIACION").isin("R"), lit(1)).otherwise(lit(0)).alias("IN_REFINANCIADO"),
                col("saldos.C5061").alias("CO_ANAL_EVAL"),
                col("saldos.C5036").alias("MO_APROBADO"),
                col("saldos.C1632").alias("VL_TASA"),
                col("saldos.FORMADESEMBOLSO").alias("CO_FORM_DESE"),
                col("saldos.MB_DESEMBOLSO_JTS_OID").alias("NU_CUEN_PASI"),
                col("saldos.TASA_OPTIMA").alias("VL_TASA_OPTI"),
                col("saldos.SEGURO").alias("IN_SEGU_DESG"),
                col("saldos.VALOR_SEGURO").alias("VL_DESGRAVAMEN"),
                col("saldos.COD_INV").alias("CO_INVERSION"),
                col("saldos.C1612").alias("MO_CUOTA"),
                col("saldos.CO_COMP_ASEG").alias("CO_COMP_ASEG"),
                col("saldos.IN_SEGU_DESG_EXTE").alias("IN_SEGU_DESG_EXTE"),
                col("saldos.C1628").alias("FE_PRIM_VENC"),
                col("saldos.CO_PROD_ORIG").alias("CO_PROD_ORIG"),
                when(col("saldos.MB_CREDITO_MIGRADO") == "S", lit(1)).otherwise(lit(0)).alias("IN_MIGRADO"),
                col("saldos.CO_OPERATIVA").alias("CO_OPERATIVA"),
                trim(col("saldos.REFINANCIACION")).alias("CO_REFINANCIADO"),
                col("saldos.C1620").alias("C1620"),
                col("saldos.C5037").alias("MO_CUOT_APRO_SOLI")
            )
        )

        self.df_stg_tmp_egp_1 = (
            saldos_sol_creds_df
            .persist(StorageLevel.MEMORY_ONLY)
        )

        return self.df_stg_tmp_egp_1

    def get_ods_hd_puesto_cargo_analista(self): #check
        """Obtiene los datos de la tabla hd_puesto_cargo_analista"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_HD_PUESTO_CARGO_ANALISTA}/{self.CONS_TABLE_HD_PUESTO_CARGO_ANALISTA}"
        )
        return self.read_data(table_path)
    
    def get_edy_cl_clientpersona(self): #check
        """Obtiene los datos de la tabla cl_clientpersona"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CL_CLIENTPERSONA}/{self.CONS_TABLE_CL_CLIENTPERSONA}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_rc_basenegmotivo(self): #check
        """Obtiene los datos de la tabla rc_basenegmotivo"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_RC_BASENEGMOTIVO}/{self.CONS_TABLE_RC_BASENEGMOTIVO}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_stg_t_cl_clientpersona(self): #check
        """Obtiene los datos de la tabla t_cl_clientpersona"""
    
        df_edy_cl_clientpersona = self.get_edy_cl_clientpersona().alias("cp")
        df_edy_rc_basenegmotivo = self.get_edy_rc_basenegmotivo().alias("a")

        rc_base = (
            df_edy_rc_basenegmotivo
            .select(
                col("a.TIPODOC").alias("TIPODOC"),
                col("a.NRODOCUMENTO_dac").alias("NRODOCUMENTO"),
                col("a.CODPAIS").alias("CODPAIS"),
                col("a.FECHAINGRESO").alias("FECHAINGRESO"),
                col("a.HORAINGRESO").alias("HORAINGRESO"),
                col("a.ESTADO").alias("ESTADO")
            )
            .alias("rc_base")
        )

        rc_max_fecha = (
            rc_base
            .groupBy(
                "TIPODOC",
                "NRODOCUMENTO",
                "CODPAIS"
            )
            .agg(
                max("FECHAINGRESO").alias("MAX_FECHAINGRESO")
            )
            .alias("rc_max_fecha")
        )

        rc_fecha_max = (
            rc_base
            .join(
                rc_max_fecha,
                (col("rc_base.TIPODOC") == col("rc_max_fecha.TIPODOC")) &
                (col("rc_base.NRODOCUMENTO") == col("rc_max_fecha.NRODOCUMENTO")) &
                (col("rc_base.CODPAIS") == col("rc_max_fecha.CODPAIS")) &
                (col("rc_base.FECHAINGRESO") == col("rc_max_fecha.MAX_FECHAINGRESO")),
                "inner"
            )
            .select(
                col("rc_base.TIPODOC").alias("TIPODOC"),
                col("rc_base.NRODOCUMENTO").alias("NRODOCUMENTO"),
                col("rc_base.CODPAIS").alias("CODPAIS"),
                col("rc_base.FECHAINGRESO").alias("FECHAINGRESO"),
                col("rc_base.HORAINGRESO").alias("HORAINGRESO"),
                col("rc_base.ESTADO").alias("ESTADO")
            )
            .alias("rc_fecha_max")
        )

        rc_max_hora = (
            rc_fecha_max
            .groupBy(
                "TIPODOC",
                "NRODOCUMENTO",
                "CODPAIS",
                "FECHAINGRESO"
            )
            .agg(
                max("HORAINGRESO").alias("MAX_HORAINGRESO")
            )
            .alias("rc_max_hora")
        )

        rc = (
            rc_fecha_max
            .join(
                rc_max_hora,
                (col("rc_fecha_max.TIPODOC") == col("rc_max_hora.TIPODOC")) &
                (col("rc_fecha_max.NRODOCUMENTO") == col("rc_max_hora.NRODOCUMENTO")) &
                (col("rc_fecha_max.CODPAIS") == col("rc_max_hora.CODPAIS")) &
                (col("rc_fecha_max.FECHAINGRESO") == col("rc_max_hora.FECHAINGRESO")) &
                (col("rc_fecha_max.HORAINGRESO") == col("rc_max_hora.MAX_HORAINGRESO")),
                "inner"
            )
            .groupBy(
                col("rc_fecha_max.TIPODOC").alias("TIPODOC"),
                col("rc_fecha_max.NRODOCUMENTO").alias("NRODOCUMENTO"),
                col("rc_fecha_max.CODPAIS").alias("CODPAIS")
            )
            .agg(
                max(col("rc_fecha_max.ESTADO")).alias("ESTADO")
            )
            .alias("rc")
        )

        df_resultado = (
            df_edy_cl_clientpersona
            .join(
                rc,
                (col("cp.C1431") == col("rc.TIPODOC")) &
                (col("cp.C1432_DAC") == col("rc.NRODOCUMENTO")) &
                (col("cp.C1435") == col("rc.CODPAIS")),
                "left"
            )
            .filter(
                col("cp.C1433") == lit("T")
            )
            .select(
                col("cp.C1430").alias("C1430"),
                col("cp.C1437").alias("C1437"),
                col("cp.C1431").alias("C1431"),
                col("cp.C1432_DAC").alias("C1432"),
                col("cp.C1435").alias("C1435"),
                coalesce(col("rc.ESTADO"), lit(0)).alias("ESTADO")
            )
        )

        return df_resultado

    def get_stg_t_cl_personasfisicas(self): #check
        """Obtiene los datos de la tabla t_cl_personasfisicas"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CL_PERSONASFISICAS}/{self.CONS_TABLE_CL_PERSONASFISICAS}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0).alias("pf").select(
            col("pf.C1401").alias("C1401"),
            col("pf.C1402_dac").alias("C1402"),
            col("pf.C1403").alias("C1403"),
            col("pf.C1404_dac").alias("C1404"),
            col("pf.C1405_dac").alias("C1405"),
            col("pf.C1406_dac").alias("C1406"),
            col("pf.C1407_dac").alias("C1407"),
            col("pf.C1409").alias("C1409"),
            col("pf.C8666").alias("C8666"),
            col("pf.C1410").alias("C1410"),
            col("pf.C1411").alias("C1411"),
            col("pf.C1451").alias("C1451"),
            col("pf.C1499").alias("C1499"),
            col("pf.C1448").alias("C1448"),
            col("pf.C1449").alias("C1449"),
            col("pf.C1599").alias("C1599"),
            col("pf.MIGRADO").alias("MIGRADO"),
            col("pf.DIRECCION").alias("DIRECCION"),
            col("pf.C1408").alias("C1408"),
            col("pf.C1420").alias("C1420"),
            col("pf.C1439").alias("C1439"),
            col("pf.C1476").alias("C1476"),
            col("pf.C1738").alias("C1738"),
            col("pf.NAC_PAI").alias("NAC_PAI")
        )

    def get_stg_t_cl_personasjuridicas(self): #check
        """Obtiene los datos de la tabla t_cl_personasjuridicas"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CL_PERSONASJURIDICAS}/{self.CONS_TABLE_CL_PERSONASJURIDICAS}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0).select(
            col("C1452"),
            col("C1453_dac").alias("C1453"),
            col("C1454_dac").alias("C1454"),
            col("C1458"),
            col("C8667"),
            col("C1505"),
            col("C1513"),
            col("C1522"),
            col("C1455"),
            col("COD_IDE_IND")
        )

    def get_stg_t_cl_clientes_01(self): #check
        """Obtiene los datos de la tabla cl_clientes"""
        return self.get_edy_cl_clientes().alias("c").select(
            col("c.C0902").alias("C0902"),
            col("c.C1000_dac").alias("C1000"),
            col("c.C1022").alias("C1022"),
            col("c.C1023").alias("C1023"),
            col("c.C1024").alias("C1024"),
            col("c.C1073").alias("C1073"),
            col("c.C1038").alias("C1038"),
            col("c.C1040").alias("C1040"),
            col("c.C1048").alias("C1048"),
            col("c.C1058").alias("C1058"),
            col("c.C1840").alias("C1840"),
            col("c.C1034").alias("C1034"),
            col("c.C3205").alias("C3205"),
            col("c.C8425").alias("C8425"),
            col("c.MED_REC_LUZ_dac").alias("MED_REC_LUZ"),
            col("c.C1584").alias("C1584"),
            col("c.C3206").alias("C3206"),
            col("c.TIPO_RESIDENCIA").alias("TIPO_RESIDENCIA"),
            col("c.CANTPREST").alias("CANTPREST"),
            col("c.EMPLEADO").alias("EMPLEADO"),
            col("c.SEGMENTO_COMERCIAL").alias("SEGMENTO_COMERCIAL"),
            col("c.C1268_dac").alias("C1268")
        )

    def get_stg_t_rel_cliente_sbs(self): #check
        """Obtiene los datos de la tabla t_rel_cliente_sbs"""
        
        df_stg_t_cl_clientes_01 = self.get_stg_t_cl_clientes_01().alias("c")
        df_stg_t_cl_clientpersona = self.get_stg_t_cl_clientpersona().alias("cp")
        df_stg_t_cl_personasfisicas = self.get_stg_t_cl_personasfisicas().alias("pf")
        df_stg_t_cl_personasjuridicas = self.get_stg_t_cl_personasjuridicas().alias("pj")

        df_resultado = (
            df_stg_t_cl_clientes_01
            .join(
                df_stg_t_cl_clientpersona,
                col("c.C0902") == col("cp.C1430"),
                "inner"
            )
            .join(
                df_stg_t_cl_personasfisicas,
                (col("cp.C1431") == col("pf.C1401")) &
                (col("cp.C1432") == col("pf.C1402")) &
                (col("cp.C1435") == col("pf.C1403")),
                "left"
            )
            .join(
                df_stg_t_cl_personasjuridicas,
                (col("cp.C1431") == col("pj.C1452")) &
                (col("cp.C1432") == col("pj.C1453")),
                "left"
            )
            .select(
                col("c.C0902").alias("CO_CLIENTE"),

                when(
                    col("cp.C1437") == "N",
                    col("pf.C8666")
                ).when(
                    col("cp.C1437") == "J",
                    col("pj.C8667")
                ).otherwise(
                    lit(0)
                ).alias("CO_CLIE_SBS")
            )
        )

        return df_resultado
    
    def get_edy_rc_rcc_cuerpo(self): #check
        """Obtiene los datos de la tabla rc_rcc_cuerpo"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_RC_RCC_CUERPO}/{self.CONS_TABLE_RC_RCC_CUERPO}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_rc_cuentascontables(self): #check
        """Obtiene los datos de la tabla rc_cuentascontables"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_RC_CUENTASCONTABLES}/{self.CONS_TABLE_RC_CUENTASCONTABLES}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_stg_t_rc_rcc_cuerpo(self): #check
        """Obtiene los datos de la tabla t_rc_rcc_cuerpo"""

        df_edy_rc_rcc_cuerpo = self.get_edy_rc_rcc_cuerpo().alias("RRC")
        df_edy_rc_cuentascontables = self.get_edy_rc_cuentascontables().alias("C")

        df_resultado = (
            df_edy_rc_rcc_cuerpo
            .join(
                df_edy_rc_cuentascontables,
                (
                    substring(col("C.CODCTADETALLE"), 1, 2)
                    == substring(col("RRC.CODCTACONTABLE"), 1, 2)
                ) &
                (
                    substring(col("C.CODCTADETALLE"), 4, 11)
                    == substring(col("RRC.CODCTACONTABLE"), 4, 11)
                ) &
                (col("C.ESTADO") == "A") &
                (
                    (
                        (substring(col("C.CODCTAPADRE"), 1, 1) == "1") &
                        (col("C.CODCTAACUM") == "00001")
                    ) |
                    (
                        (substring(col("C.CODCTADETALLE"), 1, 2) == "81") &
                        (substring(col("C.CODCTADETALLE"), 4, 11) == "3")
                    )
                ),
                "inner"
            )
            .groupBy(
                col("RRC.CODCLIENTESBS").alias("CODCLIENTESBS")
            )
            .agg(
                countDistinct(col("RRC.CODEMPRESA")).alias("NU_CANT_EMPR"),

                (sum(col("RRC.SALDO")) / lit(100)).alias("MO_SALDO_TOTAL"),

                (
                    sum(
                        when(
                            col("RRC.CODEMPRESA") == 123,
                            col("RRC.SALDO")
                        ).otherwise(lit(0))
                    ) / lit(100)
                ).alias("MO_SALDO_EDYFICAR")
            )
        )

        return df_resultado
    
    def get_edy_autorizaciones(self): #check
        """Obtiene los datos de la tabla autorizaciones"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_AUTORIZACIONES}/{self.CONS_TABLE_AUTORIZACIONES}"
        df_base = self.read_data(table_path)

        df_base = df_base.withColumn(
            "fechaproceso_date",
            to_date(col("fechaproceso"))
        )

        # Obtenemos las últimas 7 fechas distintas de ejecución
        df_ultimas_7_fechas = (
            df_base
            .select("fechaproceso_date")
            .dropDuplicates()
            .orderBy(col("fechaproceso_date").desc())
            .limit(7)
        )

        # Filtramos la tabla original usando esas 7 fechas
        df_filtrado = (
            df_base
            .join(
                df_ultimas_7_fechas,
                on="fechaproceso_date",
                how="inner"
            )
        )

        return df_filtrado
    
    def get_stg_t_prestamos_tipo_excepcion(self): 
        """Obtiene los datos de la tabla t_prestamos_tipo_excepcion"""

        df_edy_asientos = self.get_edy_asientos().alias("A")
        df_edy_autorizaciones = self.get_edy_autorizaciones().alias("AU")

        df_resultado = (
            df_edy_asientos
            .join(
                df_edy_autorizaciones,
                (col("A.ASIENTO") == col("AU.ASIENTO")) &
                (col("A.FECHAPROCESO") == col("AU.FECHAPROCESO")) &
                (col("A.SUCURSAL") == col("AU.SUCURSAL")),
                "inner"
            )
            .filter(
                (col("AU.PERMISO").isin(11, 100, 102, 110, 111, 112, 113, 653)) &
                (col("AU.INICIALES") != "????") &
                (col("A.OPERACION") == 2001) &
                (col("A.ESTADO") == 77) &
                (substring(col("A.DESCRIPCION"), 1, 16) == "Etapa 1 Des.Nro:") 
            )
            .select(
                substring(col("A.DESCRIPCION"), 17, 1000).alias("NRO_PRESTAMO"),

                col("A.FECHAPROCESO").alias("FECHAPROCESO"),

                when(
                    col("AU.PERMISO") == 111,
                    lit(1)
                ).when(
                    col("AU.PERMISO") == 113,
                    lit(2)
                ).otherwise(
                    lit(3)
                ).alias("I1"),

                when(
                    (col("AU.PERMISO") == 111) &
                    (col("AU.MENSAJE").like("Calificación%")),
                    lit(1)
                ).otherwise(
                    lit(0)
                ).alias("I2")
            )
            .orderBy(
                col("NRO_PRESTAMO"),
                col("FECHAPROCESO")
            )
        )

        return df_resultado
    
    def get_stg_t_prestamos_tipo_excepcion_2(self): 
        """Obtiene los datos de la tabla t_prestamos_tipo_excepcion_2"""

        df_stg_t_prestamos_tipo_excepcion = self.get_stg_t_prestamos_tipo_excepcion().alias("TIPO_EXCEP")
        df_stg_t_saldos_sl_solicitudcredito = self.get_stg_t_saldos_sl_solicitudcredito().alias("SALDOS_SOL_CREDS")

        df_resultado = (
            df_stg_t_prestamos_tipo_excepcion
            .join(
                df_stg_t_saldos_sl_solicitudcredito,
                (col("TIPO_EXCEP.NRO_PRESTAMO") == col("SALDOS_SOL_CREDS.CUENTA")) &
                (col("TIPO_EXCEP.FECHAPROCESO") == col("SALDOS_SOL_CREDS.FCREAC1ERET")),
                "inner"
            )
            .select(
                col("SALDOS_SOL_CREDS.JTS_OID").alias("SALDOS_JTS_OID"),

                when(
                    (col("TIPO_EXCEP.I1") == 1) &
                    (col("TIPO_EXCEP.I2") == 0),
                    col("SALDOS_SOL_CREDS.C5004")
                ).when(
                    (col("TIPO_EXCEP.I1") == 1) &
                    (col("TIPO_EXCEP.I2") == 1),
                    lit("RIESGOSO")
                ).when(
                    col("TIPO_EXCEP.I1") == 2,
                    lit("PARENTESCO")
                ).otherwise(
                    lit("OTRO")
                ).alias("TIPO_EXCEP"),

                # columnas auxiliares solo para ordenar como en el SQL
                col("TIPO_EXCEP.I1").alias("_I1"),
                col("TIPO_EXCEP.I2").alias("_I2")
            )
            .orderBy(
                col("SALDOS_JTS_OID"),
                col("_I1").desc(),
                col("_I2").desc()
            )
            .drop("_I1", "_I2")
        )

        return df_resultado
    
    def get_stg_t_sl_rel_sol_destino(self): #check
        """Obtiene los datos de la tabla t_sl_rel_sol_destino"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_SL_REL_SOL_DESTINO}/{self.CONS_TABLE_SL_REL_SOL_DESTINO}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0).select(
            col("SOLICITUD"),
            col("DESTINO"),
            col("PORCENTAJE")
        )

    def get_stg_t_rh_funcionario(self): #check
        """Obtiene los datos de la tabla rh_funcionario"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_RH_FUNCIONARIO}/{self.CONS_TABLE_RH_FUNCIONARIO}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_stg_t_adr_car_pue(self): #check
        """Obtiene los datos de la tabla adr_car_pue"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_ADR_CAR_PUE}/{self.CONS_TABLE_ADR_CAR_PUE}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_stg_t_tc_sucursales(self): #check
        """Obtiene los datos de la tabla tc_sucursales"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_TC_SUCURSALES}/{self.CONS_TABLE_TC_SUCURSALES}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_stg_tmp_egp_2(self, refresh=False):
        """Obtiene los datos de la tabla tmp_egp_2"""

        if (
                not refresh
                and self.df_stg_tmp_egp_2 is not None
            ):
                return self.df_stg_tmp_egp_2

        df_stg_t_saldos_sl_solicitudcredito = self.get_stg_t_saldos_sl_solicitudcredito().alias("SALDOS_SOL_CREDS")
        df_stg_t_tc_sucursales = self.get_stg_t_tc_sucursales().alias("SUCUR")
        df_stg_t_deveng_por_cta_venc = self.get_stg_t_deveng_por_cta_venc()
        df_stg_t_deveng_por_cta_venc_st = self.get_stg_t_deveng_por_cta_venc_st()
        df_stg_t_rh_funcionario = self.get_stg_t_rh_funcionario()
        df_stg_t_adr_car_pue = self.get_stg_t_adr_car_pue()
        df_ods_hd_puesto_cargo_analista = self.get_ods_hd_puesto_cargo_analista()
        df_stg_t_rel_cliente_sbs = self.get_stg_t_rel_cliente_sbs()
        df_stg_t_rc_rcc_cuerpo = self.get_stg_t_rc_rcc_cuerpo()
        df_stg_t_prestamos_tipo_excepcion_2 = self.get_stg_t_prestamos_tipo_excepcion_2()
        df_stg_t_sl_rel_sol_destino = self.get_stg_t_sl_rel_sol_destino()

        df_dev_venc = (
            df_stg_t_deveng_por_cta_venc
            .groupBy("SALDOS_JTS_OID")
            .agg(
                max("ESTADOATRASO").alias("ESTADOATRASO_DV")
            )
            .withColumnRenamed("SALDOS_JTS_OID", "SALDOS_JTS_OID_DV")
            .alias("DV")
        )

        df_dev_venc_st = (
            df_stg_t_deveng_por_cta_venc_st
            .groupBy("SALDOS_JTS_OID")
            .agg(
                max("ESTADOATRASO").alias("ESTADOATRASO_DV_ST")
            )
            .withColumnRenamed("SALDOS_JTS_OID", "SALDOS_JTS_OID_DV_ST")
            .alias("DV_ST")
        )

        df_funcionario = (
            df_stg_t_rh_funcionario.alias("FUNC")
            .join(
                df_stg_t_adr_car_pue.alias("ACP"),
                col("ACP.PUESTO") == col("FUNC.PUESTO"),
                "left"
            )
            .select(
                col("FUNC.CODFUNCIONARIO").alias("CODFUNCIONARIO"),
                col("ACP.CARGO").alias("CARGO"),
                col("FUNC.PUESTO").alias("PUESTO"),
                col("FUNC.CODAGENCIA").alias("CODAGENCIA"),
                col("FUNC.NOMBRE_dac").alias("NOMBRE")
            )
            .alias("FUNC")
        )

        df_hist_carg_anal = (
            df_ods_hd_puesto_cargo_analista
            .groupBy(
                "FE_PROCESO",
                "CO_USUA_TOPA"
            )
            .agg(
                first("CO_ANALISTA", ignorenulls=True).alias("CO_ANAL_ORIG"),
                first("CO_CARGO", ignorenulls=True).alias("CO_CARG_ANAL_ORIG"),
                first("CO_PUESTO", ignorenulls=True).alias("CO_PUES_ANAL_ORIG"),
                first("CO_SUCURSAL", ignorenulls=True).alias("CO_SUCU_ANAL_ORIG")
            )
            .withColumnRenamed("FE_PROCESO", "FE_PROCESO_HIST")
            .withColumnRenamed("CO_USUA_TOPA", "CO_USUA_TOPA_HIST")
            .alias("HIST")
        )

        df_cliente_sbs_rcc = (
            df_stg_t_rel_cliente_sbs.alias("CLI")
            .join(
                df_stg_t_rc_rcc_cuerpo.alias("RCCUERPO"),
                col("CLI.CO_CLIE_SBS") == col("RCCUERPO.CODCLIENTESBS"),
                "left"
            )
            .groupBy(
                col("CLI.CO_CLIENTE").alias("CO_CLIENTE_RCC")
            )
            .agg(
                max("RCCUERPO.NU_CANT_EMPR").alias("NU_CANT_EMPR"),
                max("RCCUERPO.MO_SALDO_TOTAL").alias("MO_SALDO_TOTAL"),
                max("RCCUERPO.MO_SALDO_EDYFICAR").alias("MO_SALDO_EDYFICAR")
            )
            .alias("RCC")
        )

        df_tipo_excepcion = (
            df_stg_t_prestamos_tipo_excepcion_2
            .groupBy("SALDOS_JTS_OID")
            .agg(
                first("TIPO_EXCEP", ignorenulls=True).alias("TIPO_EXCEP")
            )
            .withColumnRenamed("SALDOS_JTS_OID", "SALDOS_JTS_OID_EXCEP")
            .alias("EXCEP")
        )

        key = self.key_aes

        df_func_siniestro = (
            df_stg_t_rh_funcionario
            .filter(
                substring(upper(aes_decrypt(col("NOMBRE_dac"),lit(key),lit("ECB"),lit("PKCS")).cast("string")), 1, 15) == "E-TNG-SINIESTRO"
            )
            .select(
                col("CODFUNCIONARIO").alias("CODFUNCIONARIO_SIN")
            )
            .dropDuplicates()
            .withColumn("IN_SINIESTRO_FLAG", lit(1))
            .alias("SIN")
        )

        df_destino = (
            df_stg_t_sl_rel_sol_destino
            .groupBy(
                col("SOLICITUD").alias("SOLICITUD_DEST")
            )
            .agg(
                first("DESTINO", ignorenulls=True).alias("DESTINO")
            )
            .alias("DEST")
        )

        df_resultado = (
            df_stg_t_saldos_sl_solicitudcredito
            .join(
                df_stg_t_tc_sucursales,
                col("SALDOS_SOL_CREDS.SUCURSAL") == col("SUCUR.C6021"),
                "inner"
            )
            .join(
                df_dev_venc,
                col("DV.SALDOS_JTS_OID_DV") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .join(
                df_dev_venc_st,
                col("DV_ST.SALDOS_JTS_OID_DV_ST") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .join(
                df_funcionario,
                col("FUNC.CODFUNCIONARIO") == col("SALDOS_SOL_CREDS.C1670"),
                "left"
            )
            .join(
                df_hist_carg_anal,
                (col("HIST.FE_PROCESO_HIST") == col("SALDOS_SOL_CREDS.FECHA_ORIGINAL")) &
                (col("HIST.CO_USUA_TOPA_HIST") == col("SALDOS_SOL_CREDS.USU_ORIGINAL")),
                "left"
            )
            .join(
                df_cliente_sbs_rcc,
                col("RCC.CO_CLIENTE_RCC") == col("SALDOS_SOL_CREDS.C1803"),
                "left"
            )
            .join(
                df_tipo_excepcion,
                col("EXCEP.SALDOS_JTS_OID_EXCEP") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .join(
                df_func_siniestro,
                col("SIN.CODFUNCIONARIO_SIN") == col("SALDOS_SOL_CREDS.C1670"),
                "left"
            )
            .join(
                df_destino,
                col("DEST.SOLICITUD_DEST") == col("SALDOS_SOL_CREDS.C1704"),
                "left"
            )
            .select(
                col("SALDOS_SOL_CREDS.JTS_OID").alias("SALDOS_JTS_OID"),

                when(
                    col("SALDOS_SOL_CREDS.IN_GRUPAL") == 1,
                    lit("GRUPO SOLIDARIO")
                ).otherwise(
                    lit("INDIVIDUAL")
                ).alias("TI_PRESTAMO"),

                col("SALDOS_SOL_CREDS.MONEDA").alias("CO_MONEDA"),

                when(
                    col("SALDOS_SOL_CREDS.C4280") == "C",
                    coalesce(
                        col("DV.ESTADOATRASO_DV"),
                        col("DV_ST.ESTADOATRASO_DV_ST")
                    )
                ).otherwise(
                    col("SALDOS_SOL_CREDS.C1728")
                ).alias("CO_ESTA_OPER"),

                col("SALDOS_SOL_CREDS.SUCURSAL").alias("CO_SUCURSAL"),
                col("SALDOS_SOL_CREDS.C1670").alias("CO_ANALISTA"),
                col("SALDOS_SOL_CREDS.USUTOPAZ").alias("CO_USUA_TOPA"),

                coalesce(col("FUNC.CARGO"), lit(0)).alias("CO_CARG_ANAL"),
                coalesce(col("FUNC.PUESTO"), lit("0")).alias("CO_PUES_ANAL"),
                coalesce(col("FUNC.CODAGENCIA"), lit(0)).alias("CO_SUCU_ANAL"),

                col("SALDOS_SOL_CREDS.PRODUCTO").alias("CO_PRODUCTO"),
                col("SALDOS_SOL_CREDS.CRITERIO").alias("CO_CRITERIO"),
                col("SALDOS_SOL_CREDS.C1704").alias("NU_SOLICITUD"),
                col("SALDOS_SOL_CREDS.MB_REFERENCIA_GRUPAL").alias("NU_GRUP_SOLI"),

                col("SUCUR.C6023").alias("CO_SUCU_PADR"),

                col("HIST.CO_ANAL_ORIG").alias("CO_ANAL_ORIG"),
                col("HIST.CO_CARG_ANAL_ORIG").alias("CO_CARG_ANAL_ORIG"),
                col("HIST.CO_PUES_ANAL_ORIG").alias("CO_PUES_ANAL_ORIG"),
                col("HIST.CO_SUCU_ANAL_ORIG").alias("CO_SUCU_ANAL_ORIG"),

                col("SALDOS_SOL_CREDS.C5004").alias("TI_CLIENTE"),

                col("RCC.NU_CANT_EMPR").alias("NU_CANT_EMPR"),

                coalesce(
                    col("RCC.MO_SALDO_TOTAL"),
                    lit(0)
                ).alias("MO_SALD_SIST_FINA"),

                when(
                    coalesce(col("RCC.MO_SALDO_TOTAL"), lit(0)) == 0,
                    lit(0)
                ).otherwise(
                    col("RCC.MO_SALDO_EDYFICAR") / col("RCC.MO_SALDO_TOTAL")
                ).alias("NU_PORC_EXCL"),

                when(
                    col("SALDOS_SOL_CREDS.C5074") == "N",
                    lit(0)
                ).otherwise(
                    col("SALDOS_SOL_CREDS.C5044")
                ).alias("NU_PERI_GRAC"),

                coalesce(
                    col("SALDOS_SOL_CREDS.C5074"),
                    lit("N")
                ).alias("CO_TIPO_GRAC"),

                coalesce(
                    col("EXCEP.TIPO_EXCEP"),
                    lit("NINGUNO")
                ).alias("TI_EXCEPCION"),

                coalesce(
                    col("SIN.IN_SINIESTRO_FLAG"),
                    lit(0)
                ).alias("IN_SINIESTRO"),

                coalesce(
                    col("DEST.DESTINO"),
                    lit(".")
                ).alias("CO_DESTINO")
            )
        )

        self.df_stg_tmp_egp_2 = (
            df_resultado
            .persist(StorageLevel.MEMORY_ONLY)
        )

        return self.df_stg_tmp_egp_2
    
    def get_edy_gastos_por_cuota(self): #check
        """Obtiene los datos de la tabla gastos_por_cuota"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_GASTOS_POR_CUOTA}/{self.CONS_TABLE_GASTOS_POR_CUOTA}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_stg_t_dias_atraso_prestamos(self): 
        """Obtiene los datos de la tabla t_dias_atraso_prestamos"""

        fe_proceso = self.FEC_PROCESO

        fin_mes_anterior = date_sub(trunc(to_date(lit(fe_proceso)), "month"), 1)

        pp = self.get_edy_bs_planpagos().alias("pp")

        df_edy_saldos_activos = self.get_edy_saldos_activos().select(
            col("JTS_OID"),
            col("C1604")
        )

        df_edy_saldos_pasivos = self.get_edy_saldos_pasivos().select(
            col("JTS_OID"),
            col("C1604")
        )

        s = df_edy_saldos_activos.unionByName(df_edy_saldos_pasivos).alias("s")

        gpc = (
            self.get_edy_gastos_por_cuota()
            .filter(col("TZ_LOCK") == 0)
            .groupBy("SALDOS_JTS_OID", "NUMERO_CUOTA")
            .agg(sum("SALDO_GASTO").alias("SUM_SALDO_GASTO"))
            .alias("gpc")
        )

        pp_base = (
            pp.join(s, col("pp.SALDO_JTS_OID") == col("s.JTS_OID"), "inner")
            .join(
                gpc,
                (col("pp.SALDO_JTS_OID") == col("gpc.SALDOS_JTS_OID"))
                & (col("pp.C2300") == col("gpc.NUMERO_CUOTA")),
                "left"
            )
            .filter((col("s.C1604") < 0) & (col("pp.TZ_LOCK") == 0))
        )

        saldo_cuota = (
            col("pp.C2309")
            + col("pp.C2310")
            + coalesce(col("gpc.SUM_SALDO_GASTO"), lit(0))
        )

        sq = pp_base.select(
            col("pp.SALDO_JTS_OID").alias("SALDOS_JTS_OID"),
            when(
                (saldo_cuota != 0) & (col("pp.C2302") < fe_proceso),
                datediff(lit(fe_proceso), col("pp.C2302"))
            ).when(
                (saldo_cuota == 0) & (col("pp.FECHACANCELACION") > col("pp.C2302")),
                datediff(col("pp.FECHACANCELACION"), col("pp.C2302"))
            ).otherwise(lit(0)).alias("TOTA_DIAS_ATRASO"),
            when(
                (saldo_cuota != 0) & (col("pp.C2302") < fe_proceso),
                datediff(lit(fe_proceso), col("pp.C2302"))
            ).when(
                (saldo_cuota == 0) & (col("pp.FECHACANCELACION") > col("pp.C2302")),
                datediff(col("pp.FECHACANCELACION"), col("pp.C2302"))
            ).otherwise(lit(0)).alias("MAX_DIAS_ATRASO"),
            when(
                (col("pp.C2302") <= fe_proceso) & (saldo_cuota != 0),
                lit(1)
            ).otherwise(lit(0)).alias("NRO_CUOTAS_VENC"),
            when(
                (saldo_cuota != 0) & (col("pp.C2302") < fin_mes_anterior),
                datediff(lit(fin_mes_anterior), col("pp.C2302"))
            ).when(
                (saldo_cuota == 0)
                & (col("pp.FECHACANCELACION") > fin_mes_anterior)
                & (col("pp.C2302") < fin_mes_anterior),
                datediff(lit(fin_mes_anterior), col("pp.C2302"))
            ).otherwise(lit(0)).alias("DIAS_MORA_MES")
        )

        resultado_df = (
            sq.groupBy("SALDOS_JTS_OID")
            .agg(
                sum("TOTA_DIAS_ATRASO").alias("TOTA_DIAS_ATRASO"),
                max("MAX_DIAS_ATRASO").alias("MAX_DIAS_ATRASO"),
                sum("NRO_CUOTAS_VENC").alias("NRO_CUOTAS_VENC"),
                max("DIAS_MORA_MES").alias("DIAS_MORA_MES")
            )
        )

        return resultado_df

    def get_stg_t_prestamos_fogapi(self): #check
        """Obtiene los datos de la tabla t_prestamos_fogapi"""

        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_PRESTAMOS_FOGAPI}/{self.CONS_TABLE_PRESTAMOS_FOGAPI}"

        df_edy_prestamos_fogapi = self.read_data(table_path)

        window_pf = (
            Window
            .partitionBy("SALDO_JTS_OID")
            .orderBy(
                col("FECHA").desc(),
                col("SECUENCIADOR").desc()
            )
        )

        df_resultado = (
            df_edy_prestamos_fogapi
            .withColumn("rn", row_number().over(window_pf))
            .filter(col("rn") == 1)
            .select(
                col("SECUENCIADOR").alias("SECUENCIADOR"),
                col("LOTE").alias("LOTE"),
                col("FECHA").alias("FECHA"),
                col("SALDO_JTS_OID").alias("SALDO_JTS_OID"),
                col("FOGAPI").alias("FOGAPI"),
                col("ESTADO").alias("ESTADO"),
                lit(0).alias("SALDO_CAPITAL")
            )
        )

        return df_resultado
    
    def get_edy_gr_relaciongtiacredsolic(self): #check
        """Obtiene los datos de la tabla gr_relaciongtiacredsolic"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_GR_RELACIONGTIACREDSOLIC}/{self.CONS_TABLE_GR_RELACIONGTIACREDSOLIC}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_gr_garantias(self): #check
        """Obtiene los datos de la tabla gr_garantias"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_GR_GARANTIAS}/{self.CONS_TABLE_GR_GARANTIAS}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_stg_t_gr_garantias(self): 
        """Obtiene los datos de la tabla t_gr_garantias"""

        df_edy_gr_relaciongtiacredsolic = self.get_edy_gr_relaciongtiacredsolic().alias("RELGTIASOL")
        df_edy_gr_garantias = self.get_edy_gr_garantias().alias("GR")

        df_resultado = (
            df_edy_gr_relaciongtiacredsolic
            .join(
                df_edy_gr_garantias,
                col("GR.NROGARANTIA") == col("RELGTIASOL.NROGARANTIA"),
                "inner"
            )
            .filter(
                (col("RELGTIASOL.TZ_LOCK") == 0) &
                (col("RELGTIASOL.ESTADOCREDITO") == "A") &
                (col("GR.TZ_LOCK") == 0)
            )
            .select(
                col("GR.NROGARANTIA").alias("NROGARANTIA"),
                col("RELGTIASOL.NROSOLICITUD").alias("NROSOLICITUD"),
                col("GR.ESTADO").alias("ESTADO"),
                col("GR.CLASGARANTIA").alias("CLASGARANTIA"),
                col("RELGTIASOL.NROCREDITO").alias("NROCREDITO"),
                col("GR.GAR_MONEDA").alias("GAR_MONEDA"),
                col("GR.TIPOGARANTIA").alias("TIPOGARANTIA"),
                col("RELGTIASOL.ESTADOCREDITO").alias("ESTADOCREDITO"),
                col("RELGTIASOL.VALORUTILIZADO").alias("VALORUTILIZADO"),
                col("RELGTIASOL.PORCENTAJEUTILIZADO").alias("PORCENTAJEUTILIZADO"),
                col("GR.FECHAREGISTRO").alias("FECHAREGISTRO"),
                col("GR.VALORVALUO").alias("VALORVALUO"),
                col("GR.IMPORTE").alias("IMPORTE"),
                col("GR.EMBARGADA").alias("EMBARGADA"),
                col("GR.VALOREMBARGO").alias("VALOREMBARGO"),
                col("GR.GAR_VALORGRAVAMEN").alias("GAR_VALORGRAVAMEN"),
                col("GR.SEGURO").alias("SEGURO"),
                col("GR.NROPOLIZA").alias("NROPOLIZA"),
                col("GR.VENCIMIENTOPOLIZA").alias("VENCIMIENTOPOLIZA"),

                # @001
                col("GR.GR_VALORGRAVUTI").alias("GR_VALORGRAVUTI"),
                col("GR.SUBCLASGTIAS").alias("SUBCLASGTIAS"),
                col("GR.GAR_FCONST").alias("GAR_FCONST"),
                col("GR.GAR_PARTIDA").alias("GAR_PARTIDA")
            )
        )

        return df_resultado
    
    def get_edy_bs_pays_detail(self): #check
        """Obtiene los datos de la tabla bs_pays_detail"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_BS_PAYS_DETAIL}/{self.CONS_TABLE_BS_PAYS_DETAIL}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_bs_charge_detail(self): #check
        """Obtiene los datos de la tabla bs_charge_detail"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_BS_CHARGE_DETAIL}/{self.CONS_TABLE_BS_CHARGE_DETAIL}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_stg_t_pagos_parciales(self): 
        """Obtiene los datos de la tabla t_pagos_parciales"""

        df_edy_saldos = self.get_edy_saldos_activos().alias("s")
        df_edy_bs_pays_detail = self.get_edy_bs_pays_detail().alias("pd")
        df_edy_bs_charge_detail = self.get_edy_bs_charge_detail().alias("cd")

        pd_validos = (
            df_edy_bs_pays_detail
            .select(
                col("pd.SALDOS_JTS_OID").alias("PD_SALDOS_JTS_OID"),
                col("pd.CUOTA").alias("PD_CUOTA")
            )
            .dropDuplicates()
            .alias("pd_validos")
        )

        cd_validos = (
            df_edy_bs_charge_detail
            .select(
                col("cd.SALDOS_JTS_OID").alias("CD_SALDOS_JTS_OID"),
                col("cd.CUOTA").alias("CD_CUOTA")
            )
            .dropDuplicates()
            .alias("cd_validos")
        )

        df_resultado = (
            df_edy_saldos
            .filter(
                col("s.C1604") < 0
            )
            .join(
                pd_validos,
                (col("pd_validos.PD_SALDOS_JTS_OID") == col("s.JTS_OID")) &
                (col("pd_validos.PD_CUOTA") == (col("s.C1645") + lit(1))),
                "left"
            )
            .join(
                cd_validos,
                (col("cd_validos.CD_SALDOS_JTS_OID") == col("s.JTS_OID")) &
                (col("cd_validos.CD_CUOTA") == (col("s.C1645") + lit(1))),
                "left"
            )
            .select(
                col("s.JTS_OID").alias("SALDOS_JTS_OID"),
                when(
                    col("pd_validos.PD_SALDOS_JTS_OID").isNotNull() |
                    col("cd_validos.CD_SALDOS_JTS_OID").isNotNull(),
                    lit(1)
                ).otherwise(lit(0)).alias("FLAG_PAGO_PARCIAL")
            )
        )

        return df_resultado
    
    def get_stg_t_prest_deuda_venc(self): 
        """Obtiene los datos de la tabla t_prest_deuda_venc"""

        fe_proceso = self.FEC_PROCESO

        df_edy_bs_planpagos = self.get_edy_bs_planpagos().alias("pp")
        df_edy_gastos_por_cuota = self.get_edy_gastos_por_cuota()

        gpc_agg = (
            df_edy_gastos_por_cuota
            .groupBy(
                col("SALDOS_JTS_OID").alias("GPC_SALDOS_JTS_OID"),
                col("NUMERO_CUOTA").alias("GPC_NUMERO_CUOTA")
            )
            .agg(
                sum(col("SALDO_GASTO")).alias("GAST_VENC")
            )
            .alias("gpc")
        )

        x = (
            df_edy_bs_planpagos
            .filter(
                col("pp.C2302") < fe_proceso
            )
            .join(
                gpc_agg,
                (col("gpc.GPC_SALDOS_JTS_OID") == col("pp.SALDO_JTS_OID")) &
                (col("gpc.GPC_NUMERO_CUOTA") == col("pp.C2300")),
                "left"
            )
            .select(
                col("pp.SALDO_JTS_OID").alias("SALDOS_JTS_OID"),
                col("pp.C2300").alias("NUM_CUOTA"),
                (
                    coalesce(col("pp.C2309"), lit(0)) +
                    coalesce(col("pp.C2310"), lit(0))
                ).alias("CAPI_INTE_VENC"),
                coalesce(col("gpc.GAST_VENC"), lit(0)).alias("GAST_VENC")
            )
        )

        df_resultado = (
            x
            .groupBy(
                col("SALDOS_JTS_OID")
            )
            .agg(
                coalesce(
                    sum(
                        col("CAPI_INTE_VENC") + col("GAST_VENC")
                    ),
                    lit(0)
                ).alias("DEUDA_VENCIDA")
            )
        )

        return df_resultado
    
    def get_stg_t_rep_cartera_morosa(self): 
        """Obtiene los datos de la tabla t_rep_cartera_morosa"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_REP_CARTERA_MOROSA}/{self.CONS_TABLE_REP_CARTERA_MOROSA}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_stg_t_prest_mora_total(self): 
        """Obtiene los datos de la tabla t_prest_mora_total"""

        df_stg_t_rep_cartera_morosa = self.get_stg_t_rep_cartera_morosa().alias("rcm")

        df_stg_t_saldos_activos, df_stg_t_saldos_pasivos = self.get_stg_t_saldos()

        df_resultado = (
            df_stg_t_saldos_activos.alias("s")
            .join(
                df_stg_t_rep_cartera_morosa,
                (col("s.CUENTA") == col("rcm.PRCODPRES")) &
                (col("rcm.PRCUOTAPR").isNotNull()),
                "left"
            )
            .filter(
                (col("s.C9314") == 5) &
                (col("s.C1604") < 0)
            )
            .groupBy(
                col("s.CUENTA").alias("PRCODPRES")
            )
            .agg(
                sum(
                    coalesce(col("rcm.MORA"), lit(0)) +
                    coalesce(col("rcm.INT_COM_MOR"), lit(0))
                ).alias("TOTAL_MORA")
            )
        )

        return df_resultado
    
    def get_stg_t_prest_deuda_total(self): 
        """Obtiene los datos de la tabla t_prest_deuda_total"""

        df_edy_bs_planpagos = self.get_edy_bs_planpagos().alias("pp")
        df_edy_gastos_por_cuota = self.get_edy_gastos_por_cuota()

        gpc_agg = (
            df_edy_gastos_por_cuota
            .groupBy(
                col("SALDOS_JTS_OID").alias("GPC_SALDOS_JTS_OID"),
                col("NUMERO_CUOTA").alias("GPC_NUMERO_CUOTA")
            )
            .agg(
                sum(col("SALDO_GASTO")).alias("GAST_TOTAL")
            )
            .alias("gpc")
        )

        x = (
            df_edy_bs_planpagos
            .filter(col("pp.TZ_LOCK") == 0)
            .join(
                gpc_agg,
                (col("gpc.GPC_SALDOS_JTS_OID") == col("pp.SALDO_JTS_OID")) &
                (col("gpc.GPC_NUMERO_CUOTA") == col("pp.C2300")),
                "left"
            )
            .select(
                col("pp.SALDO_JTS_OID").alias("SALDOS_JTS_OID"),
                col("pp.C2300").alias("NUM_CUOTA"),
                (
                    coalesce(col("pp.C2309"), lit(0)) +
                    coalesce(col("pp.C2310"), lit(0))
                ).alias("CAPI_INTE_TOTAL"),
                coalesce(col("gpc.GAST_TOTAL"), lit(0)).alias("GAST_TOTAL")
            )
        )
        
        df_resultado = (
            x
            .groupBy(
                col("SALDOS_JTS_OID")
            )
            .agg(
                coalesce(
                    sum(
                        col("CAPI_INTE_TOTAL") + col("GAST_TOTAL")
                    ),
                    lit(0)
                ).alias("DEUDA_TOTAL")
            )
        )

        return df_resultado

    def get_stg_t_datos_egp_ini(self): 
        """Obtiene los datos de la tabla t_datos_egp_ini"""

        path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_HD_EGP}/{self.CONS_TABLE_HD_EGP}"

        df_hd_egp = self.read_data(path)

        ultimo_dia_mes_anterior = (
            df_hd_egp
            .filter(col("fe_proceso") <= last_day(add_months(lit(self.FEC_PROCESO), -1)))
            .agg(max("fe_proceso").alias("max_fecha"))
            .first()["max_fecha"]
        )

        table_path = df_hd_egp.filter(
            col("fe_proceso") == lit(ultimo_dia_mes_anterior)
        )

        df_ods_ud_egp = table_path.alias("egp")

        df_resultado = (
            df_ods_ud_egp
            .select(
                col("egp.IN_SALD_JTS_OID").alias("SALDOS_JTS_OID"),
                col("egp.FE_PROCESO").alias("FECHA"),
                col("egp.CO_ESTA_OPER").alias("COD_EST_INI"),

                (
                    year(to_date(col("egp.FE_PROCESO"))) * lit(100) +
                    month(to_date(col("egp.FE_PROCESO")))
                ).alias("MES"),

                col("egp.NU_DIAS_ATRA").alias("DIA_MOR_INI"),
                col("egp.MO_SALD_ACTU").alias("SALD_CAP_INI")
            )
        )

        return df_resultado
    
    def get_stg_t_bs_pays_detail_mes(self): 
        """Obtiene los datos de la tabla t_bs_pays_detail_mes"""

        fe_inicio_mes = trunc(to_date(lit(self.FEC_PROCESO)), "MM")
        fe_inicio_mes_siguiente = add_months(fe_inicio_mes, 1)

        df_edy_bs_pays_detail = self.get_edy_bs_pays_detail().alias("pd")

        df_resultado = (
            df_edy_bs_pays_detail
            .filter(
                (to_date(col("pd.FECHAVALOR")) >= fe_inicio_mes) &
                (to_date(col("pd.FECHAVALOR")) < fe_inicio_mes_siguiente)
            )
            .select(
                col("pd.SALDOS_JTS_OID").alias("SALDOS_JTS_OID"),
                col("pd.HP_JTS_OID").alias("HP_JTS_OID"),
                col("pd.CUOTA").alias("CUOTA"),
                col("pd.CAPITALPAGADO").alias("CAPITALPAGADO"),
                col("pd.INTERESPAGADO").alias("INTERESPAGADO"),
                col("pd.MORAPAGADA").alias("MORAPAGADA"),
                col("pd.INTERES_COMP_MORA").alias("INTERES_COMP_MORA"),
                col("pd.CAPITALADELANTADO").alias("CAPITALADELANTADO")
            )
        )

        return df_resultado
    
    def get_stg_t_bs_charge_detail_mes(self): 
        """Obtiene los datos de la tabla t_bs_charge_detail_mes"""

        fe_inicio_mes = trunc(to_date(lit(self.FEC_PROCESO)), "MM")
        fe_inicio_mes_siguiente = add_months(fe_inicio_mes, 1)

        df_edy_bs_historia_plazo = self.get_edy_bs_historia_plazo()
        df_edy_bs_charge_detail = self.get_edy_bs_charge_detail().alias("pd")

        hist_plazo_mes = (
            df_edy_bs_historia_plazo.alias("hp")
            .filter(
                (to_date(col("hp.FECHAVALOR")) >= fe_inicio_mes) &
                (to_date(col("hp.FECHAVALOR")) < fe_inicio_mes_siguiente)
            )
            .select(
                col("hp.JTS_OID").alias("JTS_OID")
            )
            .dropDuplicates()
            .alias("hp")
        )

        df_resultado = (
            df_edy_bs_charge_detail.alias("cd")
            .join(
                hist_plazo_mes,
                col("hp.JTS_OID") == col("cd.HP_JTS_OID"),
                "inner"
            )
            .select(
                col("cd.SALDOS_JTS_OID").alias("SALDOS_JTS_OID"),
                col("cd.HP_JTS_OID").alias("HP_JTS_OID"),
                col("cd.CUOTA").alias("CUOTA"),
                col("cd.SECUENCIAL_CARGO").alias("SECUENCIAL_CARGO"),
                col("cd.CARGO_ID").alias("CARGO_ID"),
                col("cd.COMISION_IMPUESTO").alias("COMISION_IMPUESTO"),
                col("cd.TOTAL_PAGADO").alias("TOTAL_PAGADO"),
                col("cd.SALDO_CARGO").alias("SALDO_CARGO")
            )
        )

        return df_resultado
    
    def get_stg_t_cr_recupercartera(self): 
        """Obtiene los datos de la tabla t_cr_recupercartera"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CR_RECUPERCARTERA}/{self.CONS_TABLE_CR_RECUPERCARTERA}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0).select(
            col("FECHAOPER"),      
            col("NROPRESTAMO"),      
            col("TIPOOPER"),      
            col("AGENCIA"),        
            col("MONEDA"),        
            col("CLIENTE"),        
            col("CAPITAL"),        
            col("INTCOMPENSATORIO"),
            col("INTMORATORIO"),    
            col("CARGOSFIJOS"),      
            col("MONTOCONDONADO"),    
            col("ITF"),  
            col("INTERCOND"),
            col("MORACOND"),      
            col("CARGOSCOND"),      
            col("INT_COM_MOR"),      
            col("INT_COM_MOR_CON"),
            col("FECHAPROCESO"),
            col("TRANSACCIONCOBRO"),
            col("MONTODEPOSITO"),
            col("MOR_NOR")
        )

    def get_edy_historia_vista(self): #check
        """Obtiene los datos de la tabla historia_vista"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_HISTORIA_VISTA}/{self.CONS_TABLE_HISTORIA_VISTA}"
        return self.read_data(table_path)

    def get_edy_co_concepcont(self): #check
        """Obtiene los datos de la tabla co_concepcont"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CO_CONCEPCONT}/{self.CONS_TABLE_CO_CONCEPCONT}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_stg_t_historia_vista(self): 
        """Obtiene los datos de la tabla t_historia_vista"""

        df_edy_saldos_activos = self.get_edy_saldos_activos().select(
            col("cuenta"),
            col("operacion"),
            col("JTS_OID"),
            col("C1803"),
            col("C1604")
        )

        key = self.key_aes

        df_edy_saldos_pasivos = self.get_edy_saldos_pasivos().select(
            aes_decrypt(
                        col("CUENTA_dac"),
                        lit(key),
                        lit("ECB"),
                        lit("PKCS")
                    ).cast("string")
                .alias("CUENTA"),
            col("operacion"),
            col("JTS_OID"),
            col("C1803"),
            col("C1604")
        )

        df_edy_saldos = df_edy_saldos_activos.unionByName(df_edy_saldos_pasivos).alias("SAL")
        df_edy_co_concepcont = self.get_edy_co_concepcont().alias("CON")
        df_edy_historia_vista = self.get_edy_historia_vista().alias("HIV")

        hiv_agg = (
            df_edy_historia_vista
            .groupBy(
                col("HIV.SALDO_JTS_OID").alias("HIV_SALDO_JTS_OID")
            )
            .agg(
                sum(
                    when(
                        col("HIV.DEBITO_CREDITO") == "C",
                        coalesce(col("HIV.MONTO"), lit(0))
                    ).otherwise(lit(0))
                ).alias("CREDITOS"),

                sum(
                    when(
                        col("HIV.DEBITO_CREDITO") == "D",
                        coalesce(col("HIV.MONTO"), lit(0))
                    ).otherwise(lit(0))
                ).alias("DEBITOS")
            )
            .alias("HIV_AGG")
        )

        df_resultado = (
            df_edy_saldos
            .join(
                df_edy_co_concepcont,
                (col("SAL.CUENTA") == col("CON.C7502")) &
                (col("CON.C7500").isin(23000001, 23000002)) &
                (col("SAL.OPERACION") != 0),
                "inner"
            )
            .join(
                hiv_agg,
                col("HIV_AGG.HIV_SALDO_JTS_OID") == col("SAL.JTS_OID"),
                "left"
            )
            .select(
                col("SAL.C1803").alias("CO_CLIENTE"),
                col("SAL.OPERACION").alias("CO_PRESTAMO"),
                col("SAL.JTS_OID").alias("SALDOS_JTS_OID"),
                col("SAL.C1604").alias("MO_ACT"),

                coalesce(col("HIV_AGG.CREDITOS"), lit(0)).alias("CREDITOS"),
                coalesce(col("HIV_AGG.DEBITOS"), lit(0)).alias("DEBITOS")
            )
        )

        return df_resultado
    
    
    def get_stg_t_datos_saldo_resuelto(self): 
        """Obtiene los datos de la tabla t_datos_saldo_resuelto"""


        df_stg_t_saldos_sl_solicitudcredito = self.get_stg_t_saldos_sl_solicitudcredito().alias("SALDOS_SOL_CREDS")
        df_stg_t_datos_egp_ini = self.get_stg_t_datos_egp_ini().alias("DATOS_EGP_INI")
        df_stg_t_deveng_por_cta_venc = self.get_stg_t_deveng_por_cta_venc().alias("DV")

        df_dev_venc = (
            df_stg_t_deveng_por_cta_venc
            .groupBy(
                col("DV.SALDOS_JTS_OID").alias("SALDOS_JTS_OID_DV")
            )
            .agg(
                max(col("DV.ESTADOATRASO")).alias("ESTADOATRASO")
            )
            .alias("DV_AGG")
        )

        df_resultado = (
            df_stg_t_saldos_sl_solicitudcredito
            .join(
                df_dev_venc,
                col("DV_AGG.SALDOS_JTS_OID_DV") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .join(
                df_stg_t_datos_egp_ini,
                col("SALDOS_SOL_CREDS.JTS_OID") == col("DATOS_EGP_INI.SALDOS_JTS_OID"),
                "left"
            )
            .select(
                lit(datediff(lit(self.FEC_PROCESO), col("SALDOS_SOL_CREDS.C1628"))).alias("flg2"),
                lit(self.FEC_PROCESO).alias("fechaini"),
                col("SALDOS_SOL_CREDS.C1628"),
                col("SALDOS_SOL_CREDS.JTS_OID").alias("SALDOS_JTS_OID"),

                when(
                    col("SALDOS_SOL_CREDS.C4280") == "C",
                    col("DV_AGG.ESTADOATRASO")
                ).otherwise(
                    col("SALDOS_SOL_CREDS.C1728")
                ).alias("COD_EST_FIN"),

                when(
                    datediff(lit(self.FEC_PROCESO), col("SALDOS_SOL_CREDS.C1628")) < 0,
                    lit(0)
                ).otherwise(
                    datediff(lit(self.FEC_PROCESO), col("SALDOS_SOL_CREDS.C1628"))
                ).alias("DIA_MOR_FIN"),

                (
                    col("SALDOS_SOL_CREDS.C1604") * lit(-1)
                ).alias("SALD_CAP_FIN"),

                col("DATOS_EGP_INI.COD_EST_INI").alias("COD_EST_INI"),
                col("DATOS_EGP_INI.DIA_MOR_INI").alias("DIA_MOR_INI"),
                col("DATOS_EGP_INI.SALD_CAP_INI").alias("SALD_CAP_INI"),

                (
                    dayofmonth(last_day(to_date(lit(self.FEC_PROCESO)))) -
                    dayofmonth(to_date(lit(self.FEC_PROCESO)))
                ).alias("DIAS_REST")
            )
        )

        return df_resultado

    def get_stg_t_gastos_por_cuota_summary(self): 
        """Obtiene los datos de la tabla t_gastos_por_cuota_summary"""

        df_edy_gastos_por_cuota = self.get_edy_gastos_por_cuota()

        return df_edy_gastos_por_cuota.select(
            col("SALDOS_JTS_OID"),
            col("SALDO_GASTO")
        )

    def get_stg_t_co_monedas_01(self): 
        """Obtiene los datos de la tabla t_co_monedas_01"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CO_MONEDAS}/{self.CONS_TABLE_CO_MONEDAS}"
        return self.read_data(table_path).filter(col("tz_lock") == 0).select(
            col("C6399"),
            col("C6400"),
            col("C6401"),
            col("C6440"),
            col("C6431"),
            col("C6402")
        )
    
    def get_stg_tmp_egp_3(self): 
        """Obtiene los datos de la tabla tmp_egp_3"""

        fe_inicio_mes = trunc(to_date(lit(self.FEC_PROCESO)), "MM")
        fe_inicio_mes_siguiente = add_months(fe_inicio_mes, 1)

        df_stg_t_saldos_sl_solicitudcredito = self.get_stg_t_saldos_sl_solicitudcredito().alias("SALDOS_SOL_CREDS")

        df_stg_t_co_monedas_01 = self.get_stg_t_co_monedas_01().alias("MON")
        df_stg_t_dias_atraso_prestamos = self.get_stg_t_dias_atraso_prestamos()
        df_stg_t_prestamos_fogapi = self.get_stg_t_prestamos_fogapi()
        df_stg_t_gr_garantias = self.get_stg_t_gr_garantias()
        df_stg_t_pagos_parciales = self.get_stg_t_pagos_parciales()
        df_stg_t_prest_deuda_venc = self.get_stg_t_prest_deuda_venc()
        df_stg_t_prest_mora_total = self.get_stg_t_prest_mora_total()
        df_stg_t_prest_deuda_total = self.get_stg_t_prest_deuda_total()
        df_stg_t_datos_egp_ini = self.get_stg_t_datos_egp_ini()
        df_stg_t_bs_pays_detail_mes = self.get_stg_t_bs_pays_detail_mes()
        df_stg_t_bs_charge_detail_mes = self.get_stg_t_bs_charge_detail_mes()
        df_stg_t_cr_recupercartera = self.get_stg_t_cr_recupercartera()
        df_stg_t_historia_vista = self.get_stg_t_historia_vista()
        df_stg_t_datos_saldo_resuelto = self.get_stg_t_datos_saldo_resuelto()
        df_stg_t_gastos_por_cuota_summary = self.get_stg_t_gastos_por_cuota_summary()
        
        dias_atraso = (
            df_stg_t_dias_atraso_prestamos
            .groupBy(col("SALDOS_JTS_OID"))
            .agg(
                max(col("TOTA_DIAS_ATRASO")).alias("TOTA_DIAS_ATRASO"),
                max(col("MAX_DIAS_ATRASO")).alias("MAX_DIAS_ATRASO"),
                max(col("NRO_CUOTAS_VENC")).alias("NRO_CUOTAS_VENC")
            )
            .withColumnRenamed("SALDOS_JTS_OID", "SALDOS_JTS_OID_DIAS")
            .alias("DIAS")
        )

        prest_fogapi = (
            df_stg_t_prestamos_fogapi
            .groupBy(col("SALDO_JTS_OID"))
            .agg(
                max(
                    when(
                        (col("FOGAPI") != "FE") &
                        (col("ESTADO").isin("V", "S", "H")),
                        lit(1)
                    ).otherwise(lit(0))
                ).alias("IN_FOGAPI"),
                max(
                    when(
                        (col("FOGAPI") != "FE") &
                        (col("ESTADO").isin("V", "S", "H")),
                        col("ESTADO")
                    ).otherwise(lit("X"))
                ).alias("ST_FOGAPI")
            )
            .withColumnRenamed("SALDO_JTS_OID", "SALDO_JTS_OID_FOGAPI")
            .alias("FOGAPI")
        )

        gar_acti = (
            df_stg_t_gr_garantias
            .filter(col("ESTADO") == "V")
            .select(
                col("NROCREDITO").alias("NROCREDITO_GAR_ACTI")
            )
            .dropDuplicates()
            .withColumn("IN_GARA_ACTI", lit(1))
            .alias("GAR_ACTI")
        )

        gar_real = (
            df_stg_t_gr_garantias
            .filter(col("CLASGARANTIA") == "R")
            .select(
                col("NROCREDITO").alias("NROCREDITO_GAR_REAL")
            )
            .dropDuplicates()
            .withColumn("IN_GARA_REAL", lit(1))
            .alias("GAR_REAL")
        )

        pagos_parc = (
            df_stg_t_pagos_parciales
            .groupBy(col("SALDOS_JTS_OID"))
            .agg(
                max(col("FLAG_PAGO_PARCIAL")).alias("FLAG_PAGO_PARCIAL")
            )
            .withColumnRenamed("SALDOS_JTS_OID", "SALDOS_JTS_OID_PAGOS_PARC")
            .alias("PAGOS_PARC")
        )

        deuda_venc = (
            df_stg_t_prest_deuda_venc
            .groupBy(col("SALDOS_JTS_OID"))
            .agg(
                max(col("DEUDA_VENCIDA")).alias("DEUDA_VENCIDA")
            )
            .withColumnRenamed("SALDOS_JTS_OID", "SALDOS_JTS_OID_DEUDA_VENC")
            .alias("DEUDA_VENC")
        )

        mora_total = (
            df_stg_t_prest_mora_total
            .groupBy(col("PRCODPRES"))
            .agg(
                max(col("TOTAL_MORA")).alias("TOTAL_MORA")
            )
            .withColumnRenamed("PRCODPRES", "PRCODPRES_MORA")
            .alias("MORA_TOTAL")
        )

        deuda_total = (
            df_stg_t_prest_deuda_total
            .groupBy(col("SALDOS_JTS_OID"))
            .agg(
                max(col("DEUDA_TOTAL")).alias("DEUDA_TOTAL")
            )
            .withColumnRenamed("SALDOS_JTS_OID", "SALDOS_JTS_OID_DEUDA_TOTAL")
            .alias("DEUDA_TOTAL")
        )

        datos_egp_ini = (
            df_stg_t_datos_egp_ini
            .select(
                col("SALDOS_JTS_OID").alias("SALDOS_JTS_OID_EGP_INI"),
                col("DIA_MOR_INI")
            )
            .alias("EGP_INI")
        )

        pays_mes = (
            df_stg_t_bs_pays_detail_mes
            .groupBy(col("SALDOS_JTS_OID"))
            .agg(
                sum(
                    coalesce(col("CAPITALPAGADO"), lit(0)) +
                    coalesce(col("INTERESPAGADO"), lit(0)) +
                    coalesce(col("MORAPAGADA"), lit(0)) +
                    coalesce(col("INTERES_COMP_MORA"), lit(0)) +
                    coalesce(col("CAPITALADELANTADO"), lit(0))
                ).alias("MO_PAGO_TOTAL_PAYS"),

                sum(
                    coalesce(col("CAPITALPAGADO"), lit(0)) +
                    coalesce(col("CAPITALADELANTADO"), lit(0))
                ).alias("MO_PAGO_CAPI_MES")
            )
            .withColumnRenamed("SALDOS_JTS_OID", "SALDOS_JTS_OID_PAYS")
            .alias("PAYS")
        )

        charge_mes = (
            df_stg_t_bs_charge_detail_mes
            .groupBy(col("SALDOS_JTS_OID"))
            .agg(
                sum(
                    coalesce(col("TOTAL_PAGADO"), lit(0))
                ).alias("MO_PAGO_TOTAL_CHARGE")
            )
            .withColumnRenamed("SALDOS_JTS_OID", "SALDOS_JTS_OID_CHARGE")
            .alias("CHARGE")
        )

        cr_recuper_mes = (
            df_stg_t_cr_recupercartera
            .filter(
                (col("TIPOOPER") == "C") &
                (to_date(col("FECHAOPER")) >= fe_inicio_mes) &
                (to_date(col("FECHAOPER")) < fe_inicio_mes_siguiente)
            )
            .groupBy(col("NROPRESTAMO"))
            .agg(
                sum(
                    coalesce(col("MONTOCONDONADO"), lit(0))
                ).alias("MO_PAGO_COND_CAPI"),

                sum(
                    coalesce(col("MONTOCONDONADO"), lit(0)) +
                    coalesce(col("INTERCOND"), lit(0)) +
                    coalesce(col("MORACOND"), lit(0)) +
                    coalesce(col("CARGOSCOND"), lit(0))
                ).alias("MO_PAGO_COND_TOTA")
            )
            .withColumnRenamed("NROPRESTAMO", "NROPRESTAMO_REC_MES")
            .alias("REC_MES")
        )

        cr_recuper_hist = (
            df_stg_t_cr_recupercartera
            .filter(col("TIPOOPER") == "C")
            .select(
                col("NROPRESTAMO").alias("NROPRESTAMO_REC_HIST")
            )
            .dropDuplicates()
            .withColumn("IN_CONDONACION", lit(1))
            .alias("REC_HIST")
        )

        historia_vista = (
            df_stg_t_historia_vista
            .groupBy("CO_PRESTAMO")
            .agg(
                max(
                    coalesce(col("CREDITOS"), lit(0)) -
                    coalesce(col("DEBITOS"), lit(0))
                ).alias("MO_SALD_CUEN_INTE")
            )
            .withColumnRenamed("CO_PRESTAMO", "CO_PRESTAMO_HIV")
            .alias("HIV")
        )


        datos_saldo_resuelto = (
            df_stg_t_datos_saldo_resuelto
            .alias("DATOS_SAL_RES")
        )

        gastos_summary = (
            df_stg_t_gastos_por_cuota_summary
            .groupBy(col("SALDOS_JTS_OID"))
            .agg(
                sum(col("SALDO_GASTO")).alias("MO_TOTA_GAST")
            )
            .withColumnRenamed("SALDOS_JTS_OID", "SALDOS_JTS_OID_GASTOS")
            .alias("GASTOS")
        )

        df_base = (
            df_stg_t_saldos_sl_solicitudcredito
            .join(
                df_stg_t_co_monedas_01,
                col("MON.C6399") == col("SALDOS_SOL_CREDS.MONEDA"),
                "inner"
            )
            .join(
                dias_atraso,
                col("DIAS.SALDOS_JTS_OID_DIAS") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .join(
                prest_fogapi,
                col("FOGAPI.SALDO_JTS_OID_FOGAPI") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .join(
                gar_acti,
                col("GAR_ACTI.NROCREDITO_GAR_ACTI") == col("SALDOS_SOL_CREDS.CUENTA"),
                "left"
            )
            .join(
                pagos_parc,
                col("PAGOS_PARC.SALDOS_JTS_OID_PAGOS_PARC") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .join(
                deuda_venc,
                col("DEUDA_VENC.SALDOS_JTS_OID_DEUDA_VENC") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .join(
                mora_total,
                col("MORA_TOTAL.PRCODPRES_MORA") == col("SALDOS_SOL_CREDS.CUENTA"),
                "left"
            )
            .join(
                deuda_total,
                col("DEUDA_TOTAL.SALDOS_JTS_OID_DEUDA_TOTAL") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .join(
                datos_egp_ini,
                col("EGP_INI.SALDOS_JTS_OID_EGP_INI") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .join(
                pays_mes,
                col("PAYS.SALDOS_JTS_OID_PAYS") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .join(
                charge_mes,
                col("CHARGE.SALDOS_JTS_OID_CHARGE") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .join(
                cr_recuper_mes,
                col("REC_MES.NROPRESTAMO_REC_MES") == col("SALDOS_SOL_CREDS.CUENTA"),
                "left"
            )
            .join(
                cr_recuper_hist,
                col("REC_HIST.NROPRESTAMO_REC_HIST") == col("SALDOS_SOL_CREDS.CUENTA"),
                "left"
            )
            .join(
                historia_vista,
                col("HIV.CO_PRESTAMO_HIV") == col("SALDOS_SOL_CREDS.CUENTA"),
                "left"
            )
            .join(
                gar_real,
                col("GAR_REAL.NROCREDITO_GAR_REAL") == col("SALDOS_SOL_CREDS.CUENTA"),
                "left"
            )
            .join(
                datos_saldo_resuelto,
                col("DATOS_SAL_RES.SALDOS_JTS_OID") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .join(
                gastos_summary,
                col("GASTOS.SALDOS_JTS_OID_GASTOS") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
        )

        cond_saldo_resuelto = (
            (col("DATOS_SAL_RES.COD_EST_INI") == "C") |
            (
                (col("DATOS_SAL_RES.COD_EST_INI") != "C") &
                (col("DATOS_SAL_RES.COD_EST_FIN") == "C")
            ) |
            (
                (
                    floor(
                        (
                            (
                                when(
                                    col("DATOS_SAL_RES.DIA_MOR_FIN") == 0,
                                    lit(0)
                                ).otherwise(
                                    col("DATOS_SAL_RES.DIA_MOR_FIN") +
                                    col("DATOS_SAL_RES.DIAS_REST")
                                )
                            ) - lit(1)
                        ) / lit(30)
                    )
                    -
                    floor(
                        (col("DATOS_SAL_RES.DIA_MOR_INI") - lit(1)) / lit(30)
                    )
                ) > 0
            )
        )

        monto_resuelto_expr = (
            when(
                col("EGP_INI.SALDOS_JTS_OID_EGP_INI").isNull(),
                col("SALDOS_SOL_CREDS.C1604") * lit(-1)
            )
            .otherwise(
                col("DATOS_SAL_RES.SALD_CAP_INI") -
                when(
                    cond_saldo_resuelto,
                    when(
                        col("DATOS_SAL_RES.SALD_CAP_FIN") > col("DATOS_SAL_RES.SALD_CAP_INI"),
                        col("DATOS_SAL_RES.SALD_CAP_INI")
                    ).otherwise(
                        coalesce(col("DATOS_SAL_RES.SALD_CAP_FIN"), lit(0))
                    )
                ).otherwise(lit(0))
            )
        )

        df_resultado = (
            df_base
            .select(
                col("SALDOS_SOL_CREDS.JTS_OID").alias("SALDOS_JTS_OID"),
                lit(col("EGP_INI.SALDOS_JTS_OID_EGP_INI").isNull()).alias("flg_1"),
                lit(cond_saldo_resuelto).alias("flg"),
                col("EGP_INI.SALDOS_JTS_OID_EGP_INI").alias("SALDOS_JTS_OID_EGP_INI"),
                col("SALDOS_SOL_CREDS.C1604"),
                col("DATOS_SAL_RES.SALD_CAP_INI"),
                col("DATOS_SAL_RES.SALD_CAP_FIN"),
                col("DATOS_SAL_RES.COD_EST_INI"),
                col("DATOS_SAL_RES.COD_EST_FIN"),
                col("DATOS_SAL_RES.DIA_MOR_FIN"),
                col("DATOS_SAL_RES.DIAS_REST"),
                col("DATOS_SAL_RES.DIA_MOR_INI"),
                (
                    (
                        pow(
                            lit(1) + (col("SALDOS_SOL_CREDS.C6645") / lit(100)),
                            lit(12)
                        ) - lit(1)
                    ) * lit(100)
                ).alias("VL_TCEA"),
                col("DIAS.TOTA_DIAS_ATRASO").alias("NU_DIAS_TOTA_ATRA"),
                col("DIAS.MAX_DIAS_ATRASO").alias("NU_MAXI_DIAS_ATRA"),
                col("SALDOS_SOL_CREDS.C1644").alias("NU_CUOT_PACT"),
                col("SALDOS_SOL_CREDS.C5261").alias("CO_USUA_APRO_SOLI"),
                coalesce(col("FOGAPI.IN_FOGAPI"), lit(0)).alias("IN_FOGAPI"),
                coalesce(col("FOGAPI.ST_FOGAPI"), lit("X")).alias("ST_FOGAPI"),
                col("SALDOS_SOL_CREDS.JTS_OID").alias("IN_SALD_JTS_OID"),
                coalesce(col("GAR_ACTI.IN_GARA_ACTI"), lit(0)).alias("IN_GARA_ACTI"),
                coalesce(col("PAGOS_PARC.FLAG_PAGO_PARCIAL"), lit(0)).alias("IN_PAGO_PARC"),
                (
                    coalesce(col("DEUDA_VENC.DEUDA_VENCIDA"), lit(0)) +
                    coalesce(col("MORA_TOTAL.TOTAL_MORA"), lit(0))
                ).alias("MO_DEUD_VENC"),
                (
                    coalesce(col("DEUDA_TOTAL.DEUDA_TOTAL"), lit(0)) +
                    coalesce(col("MORA_TOTAL.TOTAL_MORA"), lit(0))
                ).alias("MO_TOTA_DEUD"),
                col("DIAS.NRO_CUOTAS_VENC").alias("NU_CUOT_VENC"),
                col("SALDOS_SOL_CREDS.C1628").alias("FE_VENC_CUOT"),
                coalesce(col("EGP_INI.DIA_MOR_INI"), lit(0)).alias("NU_DIAS_MORA_MES"),
                (
                    coalesce(col("PAYS.MO_PAGO_TOTAL_PAYS"), lit(0)) +
                    coalesce(col("CHARGE.MO_PAGO_TOTAL_CHARGE"), lit(0))
                ).alias("MO_PAGO_TOTA_MES"),
                coalesce(col("PAYS.MO_PAGO_CAPI_MES"), lit(0)).alias("MO_PAGO_CAPI_MES"),
                coalesce(col("REC_MES.MO_PAGO_COND_CAPI"), lit(0)).alias("MO_PAGO_COND_CAPI"),
                coalesce(col("REC_MES.MO_PAGO_COND_TOTA"), lit(0)).alias("MO_PAGO_COND_TOTA"),
                coalesce(col("REC_HIST.IN_CONDONACION"), lit(0)).alias("IN_CONDONACION"),
                coalesce(col("HIV.MO_SALD_CUEN_INTE"), lit(0)).alias("MO_SALD_CUEN_INTE"),
                coalesce(col("GAR_REAL.IN_GARA_REAL"), lit(0)).alias("IN_GARA_REAL"),
                coalesce(monto_resuelto_expr, lit(0)).alias("MO_SALD_RESU"),
                col("SALDOS_SOL_CREDS.TASA_ICMORA").alias("VL_TASA_ICM"),
                col("SALDOS_SOL_CREDS.C1633").alias("VL_TASA_INTE_MOR"),
                coalesce(col("GASTOS.MO_TOTA_GAST"), lit(0)).alias("MO_TOTA_GAST"),
                col("MON.C6440").alias("VL_TIPO_CAMB_OFIC"),
                col("SALDOS_SOL_CREDS.TASA_MINIMA").alias("VL_TASA_MINI"),
                col("SALDOS_SOL_CREDS.TASA_MAXIMA").alias("VL_TASA_MAXI"),
                col("SALDOS_SOL_CREDS.INTE_COMP").alias("INTE_COMP"),
                col("SALDOS_SOL_CREDS.MORA_CONT").alias("MORA_CONT")
            )
        )

        return df_resultado
    
    def get_stg_t_rc_rcc_cuerpo_total(self): 
        """Obtiene los datos de la tabla t_rc_rcc_cuerpo_total"""

        df_edy_rc_rcc_cuerpo = self.get_edy_rc_rcc_cuerpo().alias("RRC")
        df_edy_rc_cuentascontables = self.get_edy_rc_cuentascontables().alias("C")

        c_filtrado = (
            df_edy_rc_cuentascontables
            .filter(
                (col("C.ESTADO") == "A") &
                (
                    (
                        (substring(col("C.CODCTAPADRE"), 1, 1) == "1") &
                        (col("C.CODCTAACUM") == "00001")
                    ) |
                    (
                        (substring(col("C.CODCTADETALLE"), 1, 2) == "81") &
                        (substring(col("C.CODCTADETALLE"), 4, 11) == "3")
                    )
                )
            )
            .select(
                substring(col("C.CODCTADETALLE"), 1, 2).alias("CODCTADETALLE_1_2"),
                substring(col("C.CODCTADETALLE"), 4, 11).alias("CODCTADETALLE_4_11")
            )
            .dropDuplicates()
            .alias("C_FILTRADO")
        )

        sq_1 = (
            df_edy_rc_rcc_cuerpo
            .filter(
                col("RRC.TZ_LOCK") == 0
            )
            .join(
                c_filtrado,
                (
                    col("C_FILTRADO.CODCTADETALLE_1_2")
                    == substring(col("RRC.CODCTACONTABLE"), 1, 2)
                ) &
                (
                    col("C_FILTRADO.CODCTADETALLE_4_11")
                    == substring(col("RRC.CODCTACONTABLE"), 4, 11)
                ),
                "left_semi"
            )
            .select(
                col("RRC.CODCLIENTESBS").alias("CODCLIENTESBS"),
                col("RRC.CODEMPRESA").alias("CODEMPRESA"),
                col("RRC.SALDO").alias("SALDO")
            )
        )

        sq_2 = (
            df_edy_rc_rcc_cuerpo
            .filter(
                (col("RRC.CODCTACONTABLE").like("81_302%")) &
                (col("RRC.TZ_LOCK") == 0)
            )
            .select(
                col("RRC.CODCLIENTESBS").alias("CODCLIENTESBS"),
                col("RRC.CODEMPRESA").alias("CODEMPRESA"),
                col("RRC.SALDO").alias("SALDO")
            )
        )

        sq = sq_1.unionByName(sq_2)

        df_resultado = (
            sq
            .groupBy(
                col("CODCLIENTESBS")
            )
            .agg(
                countDistinct(col("CODEMPRESA")).alias("NU_CANT_EMPR"),
                (sum(col("SALDO")) / 100).alias("MO_SALDO_TOTAL")
            )
        )

        return df_resultado
    
    def get_stg_tmp_egp_4(self): 
        """Obtiene los datos de la tabla tmp_egp_4"""

        saldos_sol_creds = self.get_stg_t_saldos_sl_solicitudcredito().alias("SALDOS_SOL_CREDS")

        clientes = self.get_stg_t_cl_clientes_01().alias("CLIENTES")

        cli = self.get_stg_t_rel_cliente_sbs().alias("CLI")

        rccuerpo_total = self.get_stg_t_rc_rcc_cuerpo_total().alias("RCCUERPOTOTAL")

        bshp_summary = self.get_stg_t_bs_historia_plazo_summary().alias("BSHPSUMMARY")

        df_clientes = (
            clientes
            .select(
                col("CLIENTES.C0902").alias("CO_CLIENTE"),
                col("CLIENTES.C1058").alias("CO_ACTI_ECON_INTE"),
                col("CLIENTES.C1840").alias("CO_SECT_ECON_ALTE"),
                col("CLIENTES.C3205").alias("CO_ACTIVIDAD"),
                col("CLIENTES.C1034").alias("CO_ACTI_CIIU"),
                col("CLIENTES.C1584").alias("CO_SECT_ECON_INTE"),
                when(
                    col("CLIENTES.EMPLEADO") == "E",
                    lit(1)
                ).otherwise(lit(0)).alias("IN_EMPLEADO")
            )
            .alias("CLIENTES_AUX")
        )

        df_rcc_total = (
            cli
            .join(
                rccuerpo_total,
                col("CLI.CO_CLIE_SBS") == col("RCCUERPOTOTAL.CODCLIENTESBS"),
                "left"
            )
            .groupBy(
                col("CLI.CO_CLIENTE").alias("CO_CLIENTE_RCC")
            )
            .agg(
                max(col("RCCUERPOTOTAL.NU_CANT_EMPR")).alias("NU_CANT_EMPR"),
                max(col("RCCUERPOTOTAL.MO_SALDO_TOTAL")).alias("MO_SALDO_TOTAL")
            )
            .alias("RCC_TOTAL")
        )

        df_bshp_summary = (
            bshp_summary
            .groupBy(
                col("BSHPSUMMARY.SALDOS_JTS_OID").alias("SALDOS_JTS_OID_BSHP")
            )
            .agg(
                max(col("BSHPSUMMARY.FECHAPROCESOMOV")).alias("FECHAPROCESOMOV")
            )
            .alias("BSHP")
        )

        df_resultado = (
            saldos_sol_creds
            .join(
                df_clientes,
                col("CLIENTES_AUX.CO_CLIENTE") == col("SALDOS_SOL_CREDS.C1803"),
                "left"
            )
            .join(
                df_rcc_total,
                col("RCC_TOTAL.CO_CLIENTE_RCC") == col("SALDOS_SOL_CREDS.C1803"),
                "left"
            )
            .join(
                df_bshp_summary,
                col("BSHP.SALDOS_JTS_OID_BSHP") == col("SALDOS_SOL_CREDS.JTS_OID"),
                "left"
            )
            .select(
                col("SALDOS_SOL_CREDS.JTS_OID").alias("SALDOS_JTS_OID"),

                col("CLIENTES_AUX.CO_ACTI_ECON_INTE").alias("CO_ACTI_ECON_INTE"),
                col("CLIENTES_AUX.CO_SECT_ECON_ALTE").alias("CO_SECT_ECON_ALTE"),
                col("CLIENTES_AUX.CO_ACTIVIDAD").alias("CO_ACTIVIDAD"),
                col("CLIENTES_AUX.CO_ACTI_CIIU").alias("CO_ACTI_CIIU"),
                col("CLIENTES_AUX.CO_SECT_ECON_INTE").alias("CO_SECT_ECON_INTE"),
                col("CLIENTES_AUX.IN_EMPLEADO").alias("IN_EMPLEADO"),

                coalesce(
                    col("RCC_TOTAL.NU_CANT_EMPR"),
                    lit(0)
                ).alias("NU_EMPR_REPO"),

                coalesce(
                    col("RCC_TOTAL.MO_SALDO_TOTAL"),
                    lit(0)
                ).alias("MO_TOTA_SIST_FINA"),

                col("BSHP.FECHAPROCESOMOV").alias("FE_PROC_ULTI_PAGO")
            )
        )


        return df_resultado
    
    def get_stg_t_tmp_montos_saldos(self): 
        """Obtiene los datos de la tabla t_tmp_montos_saldos"""
        df_edy_saldos = self.get_edy_saldos_activos().select(
                col("JTS_OID").alias("SALDO_JTS_OID"),
                col("C1604"),
                col("C1608"),
                col("C6393"),
                col("C1711"),
                col("ENTREGA_ACUENTA_ICMORA").alias("ENTREGA_ACUENTA_ICM"),
                col("C1615"),
                col("ICMORA_PERCIBIDO_SALDOS").alias("ICMORA_PERCIBIDO_SALD"),
                col("C1704"),
                col("C1803"),
                col("CUENTA"),
                col("C1625"),
                col("OPERACION"),
                col("ORDINAL"),
                col("PRODUCTO"),
                col("C1661")
        )

        return df_edy_saldos
    
    def get_stg_t_tmp_montos_saldos_hist(self): 
        """Obtiene los datos de la tabla t_tmp_montos_saldos_hist"""
        return self.get_stg_t_tmp_montos_saldos().select(
            lit(self.FEC_PROCESO).alias("FECHA_SALDO"),
            col("SALDO_JTS_OID"),
            col("C1604").alias("C1604_HIST"),
            col("C1608").alias("C1608_HIST"),
            col("C6393").alias("C6393_HIST"),
            col("C1711").alias("C1711_HIST"),
            col("ENTREGA_ACUENTA_ICM").alias("ENTREGA_ACUENTA_ICM_HIST"),
            col("C1615").alias("C1615_HIST"),
            col("ICMORA_PERCIBIDO_SALD").alias("ICMORA_PERCIBIDO_SALD_HIST"),
            col("C1704").alias("C1704_HIST"),
            col("C1803").alias("C1803_HIST"),
            col("CUENTA").alias("CUENTA_HIST"),
            col("C1625").alias("C1625_HIST")
        )
    
    def get_stg_t_prestamos_modificados(self): 
        """Obtiene los datos de la tabla t_prestamos_modificados"""

        df_stg_t_tmp_montos_saldos = self.get_stg_t_tmp_montos_saldos().alias("s")
        df_stg_t_tmp_montos_saldos_hist = self.get_stg_t_tmp_montos_saldos_hist()

        max_fecha_saldo = (
            df_stg_t_tmp_montos_saldos_hist
            .agg(max(col("FECHA_SALDO")).alias("MAX_FECHA_SALDO"))
        )

        sh2 = (
            df_stg_t_tmp_montos_saldos_hist.alias("sh")
            .crossJoin(max_fecha_saldo)
            .filter(col("sh.FECHA_SALDO") == col("MAX_FECHA_SALDO"))
            .select(
                col("sh.SALDO_JTS_OID").alias("SALDO_JTS_OID"),
                col("sh.C1604_HIST").alias("C1604_HIST"),
                col("sh.C1608_HIST").alias("C1608_HIST"),
                col("sh.C6393_HIST").alias("C6393_HIST"),
                col("sh.C1711_HIST").alias("C1711_HIST"),
                col("sh.ENTREGA_ACUENTA_ICM_HIST").alias("ENTREGA_ACUENTA_ICM_HIST"),
                col("sh.C1615_HIST").alias("C1615_HIST"),
                col("sh.ICMORA_PERCIBIDO_SALD_HIST").alias("ICMORA_PERCIBIDO_SALD_HIST"),
                col("sh.C1625_HIST").alias("C1625_HIST")
            )
            .alias("sh2")
        )

        sm = (
            df_stg_t_tmp_montos_saldos
            .join(
                sh2,
                col("sh2.SALDO_JTS_OID") == col("s.SALDO_JTS_OID"),
                "left"
            )
            .select(
                col("s.SALDO_JTS_OID").alias("SALDO_JTS_OID"),

                col("s.C1604").alias("C1604"),
                coalesce(col("sh2.C1604_HIST"), lit(0)).alias("C1604_HIST"),

                col("s.C1608").alias("C1608"),
                coalesce(col("sh2.C1608_HIST"), lit(0)).alias("C1608_HIST"),

                col("s.C6393").alias("C6393"),
                coalesce(col("sh2.C6393_HIST"), lit(0)).alias("C6393_HIST"),

                col("s.C1711").alias("C1711"),
                coalesce(col("sh2.C1711_HIST"), lit(0)).alias("C1711_HIST"),

                col("s.ENTREGA_ACUENTA_ICM").alias("ENTREGA_ACUENTA_ICM"),
                coalesce(col("sh2.ENTREGA_ACUENTA_ICM_HIST"), lit(0)).alias("ENTREGA_ACUENTA_ICM_HIST"),

                col("s.C1615").alias("C1615"),
                coalesce(col("sh2.C1615_HIST"), lit(0)).alias("C1615_HIST"),

                col("s.ICMORA_PERCIBIDO_SALD").alias("ICMORA_PERCIBIDO_SALD"),
                coalesce(col("sh2.ICMORA_PERCIBIDO_SALD_HIST"), lit(0)).alias("ICMORA_PERCIBIDO_SALD_HIST"),

                col("s.C1625").alias("C1625"),
                coalesce(
                    col("sh2.C1625_HIST"),
                    to_date(lit("99991231"), "yyyyMMdd")
                ).alias("C1625_HIST")
            )
            .alias("sm")
        )

        df_cambios_actual_vs_hist = (
            sm
            .filter(
                (col("sm.C1604") != col("sm.C1604_HIST")) |
                (col("sm.C1608") != col("sm.C1608_HIST")) |
                (col("sm.C6393") != col("sm.C6393_HIST")) |
                (col("sm.C1711") != col("sm.C1711_HIST")) |
                (col("sm.ENTREGA_ACUENTA_ICM") != col("sm.ENTREGA_ACUENTA_ICM_HIST")) |
                (col("sm.C1615") != col("sm.C1615_HIST")) |
                (col("sm.ICMORA_PERCIBIDO_SALD") != col("sm.ICMORA_PERCIBIDO_SALD_HIST")) |
                (col("sm.C1625") != col("sm.C1625_HIST"))
            )
            .select(
                col("sm.SALDO_JTS_OID").alias("SALDOS_JTS_OID")
            )
        )

        df_hist_sin_actual = (
            sh2
            .join(
                df_stg_t_tmp_montos_saldos.select(
                    col("s.SALDO_JTS_OID").alias("SALDO_JTS_OID_ACTUAL")
                ),
                col("sh2.SALDO_JTS_OID") == col("SALDO_JTS_OID_ACTUAL"),
                "left_anti"
            )
            .select(
                col("sh2.SALDO_JTS_OID").alias("SALDOS_JTS_OID")
            )
        )

        df_resultado = (
            df_cambios_actual_vs_hist
            .unionByName(df_hist_sin_actual)
        )

        return df_resultado
    
    def get_stg_t_bs_planpagos(self): 
        """Obtiene los datos de la tabla t_bs_planpagos"""

        df_pm = self.get_stg_t_prestamos_modificados().select(
            col("SALDOS_JTS_OID").alias("SALDO_JTS_OID")
        )

        df_pp = self.get_edy_bs_planpagos()

        df_resultado = (
            df_pm.alias("pm")
            .join(
                df_pp.alias("pp"),
                col("pp.SALDO_JTS_OID") == col("pm.SALDO_JTS_OID"),
                "inner"
            )
            .select(
                col("pp.CLIENTE").alias("CLIENTE"),
                col("pp.MONEDA").alias("MONEDA"),
                col("pp.CUENTA").alias("CUENTA"),
                col("pp.SUCURSAL").alias("SUCURSAL"),
                col("pp.C2297").alias("C2297"),
                col("pp.OPERACION").alias("OPERACION"),
                col("pp.ORDINAL").alias("ORDINAL"),
                col("pp.C2300").alias("C2300"),
                col("pp.C2301").alias("C2301"),
                col("pp.C2302").alias("C2302"),
                col("pp.C2304").alias("C2304"),
                col("pp.C2305").alias("C2305"),
                col("pp.C2309").alias("C2309"),
                col("pp.C2310").alias("C2310"),
                col("pp.SALDO_JTS_OID"),
                col("pp.FECHACANCELACION").alias("FECHACANCELACION")
            )
        )

        return df_resultado
    
    def get_ods_tp_cronogramas_creditos(self): #check

        fe_proceso = self.FEC_PROCESO

        df_stg_t_saldos_activos, df_stg_t_saldos_pasivos = self.get_stg_t_saldos()

        df_stg_t_saldos_activos_f = df_stg_t_saldos_activos.select(
            col("C1604"),
            col("JTS_OID"),
            col("CUENTA"),
            col("PRODUCTO"),
            col("SUCURSAL"),
            col("MONEDA"),
            col("C1803")
        )


        df_stg_t_saldos_pasivos_f = df_stg_t_saldos_pasivos.select(
            col("C1604"),
            col("JTS_OID"),
            col("CUENTA"),
            col("PRODUCTO"),
            col("SUCURSAL"),
            col("MONEDA"),
            col("C1803")
        )

        df_stg_t_saldos = df_stg_t_saldos_activos_f.unionByName(df_stg_t_saldos_pasivos_f).alias("S")
        df_stg_t_bs_planpagos = self.get_stg_t_bs_planpagos().alias("PP")

        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_MD_CRONOGRAMAS_CREDITOS}/{self.CONS_TABLE_MD_CRONOGRAMAS_CREDITOS}"
        )
        df_ods_md_cronogramas_creditos = self.read_data(table_path)
        in_esta_cuot_expr = (
            when(
                (
                    (col("PP.C2304") < 0) |
                    (
                        ((col("PP.C2309") + col("PP.C2310")) <= 0) &
                        ((col("PP.C2304") + col("PP.C2305")) >= 0)
                    )
                ) |
                (col("S.C1604") == 0),
                lit(1)
            ).otherwise(lit(0))
        )

        # Query principal
        df_resultado_1 = (
            df_stg_t_saldos
            .join(
                df_stg_t_bs_planpagos,
                col("S.JTS_OID") == col("PP.SALDO_JTS_OID"),
                "inner"
            )
            .filter(
                col("S.PRODUCTO") != 240
            )
            .select(
                col("S.CUENTA").alias("NU_PRESTAMO"), #desencriptar 
                col("S.PRODUCTO").alias("CO_PRODUCTO"),
                col("S.SUCURSAL").alias("CO_SUCURSAL"),
                col("S.MONEDA").alias("CO_MONEDA"),
                col("S.C1803").alias("CO_CLIENTE"),
                col("S.JTS_OID").alias("IN_SALD_JTS_OID"),

                col("PP.C2300").alias("NU_CUOTA"),

                in_esta_cuot_expr.alias("IN_ESTA_CUOT"),

                when(
                    (col("PP.C2302") < fe_proceso) &
                    (in_esta_cuot_expr == 0),
                    datediff(lit(fe_proceso), col("PP.C2302"))
                ).when(
                    in_esta_cuot_expr == 1,
                    datediff(col("PP.FECHACANCELACION"), col("PP.C2302"))
                ).otherwise(
                    lit(0)
                ).alias("NU_DIAS_ATRA_CUOT"),

                col("PP.C2302").alias("FE_VENC_CUOT"),
                col("PP.C2304").alias("MO_CAPI_CUOT"),
                col("PP.C2305").alias("MO_INTE_CUOT"),
                col("PP.C2309").alias("MO_SALD_CAPI_CUOT"),
                col("PP.C2310").alias("MO_SALD_INTE_CUOT"),
                col("PP.FECHACANCELACION").alias("FE_ULTI_PAGO_CUOT")
            )
        )

        df_resultado_2 = df_ods_md_cronogramas_creditos.select(
            col("NU_PRESTAMO"),
            col("CO_PRODUCTO"), 
            col("CO_SUCURSAL"), 
            col("CO_MONEDA"), 
            col("CO_CLIENTE"), 
            col("IN_SALD_JTS_OID"), 
            col("NU_CUOTA"), 
            col("IN_ESTA_CUOT"), 
            col("NU_DIAS_ATRA_CUOT"), 
            col("FE_VENC_CUOT"), 
            col("MO_CAPI_CUOT"), 
            col("MO_INTE_CUOT"), 
            col("MO_SALD_CAPI_CUOT"), 
            col("MO_SALD_INTE_CUOT"), 
            col("FE_ULTI_PAGO_CUOT")
        )

        df_resultado = df_resultado_1.union(df_resultado_2)

        return df_resultado
    
    def get_stg_t_sm_cronograma(self): 
        """Obtiene los datos de la tabla t_sm_cronograma"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_SM_CRONOGRAMA}/{self.CONS_TABLE_SM_CRONOGRAMA}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_stg_t_sm_cronograma_agg(self): 
        """Obtiene los datos de la tabla t_sm_cronograma_agg"""
        df_stg_t_sm_cronograma = self.get_stg_t_sm_cronograma()

        return df_stg_t_sm_cronograma.groupBy(
            col("JTS_OID_PTMO"),
            col("NRO_CUOTA_PTMO")
        ).agg(
            max(col("PRIMA_PTMO_SALDO")).alias("PRIMA_PTMO_SALDO")
        )

    def get_stg_t_saldos_egprec(self): 
        """Obtiene los datos de la tabla t_saldos_egprec"""

        df_edy_saldos = self.get_edy_saldos_activos().select(
            col("JTS_OID"),
            col("ESQUEMA_MORA"),
            col("MONEDA"),
            col("C1625")
        )

        return df_edy_saldos

    def get_ods_tp_ud_egp_01(self): 
        """Obtiene los datos de la tabla tp_ud_egp_01"""

        w_tp_crong_cred = (
            self.get_ods_tp_cronogramas_creditos().alias("BPP")
            .filter(
                (col("BPP.MO_SALD_CAPI_CUOT") != 0) |
                (col("BPP.MO_SALD_INTE_CUOT") != 0)
            )
            .select(
                col("BPP.IN_SALD_JTS_OID").alias("IN_SALD_JTS_OID"),
                col("BPP.FE_VENC_CUOT").alias("FE_VENC_CUOT"),
                col("BPP.NU_CUOTA").alias("NU_CUOTA")
            )
            .alias("BPP")
        )

        df_stg_t_saldos_activos, df_stg_t_saldos_pasivos = self.get_stg_t_saldos()

        w_t_saldos = (
            df_stg_t_saldos_activos.alias("SALDOS")
            .filter(
                (col("SALDOS.C9314") == 5) &
                (col("SALDOS.C1604") < 0)
            )
            .select(
                col("SALDOS.JTS_OID").alias("JTS_OID"),
                col("SALDOS.C1645").alias("C1645")
            )
            .alias("SALDOS")
        )

        df_stg_t_saldos_egprec = self.get_stg_t_saldos_egprec().alias("SAL")

        df_resultado = (
            w_tp_crong_cred
            .join(
                df_stg_t_saldos_egprec,
                col("SAL.JTS_OID") == col("BPP.IN_SALD_JTS_OID"),
                "inner"
            )
            .join(
                w_t_saldos,
                col("SALDOS.JTS_OID") == col("SAL.JTS_OID"),
                "inner"
            )
            .groupBy(
                col("BPP.IN_SALD_JTS_OID")
            )
            .agg(
                max(col("SAL.ESQUEMA_MORA")).alias("CO_ESQU_MORA"),
                max(col("SAL.MONEDA")).alias("CO_MONEDA"),
                max(col("SAL.C1625")).alias("FE_CONDONACION"),
                max(col("SALDOS.C1645")).alias("NU_CUOT_PAGA"),

                max(
                    when(
                        col("BPP.FE_VENC_CUOT") <= to_date(lit(self.FEC_PROCESO)),
                        col("BPP.NU_CUOTA")
                    ).otherwise(lit(0))
                ).alias("NU_MAXI_CUOT_VENC")
            )
        )

        return df_resultado
    
    def get_stg_t_gastos_por_cuota_egprec(self): 
        """Obtiene los datos de la tabla t_gastos_por_cuota_egprec"""

        return self.get_edy_gastos_por_cuota().filter((col("SALDO_GASTO") != 0) & (col("NUMERO_CUOTA").isNotNull())).select(
            col("SALDOS_JTS_OID"),
            col("SALDO_GASTO"),
            col("NUMERO_CUOTA"),
            col("CARGO_ID")
        )

    def get_stg_t_ci_cargos_egprec(self): 
        """Obtiene los datos de la tabla t_ci_cargos_egprec"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CI_CARGOS}/{self.CONS_TABLE_CI_CARGOS}"
        return self.read_data(table_path).filter(col("tz_lock") == 0).select(
            col("ID_CARGO"),
            col("AGRUPACION")
        )
    
    def get_stg_t_gast_cuo_carg_egprec(self): 
        """Obtiene los datos de la tabla t_gast_cuo_carg_egprec"""

        df_stg_t_gastos_por_cuota_egprec = self.get_stg_t_gastos_por_cuota_egprec().alias("GCI")
        df_stg_t_ci_cargos_egprec = self.get_stg_t_ci_cargos_egprec().alias("CIC")

        df_resultado = (
            df_stg_t_gastos_por_cuota_egprec
            .join(
                df_stg_t_ci_cargos_egprec,
                col("GCI.CARGO_ID") == col("CIC.ID_CARGO"),
                "left"
            )
            .groupBy(
                col("GCI.SALDOS_JTS_OID").alias("SALDO_JTS_OID"),
                col("GCI.NUMERO_CUOTA").alias("NUMERO_CUOTA")
            )
            .agg(
                sum(
                    col("GCI.SALDO_GASTO")
                ).alias("MO_SALD_CARG_CUO"),

                coalesce(
                    sum(
                        when(
                            (col("CIC.ID_CARGO").isNotNull()) &
                            (
                                (
                                    (col("CIC.AGRUPACION") != "M") &
                                    (col("CIC.AGRUPACION") != "E")
                                ) |
                                (col("CIC.AGRUPACION").isNull())
                            ),
                            col("GCI.SALDO_GASTO")
                        ).otherwise(lit(0))
                    ),
                    lit(0)
                ).alias("MO_SAL_CAR_CUO_SC"),

                coalesce(
                    sum(
                        when(
                            col("CIC.AGRUPACION") == "E",
                            col("GCI.SALDO_GASTO")
                        ).otherwise(lit(0))
                    ),
                    lit(0)
                ).alias("MO_COMI_CUO")
            )
        )

        return df_resultado

    def get_edy_ci_cargos_tarifas(self): 
        """Obtiene los datos de la tabla ci_cargos_tarifas"""
        table_path = (
            f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CI_CARGOS_TARIFAS}/{self.CONS_TABLE_CI_CARGOS_TARIFAS}"
        )
        return self.read_data(table_path).filter(col("tz_lock") == 0)
    
    def get_edy_tc_parametros(self): 
        """Obtiene los datos de la tabla tc_parametros"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_TC_PARAMETROS}/{self.CONS_TABLE_TC_PARAMETROS}"
        return self.read_data(table_path).filter(col("tz_lock") == 0)

    def get_stg_t_ci_cargos_tarifas_egprec(self): 
        """Obtiene los datos de la tabla t_ci_cargos_tarifas_egprec"""

        df_edy_ci_cargos_tarifas = self.get_edy_ci_cargos_tarifas().alias("CCT")
        df_edy_tc_parametros = self.get_edy_tc_parametros().alias("TP")

        tp_param = (
            df_edy_tc_parametros
            .filter(
                col("TP.TC_CODIGO") == 24
            )
            .agg(
                max(col("TP.TC_INTEGER")).alias("TC_INTEGER")
            )
            .alias("TP_PARAM")
        )

        df_resultado = (
            df_edy_ci_cargos_tarifas
            .join(
                tp_param,
                col("CCT.ID_CARGO") == col("TP_PARAM.TC_INTEGER"),
                "inner"
            )
            .filter(
                col("CCT.TZ_LOCK") == 0
            )
            .select(
                col("CCT.ID_CARGO").alias("ID_CARGO"),
                col("CCT.TASA").alias("TASA"),
                col("CCT.MONEDA").alias("MONEDA")
            )
        )

        return df_resultado

    
    def get_ods_tp_ud_egp_02(self): 
        """Obtiene los datos de la tabla tp_ud_egp_02"""

        w_cuo_act_crong = (
            self.get_ods_tp_cronogramas_creditos().alias("BPI")
            .join(
                self.get_stg_t_sm_cronograma_agg().alias("SMC"),
                (col("BPI.IN_SALD_JTS_OID") == col("SMC.JTS_OID_PTMO")) &
                (col("BPI.NU_CUOTA") == col("SMC.NRO_CUOTA_PTMO")),
                "left"
            )
            .filter(
                (col("BPI.MO_SALD_CAPI_CUOT") != 0) |
                (col("BPI.MO_SALD_INTE_CUOT") != 0)
            )
            .select(
                col("BPI.NU_PRESTAMO").alias("NU_PRESTAMO"),
                col("BPI.IN_SALD_JTS_OID").alias("IN_SALD_JTS_OID"),
                col("BPI.NU_CUOTA").alias("NU_CUOTA"),
                col("BPI.MO_SALD_CAPI_CUOT").alias("MO_SALD_CAPI"),
                col("BPI.MO_SALD_INTE_CUOT").alias("MO_SALD_INTE"),
                coalesce(col("SMC.PRIMA_PTMO_SALDO"), lit(0)).alias("PRIMA_PTMO_SALDO_CUO")
            )
            .alias("PC")
        )

        w_mora_cuo_max = (
            self.get_stg_t_rep_cartera_morosa().alias("M")
            .select(
                col("M.PRCODPRES").alias("NU_PRESTAMO"),
                col("M.PRCUOTAPR").alias("NRO_CUOTA"),
                col("M.INT_COM_MOR").alias("ICMORA"),
                col("M.MORA").alias("MORA")
            )
            .alias("M")
        )

        w_pagos_cuo_sal = (
            w_cuo_act_crong
            .join(
                self.get_ods_tp_ud_egp_01().alias("EGP"),
                col("EGP.IN_SALD_JTS_OID") == col("PC.IN_SALD_JTS_OID"),
                "inner"
            )
            .join(
                self.get_stg_t_gast_cuo_carg_egprec().alias("GCC"),
                (col("GCC.SALDO_JTS_OID") == col("PC.IN_SALD_JTS_OID")) &
                (col("GCC.NUMERO_CUOTA") == col("PC.NU_CUOTA")),
                "left"
            )
            .select(
                col("PC.NU_PRESTAMO").alias("NU_PRESTAMO"),
                col("PC.IN_SALD_JTS_OID").alias("IN_SALD_JTS_OID"),
                col("PC.NU_CUOTA").alias("NU_CUOTA"),
                col("PC.MO_SALD_CAPI").alias("MO_SALD_CAPI"),
                col("PC.MO_SALD_INTE").alias("MO_SALD_INTE"),
                col("PC.PRIMA_PTMO_SALDO_CUO").alias("PRIMA_PTMO_SALDO_CUO"),

                col("EGP.CO_ESQU_MORA").alias("ESQUEMA_MORA"),
                col("EGP.CO_MONEDA").alias("MONEDA"),
                col("EGP.FE_CONDONACION").alias("C1625"),
                col("EGP.NU_CUOT_PAGA").alias("NU_CUOT_PAGA"),
                col("EGP.NU_MAXI_CUOT_VENC").alias("NU_MAX_CUOT_VENC"),

                coalesce(col("GCC.MO_SAL_CAR_CUO_SC"), lit(0)).alias("MO_SALD_CARG_CUO_SC"),
                coalesce(col("GCC.MO_COMI_CUO"), lit(0)).alias("MO_COMI_CUO"),
                col("GCC.MO_SALD_CARG_CUO").alias("MO_SALD_CARG_CUO")
            )
            .alias("P")
        )

        cargos_tarifas = (
            self.get_stg_t_ci_cargos_tarifas_egprec()
            .alias("C")
        )

        pca_base = (
            w_pagos_cuo_sal
            .join(
                w_mora_cuo_max,
                (col("P.NU_PRESTAMO") == col("M.NU_PRESTAMO")) &
                (col("P.NU_CUOTA") == col("M.NRO_CUOTA")),
                "left"
            )
            .join(
                cargos_tarifas,
                col("C.MONEDA") == col("P.MONEDA"),
                "left"
            )
        )

        tf_ts_expr = (
            when(
                col("P.ESQUEMA_MORA") == "TF",
                coalesce(col("P.MO_SALD_INTE"), lit(0)) + coalesce(col("M.ICMORA"), lit(0))
            )
            .when(
                col("P.ESQUEMA_MORA") == "TS",
                coalesce(col("P.MO_SALD_INTE"), lit(0))
            )
            .otherwise(lit(0))
        )

        ts_tf_expr = (
            when(
                col("P.ESQUEMA_MORA") == "TS",
                coalesce(col("M.ICMORA"), lit(0)) + coalesce(col("M.MORA"), lit(0))
            )
            .when(
                col("P.ESQUEMA_MORA") == "TF",
                coalesce(col("M.MORA"), lit(0))
            )
            .otherwise(lit(0))
        )

        base_itf_expr = (
            coalesce(col("P.MO_SALD_CAPI"), lit(0)) +
            tf_ts_expr +
            ts_tf_expr +
            coalesce(col("P.MO_SALD_CARG_CUO_SC"), lit(0)) +
            coalesce(col("P.PRIMA_PTMO_SALDO_CUO"), lit(0)) +
            coalesce(col("P.MO_COMI_CUO"), lit(0))
        )

        monto_itf_base = base_itf_expr * col("C.TASA")

        trunc_1_expr = floor(monto_itf_base * lit(10)) / lit(10)

        trunc_2_expr = floor(monto_itf_base * lit(100)) / lit(100)

        mo_itf_expr = (
            trunc_1_expr +
            when(
                pmod(lit(100) * trunc_2_expr, lit(10)) >= 5,
                lit(0.05)
            ).otherwise(lit(0))
        )

        pca = (
            pca_base
            .select(
                col("P.IN_SALD_JTS_OID").alias("IN_SALD_JTS_OID"),
                col("P.NU_CUOTA").alias("NU_CUOTA"),
                col("P.NU_CUOT_PAGA").alias("NU_CUOT_PAGA"),
                col("P.NU_MAX_CUOT_VENC").alias("NU_MAX_CUOT_VENC"),
                col("P.C1625").alias("C1625"),

                coalesce(col("P.MO_SALD_CAPI"), lit(0)).alias("MO_SALD_CAPI"),
                coalesce(col("P.MO_SALD_INTE"), lit(0)).alias("MO_SALD_INTE"),

                col("P.MO_SALD_CARG_CUO").alias("MO_SALD_CARG_CUO"),

                tf_ts_expr.alias("TF_TS"),
                ts_tf_expr.alias("TS_TF"),

                coalesce(col("P.MO_SALD_CARG_CUO_SC"), lit(0)).alias("SALDO_GASTO_ITF"),
                coalesce(col("P.PRIMA_PTMO_SALDO_CUO"), lit(0)).alias("SM_PRIMA_PTMO_SALDO"),

                col("C.TASA").alias("TASA"),

                coalesce(col("P.MO_COMI_CUO"), lit(0)).alias("MO_COMISIONES"),

                mo_itf_expr.alias("MO_ITF"),

                col("M.ICMORA").alias("ICMORA"),
                col("M.MORA").alias("MORA")
            )
            .alias("PCA")
        )

        df_resultado = (
            pca
            .groupBy(
                col("PCA.IN_SALD_JTS_OID")
            )
            .agg(
                max(col("PCA.NU_MAX_CUOT_VENC")).alias("NU_MAXI_CUOT_VENC"),
                max(col("PCA.C1625")).alias("FE_CONDONACION"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") == (col("PCA.NU_CUOT_PAGA") + lit(1)),
                        col("PCA.MO_SALD_CAPI")
                    ).otherwise(lit(0))
                ).alias("MO_SALD_CAPI_CUOT"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") == (col("PCA.NU_CUOT_PAGA") + lit(1)),
                        col("PCA.MO_SALD_INTE")
                    ).otherwise(lit(0))
                ).alias("MO_INTE_COMP_CUOT"),

                sum(
                    when(
                        (col("PCA.NU_CUOTA") > col("PCA.NU_CUOT_PAGA")) &
                        (col("PCA.NU_CUOTA") <= col("PCA.NU_MAX_CUOT_VENC")),
                        col("PCA.MO_SALD_CAPI")
                    ).otherwise(lit(0))
                ).alias("MO_SALD_CAPI_VENC"),

                sum(
                    when(
                        (col("PCA.NU_CUOTA") > col("PCA.NU_CUOT_PAGA")) &
                        (col("PCA.NU_CUOTA") <= col("PCA.NU_MAX_CUOT_VENC")),
                        col("PCA.MO_SALD_INTE")
                    ).otherwise(lit(0))
                ).alias("MO_INTE_COMP_VENC"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") > col("PCA.NU_CUOT_PAGA"),
                        col("PCA.MO_SALD_CAPI")
                    ).otherwise(lit(0))
                ).alias("MO_SALD_CAPI_TOTA"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") > col("PCA.NU_CUOT_PAGA"),
                        col("PCA.MO_SALD_INTE")
                    ).otherwise(lit(0))
                ).alias("MO_SALD_INTE_TOTA"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") == (col("PCA.NU_CUOT_PAGA") + lit(1)),
                        coalesce(col("PCA.SALDO_GASTO_ITF"), lit(0)) +
                        coalesce(col("PCA.SM_PRIMA_PTMO_SALDO"), lit(0))
                    ).otherwise(lit(0))
                ).alias("MO_CARG_GAST_CUOT"),

                sum(
                    when(
                        (col("PCA.NU_CUOTA") > col("PCA.NU_CUOT_PAGA")) &
                        (col("PCA.NU_CUOTA") <= col("PCA.NU_MAX_CUOT_VENC")),
                        coalesce(col("PCA.SALDO_GASTO_ITF"), lit(0)) +
                        coalesce(col("PCA.SM_PRIMA_PTMO_SALDO"), lit(0))
                    ).otherwise(lit(0))
                ).alias("MO_CARG_GAST_VENC"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") > col("PCA.NU_CUOT_PAGA"),
                        coalesce(col("PCA.SALDO_GASTO_ITF"), lit(0)) +
                        coalesce(col("PCA.SM_PRIMA_PTMO_SALDO"), lit(0))
                    ).otherwise(lit(0))
                ).alias("MO_CARG_GAST_TOTA"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") == (col("PCA.NU_CUOT_PAGA") + lit(1)),
                        col("PCA.MO_COMISIONES")
                    ).otherwise(lit(0))
                ).alias("MO_COMI_CUOT"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") == (col("PCA.NU_CUOT_PAGA") + lit(1)),
                        col("PCA.MO_ITF")
                    ).otherwise(lit(0))
                ).alias("MO_ITF"),

                sum(
                    when(
                        (col("PCA.NU_CUOTA") >= (col("PCA.NU_CUOT_PAGA") + lit(1))) &
                        (col("PCA.NU_CUOTA") <= coalesce(col("PCA.NU_MAX_CUOT_VENC"), lit(0))) &
                        (coalesce(col("PCA.NU_MAX_CUOT_VENC"), lit(0)) > 0),
                        col("PCA.MO_COMISIONES")
                    ).otherwise(lit(0))
                ).alias("MO_COMI_VENC"),

                sum(
                    when(
                        (col("PCA.NU_CUOTA") >= (col("PCA.NU_CUOT_PAGA") + lit(1))) &
                        (col("PCA.NU_CUOTA") <= coalesce(col("PCA.NU_MAX_CUOT_VENC"), lit(0))) &
                        (coalesce(col("PCA.NU_MAX_CUOT_VENC"), lit(0)) > 0),
                        col("PCA.MO_ITF")
                    ).otherwise(lit(0))
                ).alias("MO_ITF_VENC"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") >= (col("PCA.NU_CUOT_PAGA") + lit(1)),
                        col("PCA.MO_COMISIONES")
                    ).otherwise(lit(0))
                ).alias("MO_COMI_TOTA"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") >= (col("PCA.NU_CUOT_PAGA") + lit(1)),
                        col("PCA.MO_ITF")
                    ).otherwise(lit(0))
                ).alias("MO_ITF_TOTA"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") == (col("PCA.NU_CUOT_PAGA") + lit(1)),
                        col("PCA.ICMORA")
                    ).otherwise(lit(0))
                ).alias("MO_INCO_MOCU_MXAT"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") == (col("PCA.NU_CUOT_PAGA") + lit(1)),
                        col("PCA.MORA")
                    ).otherwise(lit(0))
                ).alias("MO_INMO_CUOT_MXAT"),

                sum(
                    when(
                        (col("PCA.NU_CUOTA") > col("PCA.NU_CUOT_PAGA")) &
                        (col("PCA.NU_CUOTA") <= col("PCA.NU_MAX_CUOT_VENC")),
                        col("PCA.ICMORA")
                    ).otherwise(lit(0))
                ).alias("MO_INCO_MORA_VENC"),

                sum(
                    when(
                        (col("PCA.NU_CUOTA") > col("PCA.NU_CUOT_PAGA")) &
                        (col("PCA.NU_CUOTA") <= col("PCA.NU_MAX_CUOT_VENC")),
                        col("PCA.MORA")
                    ).otherwise(lit(0))
                ).alias("MO_MORA_VENC"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") > col("PCA.NU_CUOT_PAGA"),
                        col("PCA.ICMORA")
                    ).otherwise(lit(0))
                ).alias("MO_INTE_COMP_MORA"),

                sum(
                    when(
                        col("PCA.NU_CUOTA") > col("PCA.NU_CUOT_PAGA"),
                        col("PCA.MORA")
                    ).otherwise(lit(0))
                ).alias("MO_MORA_TOTA")
            )
        )

        return df_resultado
    
    def get_stg_t_calif_clientes(self):
        """Obtiene los datos de la tabla t_calif_clientes"""
        return self.get_edy_cl_clientes().select(
            col("C0902"),
            col("C1018"),
            col("C1046"),
            col("CALIFALIN"),
            col("CALIFINTERNAFINAL"),
            col("CALSISTFINAJUST"),
            col("C8273")
        )

    def get_stg_t_cr_criterios(self):
        """Obtiene los datos de la tabla cr_criterios"""
        table_path = f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_CR_CRITERIOS}/{self.CONS_TABLE_CR_CRITERIOS}"
        return self.read_data(table_path).filter(col("tz_lock") == 0).select(
            col("COD_CRITERIO"),
            col("TIPO_CREDITO")
        )

    def get_sql_query_vige(self,filtro_co_moneda,filtro_tipo_credito):
        """Obtiene los datos de la tabla sql_query"""

        tmp1 = (
            self.get_stg_tmp_egp_1()
            .filter(
                (col("MO_INTE_DEVE_VIGE") > 0) &
                (
                    (col("IN_REFINANCIADO") != 1)
                )
            )
            .select("SALDOS_JTS_OID", "MO_INTE_DEVE_VIGE", "CO_CLIENTE", "IN_REFINANCIADO")
            .alias("TMP1")
        )

        tmp2 = (
            self.get_stg_tmp_egp_2()
            .filter(col("CO_MONEDA") == filtro_co_moneda)
            .select("SALDOS_JTS_OID", "CO_PRODUCTO", "CO_CRITERIO")
            .alias("TMP2")
        )

        df_stg_t_saldos_activos, df_stg_t_saldos_pasivos = self.get_stg_t_saldos()

        # ⚡ Optimizar union
        s = (
            df_stg_t_saldos_activos
            .select("JTS_OID", "MB_CREDITO_VIGENTEO")
            .unionByName(
                df_stg_t_saldos_pasivos
                .select("JTS_OID", "MB_CREDITO_VIGENTEO")
            )
            .alias("S")
        )

        p = broadcast(
            self.get_stg_t_co_productos()
            .filter(col("C6271") == 140106)
            .select("C6250")
            .alias("P")
        )

        calif = broadcast(
            self.get_stg_t_calif_clientes()
            .select("C0902", "CALIFALIN")
            .alias("CALIF")
        )

        crit = broadcast(
            self.get_stg_t_cr_criterios()
            .filter(col("TIPO_CREDITO") == filtro_tipo_credito)
            .select("COD_CRITERIO")
            .alias("CRIT")
        )

        df = (
            tmp1
            .join(tmp2, "SALDOS_JTS_OID")
            .join(s, col("S.JTS_OID") == col("SALDOS_JTS_OID"))
            .join(p, col("C6250") == col("CO_PRODUCTO"))
            .join(calif, col("C0902") == col("CO_CLIENTE"))
            .join(crit, col("COD_CRITERIO") == col("CO_CRITERIO"))
            .filter(
                (
                    col("IN_REFINANCIADO") != 1
                ) | (col("MB_CREDITO_VIGENTEO") == "S")
            )
            .filter(
                coalesce(
                    when(trim(col("CALIFALIN")) != "", trim(col("CALIFALIN"))).cast("int"),
                    lit(0)
                ) <= 2
            )
            .select(
                col("SALDOS_JTS_OID"),
                col("MO_INTE_DEVE_VIGE")
            )
        )

        return df
    
    def get_sql_query_vige_2(self,filtro_co_moneda,filtro_tipo_credito):
        """Obtiene los datos de la tabla sql_query"""

        tmp1 = (
            self.get_stg_tmp_egp_1()
            .filter(col("MO_INTE_DEVE_VIGE") > 0)
            .select(
                "SALDOS_JTS_OID",
                "MO_INTE_DEVE_VIGE",
                "CO_CLIENTE",
                "IN_REFINANCIADO"
            )
            .alias("TMP1")
        )

        tmp2 = (
            self.get_stg_tmp_egp_2()
            .filter(col("CO_MONEDA") == filtro_co_moneda)
            .select(
                "SALDOS_JTS_OID",
                "CO_PRODUCTO",
                "CO_CRITERIO"
            )
            .alias("TMP2")
        )

        df_stg_t_saldos_activos, df_stg_t_saldos_pasivos = self.get_stg_t_saldos()

        s = (
            df_stg_t_saldos_activos.select("JTS_OID", "MB_CREDITO_VIGENTEO")
            .unionByName(
                df_stg_t_saldos_pasivos.select("JTS_OID", "MB_CREDITO_VIGENTEO")
            )
            .alias("S")
        )

        p = broadcast(
            self.get_stg_t_co_productos()
            .filter(col("C6271") != 140106)
            .select("C6250")
            .alias("P")
        )

        calif = broadcast(
            self.get_stg_t_calif_clientes()
            .select("C0902", "CALIFALIN")
            .alias("CALIF")
        )

        crit = broadcast(
            self.get_stg_t_cr_criterios()
            .filter(col("TIPO_CREDITO") == filtro_tipo_credito)
            .select("COD_CRITERIO")
            .alias("CRIT")
        )

        df = (
            tmp1
            .join(tmp2, "SALDOS_JTS_OID")
            .join(s, col("S.JTS_OID") == col("SALDOS_JTS_OID"))
            .join(p, col("C6250") == col("CO_PRODUCTO"))
            .join(calif, col("C0902") == col("CO_CLIENTE"))
            .join(crit, col("COD_CRITERIO") == col("CO_CRITERIO"))
        )

        df = df.filter(
            (
                (col("IN_REFINANCIADO") != 1) |
                (col("MB_CREDITO_VIGENTEO") == "S")
            )
        )

        df = df.filter(
            coalesce(
                when(trim(col("CALIFALIN")) != "", trim(col("CALIFALIN"))).cast("int"),
                lit(0)
            ) <= 2
        )

        df = df.select(
            col("SALDOS_JTS_OID"),
            col("MO_INTE_DEVE_VIGE")
        )

        return df
    
    def get_sql_query_vige_3(self,filtro_co_moneda,filtro_tipo_credito):
        """Obtiene los datos de la tabla sql_query"""

        tmp1 = (
            self.get_stg_tmp_egp_1()
            .filter(col("MO_INTE_DEVE_VIGE") > 0)
            .select(
                "SALDOS_JTS_OID",
                "MO_INTE_DEVE_VIGE",
                "CO_CLIENTE",
                "IN_REFINANCIADO"
            )
            .alias("TMP1")
        )

        tmp2 = (
            self.get_stg_tmp_egp_2()
            .filter(col("CO_MONEDA") == filtro_co_moneda)
            .select(
                "SALDOS_JTS_OID",
                "CO_CRITERIO"
            )
            .alias("TMP2")
        )

        df_stg_t_saldos_activos, df_stg_t_saldos_pasivos = self.get_stg_t_saldos()

        s = (
            df_stg_t_saldos_activos.select("JTS_OID", "MB_CREDITO_VIGENTEO")
            .unionByName(
                df_stg_t_saldos_pasivos.select("JTS_OID", "MB_CREDITO_VIGENTEO")
            )
            .alias("S")
        )

        calif = broadcast(
            self.get_stg_t_calif_clientes()
            .select("C0902", "CALIFALIN")
            .alias("CALIF")
        )

        crit = broadcast(
            self.get_stg_t_cr_criterios()
            .filter(col("TIPO_CREDITO").isin(filtro_tipo_credito))
            .select("COD_CRITERIO")
            .alias("CRIT")
        )

        df = (
            tmp1
            .join(tmp2, "SALDOS_JTS_OID")
            .join(s, col("S.JTS_OID") == col("SALDOS_JTS_OID"))
            .join(calif, col("C0902") == col("CO_CLIENTE"))
            .join(crit, col("COD_CRITERIO") == col("CO_CRITERIO"))
        )

        df = df.filter(
            (
                (col("IN_REFINANCIADO") != 1) |
                (col("MB_CREDITO_VIGENTEO") == "S")
            )
        )

        df = df.filter(
            coalesce(
                when(trim(col("CALIFALIN")) != "", trim(col("CALIFALIN"))).cast("int"),
                lit(0)
            ) <= 2
        )

        df = df.select(
            col("SALDOS_JTS_OID"),
            col("MO_INTE_DEVE_VIGE")
        )

        return df
    
    def get_sql_query_vige_4(self,filtro_co_moneda,filtro_tipo_credito):
        """Obtiene los datos de la tabla sql_query"""

        tmp1 = (
            self.get_stg_tmp_egp_1()
            .filter(col("MO_INTE_DEVE_VIGE") > 0)
            .select(
                "SALDOS_JTS_OID",
                "MO_INTE_DEVE_VIGE",
                "CO_CLIENTE",
                "IN_REFINANCIADO"
            )
            .alias("TMP1")
        )

        tmp2 = (
            self.get_stg_tmp_egp_2()
            .filter(col("CO_MONEDA") == filtro_co_moneda)
            .select(
                "SALDOS_JTS_OID",
                "CO_CRITERIO"
            )
            .alias("TMP2")
        )

        df_stg_t_saldos_activos, df_stg_t_saldos_pasivos = self.get_stg_t_saldos()

        s = (
            df_stg_t_saldos_activos.select("JTS_OID", "MB_CREDITO_VIGENTEO")
            .unionByName(
                df_stg_t_saldos_pasivos.select("JTS_OID", "MB_CREDITO_VIGENTEO")
            )
            .alias("S")
        )

        calif = broadcast(
            self.get_stg_t_calif_clientes()
            .select("C0902", "CALIFALIN")
            .alias("CALIF")
        )

        crit = broadcast(
            self.get_stg_t_cr_criterios()
            .filter(col("TIPO_CREDITO") == filtro_tipo_credito)
            .select("COD_CRITERIO")
            .alias("CRIT")
        )

        df = (
            tmp1
            .join(tmp2, "SALDOS_JTS_OID")  # join limpio
            .join(s, col("S.JTS_OID") == col("SALDOS_JTS_OID"))
            .join(calif, col("C0902") == col("CO_CLIENTE"))
            .join(crit, col("COD_CRITERIO") == col("CO_CRITERIO"))
        )

        df = df.filter(
            (
                (col("IN_REFINANCIADO") != 1) |
                (col("MB_CREDITO_VIGENTEO") == "S")
            )
        )

        df = df.filter(
            coalesce(
                when(trim(col("CALIFALIN")) != "", trim(col("CALIFALIN"))).cast("int"),
                lit(0)
            ) <= 2
        )

        df = df.select(
            "SALDOS_JTS_OID",
            "MO_INTE_DEVE_VIGE"
        )

        return df

    def get_stg_t_inte_deve_vige_raw(self):
        """Obtiene la unión cruda de t_inte_deve_vige sin distinct global."""

        dfs = [
            self.get_sql_query_vige(1, 7),
            self.get_sql_query_vige(1, 8),
            self.get_sql_query_vige(2, 8),
            self.get_sql_query_vige(2, 9),
            self.get_sql_query_vige(1, 9),
            self.get_sql_query_vige_2(1, 9),
            self.get_sql_query_vige_2(2, 9),
            self.get_sql_query_vige(2, 7),
            self.get_sql_query_vige(2, 10),
            self.get_sql_query_vige_2(2, 8),
            self.get_sql_query_vige_2(1, 10),
            self.get_sql_query_vige_2(1, 7),
            self.get_sql_query_vige_2(2, 7),
            self.get_sql_query_vige_2(1, 8),
            self.get_sql_query_vige(1, 10),
            self.get_sql_query_vige_3(1, [11, 12]),
            self.get_sql_query_vige_3(2, [11, 12]),
            self.get_sql_query_vige_4(1, 13),
            self.get_sql_query_vige_4(2, 13),
        ]

        return reduce(
            lambda df1, df2: df1.unionByName(df2),
            dfs
        )

    def get_stg_t_inte_deve_vige_agg(self):
        """
        Obtiene el máximo MO_INTE_DEVE_VIGE por SALDOS_JTS_OID.
        Evita hacer dropDuplicates completo sobre todas las columnas.
        """

        return (
            self.get_stg_t_inte_deve_vige_raw()
            .select(
                col("SALDOS_JTS_OID"),
                col("MO_INTE_DEVE_VIGE")
            )
            .groupBy("SALDOS_JTS_OID")
            .agg(
                max("MO_INTE_DEVE_VIGE").alias("MO_INTE_DEVE_VIGE")
            )
            .withColumnRenamed("SALDOS_JTS_OID", "SALDOS_JTS_OID_IDV")
            .alias("IDV")
        )


    def get_sql_query_sust(self,filtro_co_moneda):
        """Obtiene los datos de la tabla sql_query"""

        tmp1 = (
            self.get_stg_tmp_egp_1()
            .filter(col("MO_INTE_DEVE_SUST") > 0)
            .select(
                "SALDOS_JTS_OID",
                "MO_INTE_DEVE_SUST",
                "CO_CLIENTE"
            )
            .alias("TMP1")
        )

        tmp2 = (
            self.get_stg_tmp_egp_2()
            .filter(col("CO_MONEDA") == filtro_co_moneda)
            .select(
                "SALDOS_JTS_OID",
                "CO_PRODUCTO",
                "CO_ESTA_OPER"
            )
            .alias("TMP2")
        )

        p = broadcast(
            self.get_stg_t_co_productos()
            .filter(col("C6271") != 140106)
            .select("C6250")
            .alias("P")
        )

        calif = broadcast(
            self.get_stg_t_calif_clientes()
            .select("C0902", "CALIFALIN")
            .alias("CALIF")
        )

        df = (
            tmp1
            .join(tmp2, "SALDOS_JTS_OID")
            .join(p, col("C6250") == col("CO_PRODUCTO"))
            .join(calif, col("C0902") == col("CO_CLIENTE"))
        )

        condicion_calif = coalesce(
            when(trim(col("CALIFALIN")) != "", trim(col("CALIFALIN"))).cast("int"),
            lit(0)
        )

        df = df.filter(
            (col("CO_ESTA_OPER").isin("V", "W")) |
            (
                (col("CO_ESTA_OPER") == "N") &
                (condicion_calif >= 3)
            )
        )

        df = df.select(
            "SALDOS_JTS_OID",
            "MO_INTE_DEVE_SUST"
        )

        return df
    
    def get_sql_query_sust_2(self,filtro_co_moneda):
        """Obtiene los datos de la tabla sql_query"""

        tmp1 = (
            self.get_stg_tmp_egp_1()
            .withColumn(
                "MONTO_TOTAL",
                coalesce(col("MO_INTE_DEVE_SUST"), lit(0)) +
                coalesce(col("MO_INTE_DEVE_VIGE"), lit(0))
            )
            .filter(col("MONTO_TOTAL") > 0)
            .select(
                "SALDOS_JTS_OID",
                "CO_CLIENTE",
                "MONTO_TOTAL"
            )
            .alias("TMP1")
        )

        tmp2 = (
            self.get_stg_tmp_egp_2()
            .filter(
                (col("CO_MONEDA") == filtro_co_moneda) &
                (col("CO_ESTA_OPER") == "E")
            )
            .select(
                "SALDOS_JTS_OID",
                "CO_PRODUCTO"
            )
            .alias("TMP2")
        )

        p = broadcast(
            self.get_stg_t_co_productos()
            .select("C6250")
            .alias("P")
        )

        calif = broadcast(
            self.get_stg_t_calif_clientes()
            .select("C0902")
            .alias("CALIF")
        )

        df = (
            tmp1
            .join(tmp2, "SALDOS_JTS_OID")
            .join(p, col("C6250") == col("CO_PRODUCTO"))
            .join(calif, col("C0902") == col("CO_CLIENTE"))
        )

        df = df.select(
            col("SALDOS_JTS_OID"),
            col("MONTO_TOTAL").cast("decimal(15,2)").alias("MO_INTE_DEVE_SUST")
        )

        return df
    
    def get_sql_query_sust_3(self,filtro_co_moneda):
        """Obtiene los datos de la tabla sql_query"""

        tmp1 = (
            self.get_stg_tmp_egp_1()
            .filter(col("MO_INTE_DEVE_SUST") > 0)
            .select(
                "SALDOS_JTS_OID",
                "MO_INTE_DEVE_SUST",
                "CO_CLIENTE"
            )
            .alias("TMP1")
        )

        tmp2 = (
            self.get_stg_tmp_egp_2()
            .filter(col("CO_MONEDA") == filtro_co_moneda)
            .select(
                "SALDOS_JTS_OID",
                "CO_PRODUCTO",
                "CO_ESTA_OPER"
            )
            .alias("TMP2")
        )

        p = broadcast(
            self.get_stg_t_co_productos()
            .filter(col("C6271") == 140106)
            .select("C6250")
            .alias("P")
        )

        calif = broadcast(
            self.get_stg_t_calif_clientes()
            .select(
                "C0902",
                "CALIFALIN"
            )
            .alias("CALIF")
        )

        df = (
            tmp1
            .join(tmp2, "SALDOS_JTS_OID")
            .join(p, col("C6250") == col("CO_PRODUCTO"), "inner")
            .join(calif, col("C0902") == col("CO_CLIENTE"), "inner")
        )

        calif_alin_num = coalesce(
            when(trim(col("CALIFALIN")) != "", trim(col("CALIFALIN"))).cast("int"),
            lit(0)
        )

        df = (
            df
            .filter(
                (col("CO_ESTA_OPER").isin("V", "W")) |
                (
                    (col("CO_ESTA_OPER") == "N") &
                    (calif_alin_num >= 3)
                )
            )
            .select(
                col("SALDOS_JTS_OID"),
                col("MO_INTE_DEVE_SUST")
            )
        )

        return df
    
    def get_sql_query_sust_4(self,filtro_co_moneda):
        """Obtiene los datos de la tabla sql_query"""

        tmp1 = (
            self.get_stg_tmp_egp_1()
            .select(
                "SALDOS_JTS_OID",
                "MO_INTE_DEVE_SUST"
            )
            .alias("TMP1")
        )

        tmp2 = (
            self.get_stg_tmp_egp_2()
            .filter(
                (col("CO_MONEDA") == filtro_co_moneda) &
                (col("CO_ESTA_OPER") == "C")
            )
            .select(
                "SALDOS_JTS_OID"
            )
            .alias("TMP2")
        )

        df_resultado = (
            tmp1
            .join(
                tmp2,
                "SALDOS_JTS_OID",
                "inner"
            )
            .select(
                col("SALDOS_JTS_OID"),
                col("MO_INTE_DEVE_SUST")
            )
        )

        return df_resultado
    
    def get_sql_query_sust_5(self,filtro_co_moneda):
        """Obtiene los datos de la tabla sql_query"""

        tmp1 = (
            self.get_stg_tmp_egp_1()
            .filter(
                (col("MO_INTE_DEVE_VIGE") > 0) &
                (col("IN_REFINANCIADO") == 1)
            )
            .select(
                "SALDOS_JTS_OID",
                "MO_INTE_DEVE_VIGE",
                "CO_CLIENTE"
            )
            .alias("TMP1")
        )

        tmp2 = (
            self.get_stg_tmp_egp_2()
            .filter(col("CO_MONEDA") == filtro_co_moneda)
            .select(
                "SALDOS_JTS_OID",
                "CO_CRITERIO"
            )
            .alias("TMP2")
        )

        df_stg_t_saldos_activos, df_stg_t_saldos_pasivos = self.get_stg_t_saldos()

        s = (
            df_stg_t_saldos_activos
            .select(
                "JTS_OID",
                "MB_CREDITO_VIGENTEO"
            )
            .unionByName(
                df_stg_t_saldos_pasivos
                .select(
                    "JTS_OID",
                    "MB_CREDITO_VIGENTEO"
                )
            )
            .filter(col("MB_CREDITO_VIGENTEO") != "S")
            .alias("S")
        )

        calif = broadcast(
            self.get_stg_t_calif_clientes()
            .select("C0902")
            .alias("CALIF")
        )

        crit = broadcast(
            self.get_stg_t_cr_criterios()
            .select("COD_CRITERIO")
            .alias("CRIT")
        )

        df_resultado = (
            tmp1
            .join(
                tmp2,
                "SALDOS_JTS_OID",
                "inner"
            )
            .join(
                s,
                col("S.JTS_OID") == col("SALDOS_JTS_OID"),
                "inner"
            )
            .join(
                calif,
                col("C0902") == col("CO_CLIENTE"),
                "inner"
            )
            .join(
                crit,
                col("COD_CRITERIO") == col("CO_CRITERIO"),
                "inner"
            )
            .select(
                col("SALDOS_JTS_OID"),
                col("MO_INTE_DEVE_VIGE").alias("MO_INTE_DEVE_SUST")
            )
        )

        return df_resultado


    def get_stg_t_inte_deve_sust_raw(self):
        """Obtiene la unión cruda de intereses devengados suspendidos, sin distinct."""

        dfs = [
            self.get_sql_query_sust(2),
            self.get_sql_query_sust_2(1),
            self.get_sql_query_sust_2(2),
            self.get_sql_query_sust(1),
            self.get_sql_query_sust_3(1),
            self.get_sql_query_sust_3(2),
            self.get_sql_query_sust_4(1),
            self.get_sql_query_sust_4(2),
            self.get_sql_query_sust_5(1),
            self.get_sql_query_sust_5(2),
        ]

        return reduce(
            lambda df1, df2: df1.unionByName(df2),
            dfs
        )


    def get_stg_t_inte_deve_sust(self):
        """
        Mantiene el método original por compatibilidad.
        Usar solo si realmente necesitas filas distintas completas.
        """

        return self.get_stg_t_inte_deve_sust_raw().dropDuplicates()


    def get_stg_t_inte_deve_sust_agg(self):
        """
        Versión optimizada para ud_egp.
        Evita dropDuplicates completo y agrega directamente por SALDOS_JTS_OID.
        """

        return (
            self.get_stg_t_inte_deve_sust_raw()
            .select(
                col("SALDOS_JTS_OID"),
                col("MO_INTE_DEVE_SUST")
            )
            .groupBy("SALDOS_JTS_OID")
            .agg(
                max("MO_INTE_DEVE_SUST").alias("MO_INTE_DEVE_SUST")
            )
            .withColumnRenamed("SALDOS_JTS_OID", "SALDOS_JTS_OID_IDS")
            .alias("IDS")
        )

    def preparar_dataframes_base(self):
        """Deja listos los DataFrames de saldos que usan las etapas siguientes."""

        df_activos, df_pasivos = self.get_stg_t_saldos()

        df_activos.count()
        df_pasivos.count()
    
        df_egp_1 = self.get_stg_tmp_egp_1()

        df_egp_1.count()
    
        df_egp_2 = self.get_stg_tmp_egp_2()

        df_egp_2.count()

    def ud_egp(self): 
        """Obtiene los datos de la tabla ud_egp_final"""

        self.preparar_dataframes_base()
        
        tmp_ahorro = (
            self.get_ods_ud_egd().alias("EGD")
            .filter(
                (col("EGD.IN_ESTA_PASI") != 1) &
                (col("EGD.CO_TIPO_PASI") == 3)
            )
            .select(
                col("EGD.CO_CLIENTE").alias("CO_CLIENTE")
            )
            .dropDuplicates()
            .alias("AH")
        )

        tmp1 = self.get_stg_tmp_egp_1().alias("TMP1")

        tmp2 = self.get_stg_tmp_egp_2().alias("TMP2")

        tmp3 = self.get_stg_tmp_egp_3().alias("TMP3")

        tmp4 = self.get_stg_tmp_egp_4().alias("TMP4")

        tprec = self.get_ods_tp_ud_egp_02().alias("TPREC")

        idv = self.get_stg_t_inte_deve_vige_agg()

        ids = self.get_stg_t_inte_deve_sust_agg()

        prod = (
            self.get_stg_t_co_productos()
            .select(
                col("C6250").alias("CO_PRODUCTO_PROD"),
                col("C6253").alias("C6253")
            )
            .dropDuplicates(["CO_PRODUCTO_PROD"])
            .alias("PROD")
        )

        bshp_summary3 = (
            self.get_stg_t_bs_historia_plazo_summary3()
            .groupBy("SALDOS_JTS_OID")
            .agg(
                max("FECHAVALOR").alias("FE_PSJE_JUDI")
            )
            .withColumnRenamed("SALDOS_JTS_OID", "SALDOS_JTS_OID_BSHP3")
            .alias("BSHP3")
        )

        df_base = (
            tmp1
            .join(
                tmp2,
                col("TMP1.SALDOS_JTS_OID") == col("TMP2.SALDOS_JTS_OID"),
                "inner"
            )
            .join(
                tmp3,
                col("TMP1.SALDOS_JTS_OID") == col("TMP3.SALDOS_JTS_OID"),
                "inner"
            )
            .join(
                tmp4,
                col("TMP1.SALDOS_JTS_OID") == col("TMP4.SALDOS_JTS_OID"),
                "inner"
            )
            .join(
                tmp_ahorro,
                col("AH.CO_CLIENTE") == col("TMP1.CO_CLIENTE"),
                "left"
            )
            .join(
                tprec,
                col("TMP1.SALDOS_JTS_OID") == col("TPREC.IN_SALD_JTS_OID"),
                "left"
            )
            .join(
                idv,
                col("IDV.SALDOS_JTS_OID_IDV") == col("TMP1.SALDOS_JTS_OID"),
                "left"
            )
            .join(
                ids,
                col("IDS.SALDOS_JTS_OID_IDS") == col("TMP1.SALDOS_JTS_OID"),
                "left"
            )
            .join(
                prod,
                col("PROD.CO_PRODUCTO_PROD") == col("TMP2.CO_PRODUCTO"),
                "left"
            )
            .join(
                bshp_summary3,
                col("BSHP3.SALDOS_JTS_OID_BSHP3") == col("TMP1.SALDOS_JTS_OID"),
                "left"
            )
        )

        vl_tasa_mini_expr = round(
            when(
                col("PROD.C6253") == "E",
                col("TMP3.VL_TASA_MINI")
            ).when(
                col("PROD.C6253") == "M",
                (
                    (
                        pow(
                            lit(1) + ((col("TMP3.VL_TASA_MINI") / lit(12)) / lit(100)),
                            lit(12)
                        ) - lit(1)
                    ) * lit(100)
                )
            ).when(
                col("PROD.C6253") == "N",
                col("TMP3.VL_TASA_MINI")
            ),
            7
        )

        vl_tasa_maxi_expr = round(
            when(
                col("PROD.C6253") == "E",
                col("TMP3.VL_TASA_MAXI")
            ).when(
                col("PROD.C6253") == "M",
                (
                    (
                        pow(
                            lit(1) + ((col("TMP3.VL_TASA_MAXI") / lit(12)) / lit(100)),
                            lit(12)
                        ) - lit(1)
                    ) * lit(100)
                )
            ).when(
                col("PROD.C6253") == "N",
                col("TMP3.VL_TASA_MAXI")
            ),
            7
        )

        df_resultado = (
            df_base
            .select(
                col("TMP1.NU_PERI_MES").cast("int").alias("Nu_peri_mes"),
                col("TMP1.FE_SALDO").cast("date").alias("Fe_saldo"),
                col("TMP1.FE_PROCESO").cast("date").alias("Fe_proceso"),
                col("TMP1.NU_PRESTAMO").cast("bigint").alias("Nu_prestamo"),
                col("TMP1.CO_CLIENTE").cast("int").alias("Co_cliente"),
                col("TMP1.IN_SIN_HIPO_INSC").cast("int").alias("In_sin_hipo_insc"),
                col("TMP1.NU_PLAZ_DIAS").cast("int").alias("Nu_plaz_dias"),
                col("TMP1.VL_TEA").cast("decimal(15,7)").alias("Vl_tea"),
                col("TMP1.FE_VENCIMIENTO").cast("date").alias("Fe_vencimiento"),
                col("TMP1.FE_DESEMBOLSO").cast("date").alias("Fe_desembolso"),
                col("TMP1.FE_ULTI_PAGO").cast("date").alias("Fe_ulti_pago"),
                col("TMP1.FE_CASTIGO").cast("date").alias("Fe_castigo"),
                col("TMP1.NU_DIAS_ATRA").cast("int").alias("Nu_dias_atra"),
                col("TMP1.MO_DESEMBOLSO").cast("decimal(15,2)").alias("Mo_desembolso"),
                col("TMP1.MO_SALD_VENC").cast("decimal(15,2)").alias("Mo_sald_venc"),
                col("TMP1.MO_SALD_VIGE").cast("decimal(15,2)").alias("Mo_sald_vige"),
                col("TMP1.MO_SALD_ACTU").cast("decimal(15,2)").alias("Mo_sald_actu"),

                coalesce(col("IDV.MO_INTE_DEVE_VIGE"), lit(0)).cast("decimal(15,2)").alias("Mo_inte_deve_vige"),
                coalesce(col("IDS.MO_INTE_DEVE_SUST"), lit(0)).cast("decimal(15,2)").alias("Mo_inte_deve_sust"),

                col("TMP1.NU_CUOT_PAGA").cast("int").alias("Nu_cuot_paga"),
                col("TMP1.MO_SALD_CAPI_MN").cast("decimal(15,2)").alias("Mo_sald_capi_mn"),
                col("TMP1.NU_LINE_CRED").cast("int").alias("Nu_line_cred"),
                col("TMP1.MO_SALD_DISP_LINE").cast("decimal(15,2)").alias("Mo_sald_disp_line"),
                col("TMP1.FE_VENC_LINE").cast("date").alias("Fe_venc_line"),

                col("TMP2.TI_PRESTAMO").cast("string").alias("Ti_prestamo"),
                col("TMP2.CO_MONEDA").cast("int").alias("Co_moneda"),
                col("TMP2.CO_ESTA_OPER").cast("string").alias("Co_esta_oper"),
                col("TMP2.CO_SUCURSAL").cast("int").alias("Co_sucursal"),
                col("TMP2.CO_ANALISTA").cast("int").alias("Co_analista"),
                col("TMP2.CO_USUA_TOPA").cast("string").alias("Co_usua_topa"),
                col("TMP2.CO_CARG_ANAL").cast("int").alias("Co_carg_anal"),
                col("TMP2.CO_PUES_ANAL").cast("string").alias("Co_pues_anal"),
                col("TMP2.CO_SUCU_ANAL").cast("int").alias("Co_sucu_anal"),
                col("TMP2.CO_PRODUCTO").cast("int").alias("Co_producto"),
                col("TMP2.CO_CRITERIO").cast("int").alias("Co_criterio"),
                col("TMP2.NU_SOLICITUD").cast("int").alias("Nu_solicitud"),
                col("TMP2.NU_GRUP_SOLI").cast("int").alias("Nu_grup_soli"),
                col("TMP2.CO_SUCU_PADR").cast("int").alias("Co_sucu_padr"),
                col("TMP2.CO_ANAL_ORIG").cast("int").alias("Co_anal_orig"),
                col("TMP2.CO_CARG_ANAL_ORIG").cast("int").alias("Co_carg_anal_orig"),
                col("TMP2.CO_PUES_ANAL_ORIG").cast("string").alias("Co_pues_anal_orig"),
                col("TMP2.CO_SUCU_ANAL_ORIG").cast("int").alias("Co_sucu_anal_orig"),
                col("TMP2.TI_CLIENTE").cast("string").alias("Ti_cliente"),

                coalesce(col("TMP2.NU_CANT_EMPR"), lit(0)).cast("int").alias("Nu_cant_empr"),
                coalesce(col("TMP2.MO_SALD_SIST_FINA"), lit(0)).cast("decimal(15,2)").alias("Mo_sald_sist_fina"),
                coalesce(col("TMP2.NU_PORC_EXCL"), lit(0)).cast("decimal(15,7)").alias("Nu_porc_excl"),

                col("TMP2.NU_PERI_GRAC").cast("int").alias("Nu_peri_grac"),
                col("TMP2.CO_TIPO_GRAC").cast("string").alias("Co_tipo_grac"),
                col("TMP2.TI_EXCEPCION").cast("string").alias("Ti_excepcion"),

                col("TMP3.VL_TCEA").cast("decimal(15,2)").alias("Vl_tcea"),
                vl_tasa_mini_expr.cast("decimal(15,2)").alias("Vl_tasa_mini"),
                vl_tasa_maxi_expr.cast("decimal(15,2)").alias("Vl_tasa_maxi"),

                col("TMP3.NU_DIAS_TOTA_ATRA").cast("int").alias("Nu_dias_tota_atra"),
                col("TMP3.NU_MAXI_DIAS_ATRA").cast("int").alias("Nu_maxi_dias_atra"),
                col("TMP3.NU_CUOT_PACT").cast("int").alias("Nu_cuot_pact"),
                col("TMP3.CO_USUA_APRO_SOLI").cast("string").alias("Co_usua_apro_soli"),
                col("TMP3.IN_FOGAPI").cast("int").alias("In_fogapi"),
                col("TMP3.ST_FOGAPI").cast("string").alias("St_fogapi"),
                col("TMP3.IN_SALD_JTS_OID").cast("int").alias("In_sald_jts_oid"),
                col("TMP3.IN_GARA_ACTI").cast("int").alias("In_gara_acti"),
                col("TMP3.IN_PAGO_PARC").cast("int").alias("In_pago_parc"),
                col("TMP3.MO_DEUD_VENC").cast("decimal(15,2)").alias("Mo_deud_venc"),
                col("TMP3.MO_TOTA_DEUD").cast("decimal(15,2)").alias("Mo_tota_deud"),
                col("TMP3.NU_CUOT_VENC").cast("int").alias("Nu_cuot_venc"),
                col("TMP3.FE_VENC_CUOT").cast("date").alias("Fe_venc_cuot"),
                col("TMP3.NU_DIAS_MORA_MES").cast("int").alias("Nu_dias_mora_mes"),
                col("TMP3.MO_PAGO_TOTA_MES").cast("decimal(15,2)").alias("Mo_pago_tota_mes"),
                col("TMP3.MO_PAGO_CAPI_MES").cast("decimal(15,2)").alias("Mo_pago_capi_mes"),
                col("TMP3.MO_PAGO_COND_CAPI").cast("decimal(15,2)").alias("Mo_pago_cond_capi"),
                col("TMP3.MO_PAGO_COND_TOTA").cast("decimal(15,2)").alias("Mo_pago_cond_tota"),
                col("TMP3.IN_CONDONACION").cast("int").alias("In_condonacion"),
                col("TMP3.MO_SALD_CUEN_INTE").cast("decimal(15,2)").alias("Mo_sald_cuen_inte"),
                col("TMP3.IN_GARA_REAL").cast("int").alias("In_gara_real"),
                col("TMP3.MO_SALD_RESU").cast("decimal(15,2)").alias("Mo_sald_resu"),
                col("TMP3.VL_TASA_ICM").cast("decimal(11,7)").alias("Vl_tasa_icm"),
                col("TMP3.VL_TASA_INTE_MOR").cast("decimal(11,7)").alias("Vl_tasa_inte_mor"),
                col("TMP3.MO_TOTA_GAST").cast("decimal(15,2)").alias("Mo_tota_gast"),
                col("TMP3.VL_TIPO_CAMB_OFIC").cast("decimal(15,9)").alias("Vl_tipo_camb_ofic"),

                col("TMP4.CO_ACTI_ECON_INTE").cast("string").alias("Co_acti_econ_inte"),
                col("TMP4.CO_SECT_ECON_ALTE").cast("int").alias("Co_sect_econ_alte"),
                col("TMP4.CO_ACTIVIDAD").cast("int").alias("Co_actividad"),
                col("TMP4.CO_ACTI_CIIU").cast("string").alias("Co_acti_ciiu"),
                col("TMP4.CO_SECT_ECON_INTE").cast("int").alias("Co_sect_econ_inte"),
                col("TMP4.IN_EMPLEADO").cast("int").alias("In_empleado"),
                col("TMP4.NU_EMPR_REPO").cast("int").alias("Nu_empr_repo"),
                col("TMP4.MO_TOTA_SIST_FINA").cast("decimal(15,2)").alias("Mo_tota_sist_fina"),
                col("TMP4.FE_PROC_ULTI_PAGO").cast("date").alias("Fe_proc_ulti_pago"),

                col("BSHP3.FE_PSJE_JUDI").cast("date").alias("Fe_psje_judi"),

                col("TMP1.FE_PSJE_VENC").cast("date").alias("Fe_psje_venc"),
                col("TMP1.IN_MIGRADO").cast("int").alias("In_migrado"),
                col("TMP1.IN_REFINANCIADO").cast("int").alias("In_refinanciado"),
                col("TMP2.IN_SINIESTRO").cast("int").alias("In_siniestro"),
                col("TMP1.MO_CUOTA").cast("decimal(15,2)").alias("Mo_cuota"),
                col("TMP1.VL_DESGRAVAMEN").cast("decimal(15,2)").alias("Vl_desgravamen"),
                col("TMP1.VL_TASA_OPTI").cast("decimal(11,7)").alias("Vl_tasa_opti"),
                col("TMP1.CO_PROD_ORIG").cast("int").alias("Co_prod_orig"),

                when(
                    col("AH.CO_CLIENTE").isNotNull(),
                    lit(1)
                ).otherwise(lit(0)).cast("int").alias("In_ahorro"),

                col("TMP2.CO_DESTINO").cast("string").alias("Co_destino"),
                col("TMP1.CO_REFINANCIADO").cast("string").alias("Co_refinanciado"),

                col("TMP3.INTE_COMP").cast("decimal(15,2)").alias("Mo_inte_comp_atra"),
                col("TMP3.MORA_CONT").cast("decimal(15,2)").alias("Mo_inte_mora_pend"),

                col("TMP1.C1620").cast("date").alias("Fe_aper_cuen"),

                col("TPREC.FE_CONDONACION").cast("date").alias("Fe_condonacion"),
                col("TPREC.MO_SALD_CAPI_CUOT").cast("decimal(15,2)").alias("Mo_saca_cuot_mxat"),
                col("TPREC.MO_INTE_COMP_CUOT").cast("decimal(15,2)").alias("Mo_inco_cuot_mxat"),

                coalesce(col("TPREC.MO_INCO_MOCU_MXAT"), lit(0)).cast("decimal(15,2)").alias("Mo_inco_mocu_mxat"),
                coalesce(col("TPREC.MO_INMO_CUOT_MXAT"), lit(0)).cast("decimal(15,2)").alias("Mo_inmo_cuot_mxat"),
                coalesce(col("TPREC.MO_CARG_GAST_CUOT"), lit(0)).cast("decimal(15,2)").alias("Mo_cagt_cuot_mxat"),
                coalesce(col("TPREC.MO_ITF"), lit(0)).cast("decimal(15,2)").alias("Mo_itf_cuot_mxat"),
                coalesce(col("TPREC.MO_COMI_CUOT"), lit(0)).cast("decimal(15,2)").alias("Mo_comi_cuot_mxat"),
                coalesce(col("TPREC.NU_MAXI_CUOT_VENC"), lit(0)).cast("int").alias("Nu_ulti_cuot_venc"),
                coalesce(col("TPREC.MO_SALD_CAPI_VENC"), lit(0)).cast("decimal(15,2)").alias("Mo_sald_capi_venc"),
                coalesce(col("TPREC.MO_INTE_COMP_VENC"), lit(0)).cast("decimal(15,2)").alias("Mo_inte_comp_venc"),
                coalesce(col("TPREC.MO_CARG_GAST_VENC"), lit(0)).cast("decimal(15,2)").alias("Mo_carg_gast_venc"),
                coalesce(col("TPREC.MO_INCO_MORA_VENC"), lit(0)).cast("decimal(15,2)").alias("Mo_inco_mora_venc"),
                coalesce(col("TPREC.MO_MORA_VENC"), lit(0)).cast("decimal(15,2)").alias("Mo_mora_venc"),
                coalesce(col("TPREC.MO_COMI_VENC"), lit(0)).cast("decimal(15,2)").alias("Mo_comi_venc"),
                coalesce(col("TPREC.MO_ITF_VENC"), lit(0)).cast("decimal(15,2)").alias("Mo_itf_venc"),
                coalesce(col("TPREC.MO_SALD_CAPI_TOTA"), lit(0)).cast("decimal(15,2)").alias("Mo_sald_capi_tota"),
                coalesce(col("TPREC.MO_SALD_INTE_TOTA"), lit(0)).cast("decimal(15,2)").alias("Mo_sald_inco_tota"),
                coalesce(col("TPREC.MO_CARG_GAST_TOTA"), lit(0)).cast("decimal(15,2)").alias("Mo_carg_gast_tota"),
                coalesce(col("TPREC.MO_INTE_COMP_MORA"), lit(0)).cast("decimal(15,2)").alias("Mo_inco_mora_tota"),
                coalesce(col("TPREC.MO_MORA_TOTA"), lit(0)).cast("decimal(15,2)").alias("Mo_mora_tota"),
                coalesce(col("TPREC.MO_COMI_TOTA"), lit(0)).cast("decimal(15,2)").alias("Mo_comi_tota"),
                coalesce(col("TPREC.MO_ITF_TOTA"), lit(0)).cast("decimal(15,2)").alias("Mo_itf_tota"),

                col("TMP1.MO_CUOT_APRO_SOLI").cast("decimal(15,2)").alias("Mo_cuot_apro_soli"),
                current_timestamp().alias("_ingestion_time"),
                current_timestamp().alias("_processing_time")

            )
            .dropDuplicates(["Nu_peri_mes", "Fe_saldo", "Fe_proceso", "Nu_prestamo"])
        )
        
        self.delete_data(self.CONS_AMBIENTE,self.CONS_TABLE_FINAL)

        self.write_delta_table(df_resultado,
                          f"{self.CONS_STORAGELOCATION}{self.CONS_RUTA_TABLE_FINAL}/{self.CONS_TABLE_FINAL}",
            self.CONS_MODE_OVERWRITE)

        self.df_stg_tmp_egp_1.unpersist()

        self.df_stg_t_saldos_activos.unpersist()

        self.df_stg_t_saldos_pasivos.unpersist()

        self.df_stg_tmp_egp_2.unpersist()

        
        
        

In [0]:
# Constante del proceso
VAR_DES_PROCESO = "E4F3SF2_ODS.UD_EGP_1"

def main():
    """Función principal: instancia la clase y ejecuta la carga."""
    dict_parametros = get_parametros_proceso(var_ambiente, VAR_DES_PROCESO)
    proceso = UD_EGP(var_ambiente, var_fechaproceso, dict_parametros)
    proceso.ud_egp()

In [0]:
if __name__ == '__main__':
    ini_proceso = time.perf_counter()
    logger.info("Inicio del proceso %s. ambiente=%s fechaproceso=%s",
                VAR_DES_PROCESO, var_ambiente, var_fechaproceso)

    dict_parametros = get_parametros_proceso(var_ambiente, VAR_DES_PROCESO)

    registrar_bitacora(
        func_main         = main,
        ambiente          = var_ambiente,
        fechaproceso      = var_fechaproceso,
        des_proceso       = VAR_DES_PROCESO,
        nom_tabla_destino = dict_parametros["PRM_TABLE_FINAL"]
    )

    logger.info("Fin del proceso %s. Duracion total: %.2f segundos",
                VAR_DES_PROCESO, time.perf_counter() - ini_proceso)
